# MedGemma — Persian drug information with vs. without drug context

A controlled two-condition experiment. For every drug below, MedGemma
gets the **same** prompt, model, generation settings and message structure twice:

| | `drug_context` |
|---|---|
| Scenario 1 | `"No additional drug-specific reference information is provided."` |
| Scenario 2 | the full cleaned text of that drug's DOCX |

`drug_context` is the only variable. There is no retrieval: the document is known from
the drug name. There is no scoring either — raw responses are saved for human review.

## 1. Imports

In [1]:
import json
import os
from datetime import UTC, datetime
from html import escape
from pathlib import Path

from IPython.display import HTML, display
from openai import OpenAI
from tqdm.auto import tqdm

## 2. Configuration

Endpoint values come from the environment (`MEDGEMMA_TEXT_*`, same names as
`medgemma_persian_eval.ipynb`) and fall back to the defaults below.

In [2]:
BASE_URL = "https://opz4twy3jr1xcy-8000.proxy.runpod.net/v1"
API_KEY = "EMPTY"
MODEL_NAME = "google/medgemma-27b-text-it"

OUTPUT_PATH = Path("medgemma_drug_context_experiment.json")

# None = every drug in drug_documents; otherwise drug names, e.g. ["metformin", "warfarin"].
DRUGS_TO_RUN: list[str] | None = None

NO_CONTEXT = "No additional drug-specific reference information is provided."

# Identical for both scenarios. temperature=0 is greedy decoding; the seed pins
# anything vLLM still samples.
GENERATION_CONFIG = {
    "model": MODEL_NAME,
    "temperature": 0.0,
    "seed": 42,
}

## 3. Drug documents

Extracted once from `hiro/Docs/*.docx` and pasted here, so the notebook does not read
the DOCX files. Each drug's table becomes `content` as `criterion -> value` (the row
number column is dropped; the title and source line become `عنوان` and `منبع`). The
colour legend under each table is dropped — it describes cell colours that plain text
does not have. `insulin_types` is a comparison table, so each insulin maps to its own
`column -> value` dict. Re-extract if a DOCX changes.

In [3]:
drug_documents = {
    "acetaminophen": {
        "drug_name": "acetaminophen",
        "source_file": "acetaminophen.docx",
        "content": {
            "عنوان": "استامینوفن (Acetaminophen)",
            "منبع": "daroobin.com/fa/drug/acetaminophen",
            "نام ژنریک فارسی و انگلیسی": "استامینوفن (acetaminophen)",
            "نام‌های تجاری ایرانی": "کوریزان (دکتر عبیدی)، تهران شیمی، تایلانول، پاراستامین (شکل تزریقی، تهران شیمی)، کدیمال (دکتر عبیدی)، اکتوکلد (اکتوورکو)، نوپایرو (اکتوورکو)",  # noqa: E501
            "شکل و قدرت دارویی": "قرص ۳۲۵، ۱۲۵ و ۸۰ میلی‌گرم؛ شربت ۱۲۵ میلی‌گرم؛ اشکال: قرص، شربت، شیاف",  # noqa: E501
            "موارد مصرف": "درمان دردهای خفیف تا متوسط (سردرد، قاعدگی، دندان‌درد، کمردرد، آرتروز، سرماخوردگی) و کاهش تب",  # noqa: E501
            "نحوه مصرف": "دوز بر اساس سن و وزن کودک؛ راهنمای فرم‌های خاص (جویدنی، پیوسته‌رهش، جوشان، تزریقی وریدی طی ۱۵ دقیقه)؛ حداکثر ۳ روز برای تب و ۱۰ روز برای درد در بزرگسال (۵ روز در کودک) بدون تجویز پزشک",  # noqa: E501
            "فراموشی یک نوبت مصرف": "در صورت فراموشی، به محض یادآوری مصرف شود؛ اگر نزدیک نوبت بعد بود، دوز فراموش‌شده حذف شود و دوز دو برابر نشود",  # noqa: E501
            "عوارض شایع": "مشکلات تنفسی، ورم لب و صورت؛ در مصرف طولانی‌مدت: تیرگی ادرار و درد شکم",
            "عوارض جدی و علائم خطر": "سرفه و گلودرد شدید، خشکی دهان، التهاب دور چشم (نیاز به مراجعه فوری اورژانس)؛ علائم مصرف بیش‌ازحد: تهوع، استفراغ، بی‌اشتهایی، تعریق، خستگی مفرط، زردی چشم یا پوست، ادرار تیره",  # noqa: E501
            "تداخلات دارویی": "حساسیت یا آلرژی قبلی باید به پزشک یا داروساز اطلاع داده شود؛ توصیه به اطلاع دادن تمام داروها پیش از عمل جراحی، به‌ویژه سابقه بیماری کبد یا مصرف منظم یا سوء مصرف الکل؛ تداخل با کتوکونازول و لووکتوکونازول؛ احتمال تداخل با برخی آزمایش‌های آزمایشگاهی و ایجاد نتایج نادرست؛ احتیاط در فرمولاسیون‌های حاوی قند یا آسپارتام برای بیماران مبتلا به فنیل‌کتونوری یا دیابت؛ داروهایی که اثر استامینوفن را کاهش می‌دهند: باربیتورات‌ها، کاربامازپین، فوس‌فنی‌توئین، فنی‌توئین، ایزونیازید، لورلاتینیب؛ داروهایی که اثر استامینوفن را افزایش می‌دهند: الکل، داپسون موضعی، داساتینیب، فلوکلوگزاسیلین، ایزونیازید، متی‌راپون، نیتریک اکسید، پروبنسید، سورافنیب؛ داروهایی که استامینوفن بر عملکردشان اثر می‌گذارد: بوسولفان، داساتینیب، ایماتینیب، بی‌حس‌کننده‌های موضعی، میپومرسن، فنیل‌آفرین سیستمیک، پریلوکائین، سدیم نیتریت، سورافنیب، آنتاگونیست‌های ویتامین کا",  # noqa: E501
            "منع مصرف و احتیاطات": "ارائه نشده",
            "مصرف در بارداری و شیردهی": "در شیردهی فقط تحت نظر پزشک متخصص؛ برای کودکان زیر ۲ سال توصیه نمی‌شود مگر با تشخیص پزشک؛ در بارداری باید به پزشک اطلاع داده شود",  # noqa: E501
            "مصرف همراه غذا": "ارائه نشده",
            "شرایط نگهداری": "ارائه نشده",
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "درمان التهاب، تب و درد",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: ضدالتهاب. طبقه‌بندی مارتیندل: سیستم‌های عصبی، داروهای مسکن_ضدالتهاب و ضدتب‌ها",  # noqa: E501
            "تاریخچه‌ی دارو": "حدود ۱۵۰ سال سابقه‌ی مصرف؛ اولین بار در سال ۱۸۷۷ مورد استفاده قرار گرفته است",  # noqa: E501
            "ترکیب دارویی": "این دارو می‌تواند با کدئین (نوعی مخدر) ترکیب شود و قرص استامینوفن کدئین را تشکیل دهد",  # noqa: E501
        },
    },
    "alprazolam": {
        "drug_name": "alprazolam",
        "source_file": "alprazolam.docx",
        "content": {
            "عنوان": "آلپرازولام (Alprazolam)",
            "منبع": "daroobin.com/fa/drug/alprazolam",
            "نام ژنریک فارسی و انگلیسی": "آلپرازولام (Alprazolam)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص ۰.۵ میلی‌گرم، ۱ میلی‌گرم و ۲ میلی‌گرم",
            "موارد مصرف": "اختلالات اضطرابی؛ حملات هراس (پانیک اتک)",
            "نحوه مصرف": "خوراکی طبق تجویز پزشک؛ دوز بر اساس وضعیت پزشکی، سن و پاسخ به درمان تنظیم می‌شود و ممکن است تا رسیدن به پاسخ درمانی مطلوب افزایش یابد؛ خطر اعتیاد در مصرف طولانی‌مدت یا در افراد با سابقه اختلال مصرف مواد، نیازمند احتیاط ویژه؛ کاهش تدریجی اثر دارو (تحمل) با مصرف طولانی‌مدت ممکن است رخ دهد",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "خواب‌آلودگی، سرگیجه، افزایش تولید بزاق، تغییر در میل یا توانایی جنسی، سبکی سر",  # noqa: E501
            "عوارض جدی و علائم خطر": "تغییرات روانی یا خلقی (مانند توهم، افکار خودکشی)؛ مشکل در صحبت کردن؛ از دست دادن هماهنگی؛ مشکل در راه رفتن؛ مشکلات حافظه؛ زردی چشم یا پوست؛ تشنج؛ علائم آلرژی جدی (بثورات پوستی، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "کاوا، سدیم اکسی‌بات (تداخلات پرهیزی)؛ داروهای مؤثر بر حذف آلپرازولام از بدن: ضدقارچ‌های آزول (ایتراکونازول، کتوکونازول)، سایمتیدین، برخی ضدافسردگی‌ها (فلوکستین، فلووکسامین، نفازودون)، داروهای درمان HIV (دلاویردین، مهارکننده‌های پروتئاز مانند ایندیناویر)، آنتی‌بیوتیک‌های ماکرولیدی، ریفامایسین‌ها (ریفابوتین)، مخمر سنت جان، داروهای ضدتشنج (فنی‌توئین)؛ داروهای خواب‌آور یا مؤثر بر تنفس: مسکن‌های اوپیوئیدی (کدئین، هیدروکودون)، الکل، ماری‌جوانا، سایر داروهای خواب یا اضطراب (دیازپام، لورازپام، زولپیدم)، شل‌کننده‌های عضلانی، آنتی‌هیستامین‌ها؛ استعمال دخانیات سطح خونی آلپرازولام را کاهش می‌دهد",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت مفرط به آلپرازولام یا سایر بنزودیازپین‌ها (دیازپام، لورازپام)؛ سابقه شخصی یا خانوادگی اختلال مصرف مواد یا الکل؛ مشکلات شدید ریوی یا تنفسی (COPD، آپنه خواب)؛ بیماری کبد؛ بیماری کلیوی؛ گلوکوم؛ احتیاط بیشتر در سالمندان (خواب‌آلودگی)؛ پرهیز از الکل و رانندگی/کار با ماشین‌آلات تا رفع سرگیجه؛ مشورت با پزشک در صورت مصرف ماری‌جوانا",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود، ممکن است به جنین آسیب برساند؛ نوزادانی که در سه‌ماهه سوم بارداری در معرض دارو بوده‌اند ممکن است خواب‌آلودگی غیرمعمول، مشکل تغذیه یا تنفس، لرزش ماهیچه‌ها یا گریه مداوم داشته باشند؛ شیردهی: دارو وارد شیر مادر می‌شود و ممکن است اثرات نامطلوبی بر شیرخوار داشته باشد، شیردهی در حین مصرف این دارو توصیه نمی‌شود",  # noqa: E501
            "مصرف همراه غذا": "ارائه نشده",
            "شرایط نگهداری": "برندهای مختلف نیازهای نگهداری متفاوتی دارند؛ بررسی بسته محصول یا پرسش از داروساز توصیه می‌شود؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در توالت یا فاضلاب؛ مشورت با داروساز یا شرکت دفع زباله محلی برای امحای ایمن دارو",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "اختلال اضطراب و حملات پانیک",
            "طبقه‌بندی دارویی": "خواب‌آور و آرام‌بخش‌ها؛ طبقه‌بندی مارتیندل: داروهای اعصاب و روان؛ از خانواده بنزودیازپین‌ها",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "ارائه نشده",
            "تفاوت دوزهای دارویی (۰.۵ در برابر ۱ و ۲ میلی‌گرم)": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "علائم خاص اوردوز به‌طور مشخص ذکر نشده؛ فقط توصیه شده در صورت بروز بحران پزشکی مانند غش یا مشکل تنفسی، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
            "خطر اعتیاد و وابستگی دارویی (معیار اضافه)": "آلپرازولام می‌تواند موجب اعتیاد شود؛ احتیاط ویژه در افراد با سابقه اختلال مصرف مواد یا الکل (مانند مصرف بیش از حد یا اعتیاد به مواد مخدر یا الکل) لازم است و باید طبق دستورالعمل‌های پیشگیری از اعتیاد مصرف شود؛ مصرف طولانی‌مدت می‌تواند به کاهش تدریجی اثر دارو (تحمل دارویی) منجر شود",  # noqa: E501
        },
    },
    "amlodipine": {
        "drug_name": "amlodipine",
        "source_file": "amlodipine.docx",
        "content": {
            "عنوان": "آملودیپین (Amlodipine)",
            "منبع": "daroobin.com/fa/drug/amlodipine",
            "نام ژنریک فارسی و انگلیسی": "آملودیپین (Amlodipine)",
            "نام‌های تجاری ایرانی": "آنژیووسک (تولیدکننده: تهران شیمی)؛ آملوبر (تولیدکننده: اکتاوکو)",  # noqa: E501
            "شکل و قدرت دارویی": "قرص و شربت؛ دوزهای ۲.۵، ۵ و ۱۰ میلی‌گرم",
            "موارد مصرف": "فشار خون بالا (به تنهایی یا ترکیب با سایر داروها)؛ پیشگیری از آنژین صدری (درد قفسه سینه)",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولاً یک‌بار در روز؛ بهتر است با معده خالی مصرف شود مگر پزشک دستور دیگری بدهد؛ دوز بر اساس وضعیت بیماری و پاسخ بدن تنظیم می‌شود، معمولاً با دوز کم شروع و به‌تدریج افزایش می‌یابد؛ مصرف منظم و سر وقت برای بیشترین اثر؛ در هنگام بروز حمله آنژین نباید برای تسکین حاد آن استفاده شود، بلکه باید طبق تجویز پزشک از داروهایی مانند نیتروگلیسیرین زیر زبانی استفاده کرد؛ تغییرات سبک زندگی مانند ورزش، ترک سیگار، کاهش استرس و رژیم غذایی کم‌چرب و کم‌کلسترول می‌تواند به عملکرد بهتر دارو کمک کند",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر نزدیک به نوبت بعد بود، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "سرگیجه، سبکی سر، تورم مچ پا یا پا، گرگرفتگی، افت ناگهانی فشار خون",
            "عوارض جدی و علائم خطر": "ضربان قلب سریع، نامنظم یا تپشی، و غش؛ در برخی افراد با سابقه بیماری قلبی شدید، امکان بروز یا تشدید درد قفسه سینه یا حمله قلبی پس از شروع دارو یا افزایش دوز (علائم: درد قفسه سینه/فک/بازوی چپ، تنگی نفس، تعریق غیرعادی)؛ واکنش آلرژیک جدی (بثورات پوستی، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکلات تنفسی)",  # noqa: E501
            "تداخلات دارویی": "محصولات ضدسرفه و سرماخوردگی، کمک‌های رژیمی و NSAIDها (مانند ایبوپروفن و ناپروکسن) که ممکن است فشار خون را افزایش دهند؛ تداخلات پرهیزی: برومپریدول، کانیواپتان، فوزیدیک اسید (سیستمیک)، ایدلالیسیب، پیموزاید",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به آملودیپین یا سایر مسدودکننده‌های کانال کلسیم دی‌هیدروپیریدینی (مانند نیکاردیپین، نیفدیپین)؛ سابقه مشکل ساختاری قلبی خاص مانند تنگی آئورت، فشار خون پایین، بیماری کبدی؛ احتیاط در رانندگی و کار با ماشین‌آلات تا رفع سرگیجه؛ پرهیز از الکل و ماری‌جوانا؛ اطلاع به پزشک یا دندانپزشک پیش از جراحی یا اقدامات دندانپزشکی؛ احتیاط بیشتر در سالمندان به دلیل حساسیت بیشتر به سرگیجه",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز و طبق دستور پزشک مصرف شود؛ شیردهی: دارو وارد شیر مادر می‌شود، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "بهتر است با معده خالی مصرف شود تا اثربخشی بیشتری داشته باشد",
            "شرایط نگهداری": "قرص: در دمای اتاق، دور از نور و رطوبت، خارج از محیط حمام؛ شربت: در یخچال و دور از نور، بدون یخ‌زدگی؛ دور از دسترس کودکان و حیوانات خانگی",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "فشار خون بالا؛ آنژین صدری (درد قفسه سینه)",
            "طبقه‌بندی دارویی": "مسدودکننده‌های کانال کلسیم؛ طبقه‌بندی مارتیندل: مسدودکننده‌های منتخب کانال کلسیمی با اثرات عمدتاً عروقی، داروهای مسدودکننده کانال کلسیم",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور خاص ذکر شده: بررسی و کنترل منظم فشار خون در طول مصرف دارو توصیه شده است",  # noqa: E501
            "تفاوت دوزهای دارویی (۲.۵ در برابر ۵ و ۱۰ میلی‌گرم)": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "غش و سرگیجه شدید؛ در صورت بروز بحران‌های پزشکی مانند غش یا مشکل تنفسی، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "amoxicillin": {
        "drug_name": "amoxicillin",
        "source_file": "amoxicillin.docx",
        "content": {
            "عنوان": "آموکسی‌سیلین (Amoxicillin)",
            "منبع": "daroobin.com/fa/drug/amoxicillin",
            "نام ژنریک فارسی و انگلیسی": "آموکسی‌سیلین (Amoxicillin)",
            "نام‌های تجاری ایرانی": "ارائه نشده (فقط اشاره کلی به «برندهای مختلف» بدون ذکر نام مشخص)",  # noqa: E501
            "شکل و قدرت دارویی": "آمپول، قرص، قرص جویدنی، کپسول، سوسپانسیون خوراکی؛ دوزهای ۱۲۵، ۲۰۰، ۲۵۰، ۴۰۰ و ۵۰۰ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "طیف گسترده‌ای از عفونت‌های باکتریایی از جمله عفونت‌های قفسه سینه، سینوس، عفونت ادراری، برخی عفونت‌های دندانی، پیشگیری از برخی عفونت‌ها؛ درمان عفونت هلیکوباکتر پیلوری همراه با سایر داروها؛ درمان زخم معده یا روده ناشی از H. پیلوری و جلوگیری از بازگشت زخم",  # noqa: E501
            "نحوه مصرف": "خوراکی، با یا بدون غذا، هر ۸ تا ۱۲ ساعت یکبار (۱ تا ۳ بار در روز)؛ نوشیدن مایعات زیاد در طول مصرف مگر توصیه دیگری از پزشک؛ مصرف منظم و در ساعت مشخص؛ تکمیل کامل دوره تجویزی حتی با بهبود علائم (دوره درمان معمولاً بیش از ۲ هفته نیست)؛ عدم کم یا زیاد کردن خودسرانه دوز؛ در کودکان دوز بر اساس وزن و نوع عفونت تنظیم می‌شود؛ شکل شربت ممکن است در کودکان باعث لکه دندان شود که با مراقبت دهان و دندان قابل پیشگیری است",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر نزدیک به نوبت بعد بود، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، اسهال",
            "عوارض جدی و علائم خطر": "برفک دهان یا عفونت قارچی واژن در مصرف طولانی یا مکرر؛ ادرار تیره، تهوع یا استفراغ مداوم، درد معده یا شکم، زردی چشم یا پوست، کبودی یا خونریزی آسان، گلودرد، تب مداوم؛ بیماری شدید روده ناشی از باکتری C. difficile (اسهال مداوم، درد یا گرفتگی شکم، خون یا مخاط در مدفوع)؛ بثورات پوستی که معمولاً خفیف‌اند اما گاهی می‌توانند نشانه واکنش آلرژیک شدید باشند؛ واکنش آلرژیک جدی (بثورات، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "متوترکسات؛ کاهش اثر واکسن‌های باکتریایی زنده (مانند واکسن تیفوئید)؛ تداخل با نتایج آزمایش ادرار دیابتی نوع سولفات کوپریک (احتمال نتیجه مثبت کاذب)؛ کاهش احتمالی اثر قرص‌های ضدبارداری (به دلیل اسهال یا استفراغ ناشی از دارو، توصیه به استفاده از روش‌های غیردارویی مکمل)",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به آموکسی‌سیلین یا سایر آنتی‌بیوتیک‌های پنی‌سیلین یا سفالوسپورین؛ سابقه بیماری کلیوی؛ نوع خاصی از عفونت ویروسی (مونونوکلئوز عفونی)؛ اطلاع به مسئولین واکسیناسیون پیش از هر نوع ایمن‌سازی به دلیل کاهش اثر واکسن‌های زنده؛ اطلاع به پزشک یا دندانپزشک پیش از هرگونه جراحی",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود، مشورت با پزشک درباره خطرات و فواید لازم است؛ شیردهی: دارو وارد شیر مادر می‌شود، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با یا بدون غذا مصرف شود؛ منافاتی با غذا ندارد",
            "شرایط نگهداری": "برندهای مختلف نیاز به نگهداری متفاوتی دارند؛ مطالعه راهنمای برند خریداری‌شده توصیه می‌شود؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب یا سینک؛ دور ریختن صحیح دارو پس از پایان انقضا یا عدم نیاز",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "عفونت‌های باکتریایی",
            "طبقه‌بندی دارویی": "آنتی‌بیوتیک‌ها؛ طبقه‌بندی مارتیندل: ضد عفونت‌ها؛ از خانواده پنی‌سیلین",
            "نظارت آزمایشگاهی دوره‌ای": "ارائه نشده",
            "تفاوت دوزهای دارویی (۲۵۰ در برابر ۵۰۰ میلی‌گرم)": "به‌صورت جزئی ذکر شده: دوز ۵۰۰ میلی‌گرمی در بیماری‌هایی که نیاز به آنتی‌بیوتیک با قدرت بالاتر دارند رایج‌تر است؛ توضیح بالینی کامل‌تری بین سایر دوزها (۱۲۵، ۲۰۰، ۴۰۰ میلی‌گرم) ارائه نشده است",  # noqa: E501
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "غش یا مشکل تنفسی؛ در صورت بروز این علائم، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
            "کاربردهای تکمیلی ذکر شده در متن (معیار اضافه)": "امکان استفاده از آموکسی‌سیلین برای درمان جوش صورت با منشا التهابی یا باکتریایی (مشابه استفاده از مترونیدازول برای همین منظور)؛ در برخی افراد ممکن است مصرف آن باعث بروز لک یا التهاب اضافی پوست شود که در این صورت باید به متخصص مراجعه کرد",  # noqa: E501
        },
    },
    "aspirin": {
        "drug_name": "aspirin",
        "source_file": "aspirin.docx",
        "content": {
            "عنوان": "آسپیرین (Aspirin)",
            "منبع": "daroobin.com/fa/drug/aspirin",
            "نام ژنریک فارسی و انگلیسی": "آسپیرین، استیل سالیسیلات، آ اس آ (Aspirin، ASA)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "ارائه نشده",
            "موارد مصرف": "کاهش دردهای خفیف تا متوسط، تورم و تب؛ رقیق‌کننده خون برای پیشگیری از لخته خون و کاهش احتمال سکته و حمله قلبی",  # noqa: E501
            "نحوه مصرف": "با یک لیوان کامل آب مصرف شود؛ حداقل ۱۰ دقیقه بعد از مصرف دراز نکشید؛ در صورت شکم‌درد با غذا یا شیر مصرف شود؛ قرص‌های با پوشش روده‌ای (E.C.) کامل بلعیده شوند و جویده نشوند؛ قرص‌های آهسته‌رهش خرد یا جویده نشوند مگر خط تقسیم داشته باشند؛ حداکثر ۱۰ روز برای درد و ۳ روز برای تب بدون تجویز پزشک؛ هشدار: در صورت مشکل تکلم، ضعف یک‌طرفه بدن یا تغییرات ناگهانی بینایی هنگام خوددرمانی سردرد، فوراً با اورژانس تماس گرفته شود.",  # noqa: E501
            "فراموشی یک نوبت مصرف": "در صورت فراموشی، به محض یادآوری مصرف شود مگر نزدیک نوبت بعد باشد که در این صورت نادیده گرفته شده و دوز دو برابر نشود؛ در اوردوز همراه با بی‌هوشی یا تنگی نفس با اورژانس تماس گرفته شود؛ سایر علائم اوردوز: درد یا سوزش گلو یا معده، گیجی، تغییرات خلقی، غش، ضعف، زنگ زدن گوش، تب، تنفس سریع، تغییر مقدار ادرار، تشنج، کاهش هوشیاری",  # noqa: E501
            "عوارض شایع": "درد یا سوزش معده",
            "عوارض جدی و علائم خطر": "تهوع یا استفراغ ادامه‌دار، افزایش خونریزی یا کبودی، مشکل شنوایی، زنگ زدن گوش، علائم مشکل کبدی، خستگی بدون دلیل، گیجی، ادرار تیرهزردی پوست یا چشم؛ علائم خونریزی گوارشی (درد مداوم شکم، مدفوع تیره یا قیرمانند، استفراغ شبیه تفاله قهوه، مشکل تکلم، ضعف یک‌طرفه، تغییر بینایی، سردرد شدید) نیازمند تماس فوری با اورژانس؛ علائم آلرژی شدید (تب، تورم غدد لنفاوی، قرمزی، خارش یا تورم صورت‌زبان‌گلو، سرگیجه شدید، تنگی نفس)؛ هشدار سندروم ری در کودکان و نوجوانان دارای آبله‌مرغان، آنفولانزا یا واکسن اخیر، با علامت اولیه تغییر در تهوع یا استفراغ",  # noqa: E501
            "تداخلات دارویی": "میفپریستون، استازولامید، رقیق‌کننده‌های خون (وارفارین، هپارین)، کورتیکواستروئیدها (پردنیزون)، دیکلرفنامید، متوترکسات، اسید والپروئیک، داروهای گیاهی مانند کهن‌دار یا جینکو بیلوبا؛ احتیاط در صورت دریافت اخیر واکسن‌های زنده مانند واریسل یا آنفولانزا؛ مصرف روزانه NSAIDها مانند ایبوپروفن می‌تواند اثر محافظت قلبی آسپیرین را کاهش دهد؛ تداخل با تست‌های آزمایشگاهی مانند تست قند ادرار؛ ویژگی‌های کلی تداخل: نفروپاتی ناشی از مسکن‌ها، اثر ضدپلاکتی، اتصال به کاتیون‌های چندظرفیتی، کاهش پرفیوژن یا عملکرد کلیوی، افزایش خطر خونریزی گوارشی، اسیدوز متابولیک، اثرات اوریکوزوریک و ضداوریکوزوریک، حساسیت به pH ادرار؛ تداخلات پرهیزی: دکس‌ایبوپروفن، دکس‌کتوپروفن، فلوکتافنین، واکسن زنده آنفولانزا، کتورولاک، ماسیمورلین، اوماستاکسین، سولفین‌پیرازون، اوروکیناز، واکسن آبله‌مرغان",  # noqa: E501
            "منع مصرف و احتیاطات": "کودکان زیر ۱۲ سال نیازمند مشورت پزشک؛ نوجوانان زیر ۱۸ سال با واکسن اخیر، آبله‌مرغان، آنفولانزا یا بیماری تشخیص‌نشده نباید مصرف کنند به دلیل خطر سندروم ری؛ ممنوع در اختلالات خونریزی یا لخته خون مانند هموفیلی، کمبود ویتامین K، پلاکت پایین؛ احتیاط در بیماری کلیوی، بیماری کبدی، دیابت، مشکلات معده مانند زخم یا سوزش، آسم حساس به آسپیرین، پولیپ بینی، نقرس، نارسایی آنزیمی G6PD یا پیروات کیناز؛ حساسیت به سایر سالیسیلات‌ها یا پرهیز از الکل و تنباکو؛ اطلاع پیش از عمل جراحی؛ احتیاط بیشتر در سالمندان به دلیل حساسیت بیشتر به خونریزی گوارشی و زخم معده",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: به عنوان مسکن یا تب‌بر توصیه نمی‌شود، به‌ویژه از هفته ۲۰ به بعد؛ در صورت تجویز پزشک بین هفته ۲۰ تا ۳۰ با کمترین دوز ممکن؛ دوز پایین ۸۱ تا ۱۶۲ میلی‌گرم روزانه برای پیشگیری از برخی بیماری‌ها ایمن تلقی می‌شود؛ شیردهی: دوز بالا توصیه نمی‌شود، دوز پایین برای پیشگیری قلبی یا سکته با نظر پزشک بلامانع است",  # noqa: E501
            "مصرف همراه غذا": "برای کاهش مشکلات گوارشی با غذا یا مقدار زیادی شیر مصرف شود؛ غذا سرعت جذب را کم می‌کند اما میزان کلی جذب را تغییر نمی‌دهد؛ مصرف الکل، آلو، کشمش، چای، خیارشور، پودر کاری، فلفل قرمز و شیرین‌بیان می‌تواند باعث تجمع سالیسیلات شود؛ میوه‌های تازه حاوی ویتامین سی می‌توانند ترشح ادراری آسپیرین را افزایش دهند",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ دور از دسترس اطفال و حیوانات خانگی؛ داروها در توالت یا سینک تخلیه نشوند؛ هشدار: در صورت بوی شدید شبیه سرکه، دارو فاسد شده و نباید مصرف شود",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "عدم وجود بیماری هدف در جعبه اطلاعات": "برخلاف داروهای قبلی بررسی‌شده، در جعبه اطلاعات تخصصی آسپرین هیچ ردیف بیماری هدف ذکر نشده است؛ فقط طبقه‌بندی و شکل دارو آمده",  # noqa: E501
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای ضدالتهاب و ضددرد. طبقه‌بندی مارتیندل: سیستم‌های عصبی، ضدترومبوتیک‌ها، خون و اندام‌های خون‌ساز، داروهای مسکن_ضدالتهاب و ضدتب‌ها",  # noqa: E501
            "طبقه‌بندی تکمیلی": "این دارو همچنین در دسته داروهای ضدانعقاد خون طبقه‌بندی می‌شود، مشابه انوکساپارین، پراسوگرل و تینزاپارین",  # noqa: E501
            "تست‌های آزمایشگاهی دوره‌ای": "در مصرف منظم و پرمقدار، توصیه به انجام دوره‌ای تست عملکرد کبد و کلیه، آزمایش خون کامل و سطح سالیسیلات خون",  # noqa: E501
        },
    },
    "atorvastatin": {
        "drug_name": "atorvastatin",
        "source_file": "atorvastatin.docx",
        "content": {
            "عنوان": "آتورواستاتین (Atorvastatin)",
            "منبع": "daroobin.com/fa/drug/atorvastatin",
            "نام ژنریک فارسی و انگلیسی": "آتورواستاتین (Atorvastatin)",
            "نام‌های تجاری ایرانی": "آتورکسین (اکتوورکو)، آتوسترول (تهران شیمی)؛ در متن نیز به تولیدکنندگان دیگری مانند سبحان دارو، اکسیر، عبیدی، شفا و روزدارو اشاره شده بدون ذکر نام تجاری دقیق هرکدام",  # noqa: E501
            "شکل و قدرت دارویی": "قرص در دوزهای ۱۰، ۲۰ و ۴۰ میلی‌گرم",
            "موارد مصرف": "کاهش کلسترول بد و تری‌گلیسیرید و افزایش کلسترول خوب در خون؛ کاهش خطر بیماری قلبی، سکته مغزی و حمله قلبی",  # noqa: E501
            "نحوه مصرف": "یکبار در روز و در ساعت مشخص مصرف شود؛ حتی در صورت احساس خوب بودن باید مصرف ادامه یابد؛ ممکن است تا چهار هفته طول بکشد تا فواید کامل دارو ظاهر شود؛ در صورت مصرف رزین‌های متصل به اسید صفراوی مانند کلستیرامین یا کلستیپول، باید حداقل یک ساعت قبل یا چهار ساعت بعد از آنها مصرف شود؛ قطع ناگهانی دارو می‌تواند باعث فعال شدن مجدد آنزیم تولیدکننده کلسترول و افزایش سرکش آن ظرف یک تا ده روز شود؛ جایگزین کردن خودسرانه این دارو با سایر داروهای قلب و عروق بسیار خطرناک اعلام شده است",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود مگر نزدیک نوبت بعد باشد که در این صورت نادیده گرفته شده و دوز دو برابر نشود؛ در اوردوز همراه با غش یا مشکل تنفسی با اورژانس تماس گرفته شود؛ سایر علائم اوردوز شامل سرگیجه شدید، غش و تغییرات روانی یا خلقی مانند توهم است",  # noqa: E501
            "عوارض شایع": "ارائه نشده",
            "عوارض جدی و علائم خطر": "مشکلات خفیف حافظه یا گیجی به‌ندرت؛ احتمال ایجاد یا بدتر شدن دیابت به‌ندرت؛ مشکلات عضلانی که به‌ندرت می‌تواند به شرایط جدی رابدومیولیز و میوپاتی خودایمنی منجر شود، با علائم درد، حساسیت یا ضعف عضلانی به‌ویژه همراه با تب یا خستگی غیرعادی؛ علائم مشکلات کلیوی مانند تغییر در مقدار ادرار؛ مشکلات کبدی نادر با علائم تهوع یا استفراغ مداوم، زردی چشم یا پوست، ادرار تیره و درد شدید معده؛ علائم آلرژی جدی شامل بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و مشکلات تنفسی",  # noqa: E501
            "تداخلات دارویی": "داپتومایسین و جم‌فیبروزیل؛ داروهایی که بر حذف یا عملکرد آتورواستاتین اثر می‌گذارند شامل سیکلوسپورین، ترکیب گلکاپرویر و پیبرنتاسویر تلیترومایسین و ریتوناویر؛ ممنوعیت کامل مصرف همزمان با محصولات حاوی مخمر برنج قرمز به دلیل وجود لواستاتین طبیعی و افزایش خطر مشکلات عضلانی و کبدی؛ تداخلات پرهیزی شامل فرآورده‌های ترکیبی ضدویروس هپاتیت سی، کانیواپتان، سیکلوسپورین سیستمیک، فوزیدیک اسید سیستمیک، جم‌فیبروزیل، گلکاپرویر و پیبرنتاسویر، ایدلالیسیب، پوساکونازول، مخمر برنج قرمز و تیپراناویر",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به دارو؛ سابقه بیماری کبدی، بیماری کلیوی و مصرف الکل؛ اطلاع پیش از عمل جراحی؛ محدود کردن مصرف الکل به دلیل افزایش خطر مشکلات کبدی همراه با این دارو؛ احتیاط بیشتر در سالمندان به دلیل حساسیت بیشتر به مشکلات عضلانی؛ ممنوعیت کامل مصرف در تمام دوزها و اشکال در دوران بارداری؛ پرهیز از جایگزینی خودسرانه با داروهای مشابه",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: ممنوعیت کامل به دلیل خطر آسیب به جنین؛ استفاده از روش‌های موثر پیشگیری از بارداری حین مصرف ضروری است و در صورت بروز بارداری باید فورا به پزشک اطلاع داده شود؛ شیردهی: مشخص نیست دارو وارد شیر مادر می‌شود یا نه، به دلیل خطر احتمالی برای نوزاد توصیه نمی‌شود",  # noqa: E501
            "مصرف همراه غذا": "با هیچ غذایی به‌جز گریپ‌فروت و مخمر برنج قرمز تداخل ندارد و قابل مصرف در هر ساعت مشخص، قبل یا بعد از غذا است؛ مصرف گریپ‌فروت یا آب آن ممنوع است زیرا اثر و عوارض دارو را افزایش می‌دهد؛ مصرف محصولات حاوی مخمر برنج قرمز نیز ممنوع است",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ در حمام نگهداری نشود؛ دور از دسترس کودکان و حیوانات خانگی؛ داروها در توالت یا سینک تخلیه نشوند مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "قلب و عروق، چربی خون",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای ضدچربی خون. طبقه‌بندی مارتیندل: اصلاح‌کننده‌های چربی ساده، داروهای قلبی عروقی",  # noqa: E501
            "توصیه‌های سبک زندگی مکمل": "رژیم غذایی مناسب، ورزش، کاهش وزن در صورت اضافه‌وزن، ترک سیگار و کاهش نمک به‌عنوان مکمل درمان دارویی توصیه شده است",  # noqa: E501
            "توضیح مکانیسمی قطع ناگهانی دارو": "آتورواستاتین آنزیم تولیدکننده کلسترول را مهار و در طولانی‌مدت غیرفعال می‌کند؛ قطع ناگهانی دارو می‌تواند باعث فعال شدن مجدد و افزایش سرکش این آنزیم در خون ظرف یک تا ده روز شود",  # noqa: E501
            "هشدار جایگزینی خودسرانه": "جایگزین کردن این دارو با سایر داروهای دسته قلب و عروق مانند روزوواستاتین، لوواستاتین یا کلوفیبرات بدون تجویز پزشک بسیار خطرناک اعلام شده است",  # noqa: E501
        },
    },
    "azithromycin": {
        "drug_name": "azithromycin",
        "source_file": "azithromycin(1).docx",
        "content": {
            "عنوان": "آزیترومایسین (Azithromycin)",
            "منبع": "daroobin.com/fa/drug/azithromycin",
            "نام ژنریک فارسی و انگلیسی": "آزیترومایسین (azithromycin)",
            "نام‌های تجاری ایرانی": "آزیترومایسین کیمیدارو، آزیترودر، آزیترومایسین لقمان، آزیترومایسین روزدارو، آزیروسین (تهران شیمی)، آزیتروژن، زیتروکم، زیمکسیر",  # noqa: E501
            "شکل و قدرت دارویی": "قرص ۲۵۰ و ۵۰۰ میلی‌گرم؛ کپسول ۲۵۰ و ۵۰۰ میلی‌گرم؛ ویال ۵۰۰ میلی‌گرم؛ سوسپانسیون ۱۰۰ و ۲۰۰ میلی‌گرم و ۲ گرم؛ قطره چشمی ۱ درصد و ۱.۵ درصد",  # noqa: E501
            "موارد مصرف": "عفونت‌های ریه، عفونت گوش میانی، عفونت‌های پوستی، عفونت‌های واژینال و تناسلی، زخم‌های تناسلی در آقایان، بیماری انسداد مزمن ریوی، بیماری التهابی لگن، پنومونی، فارنژیت، آکنه وولگاریس، برونشیت، عفونت‌های کمپلکس مایکوباکتری، عفونت‌های کلامیدیایی، عفونت‌های گنکوکی؛ فقط برای عفونت باکتریایی و بدون اثر بر سرماخوردگی یا آنفولانزای ویروسی",  # noqa: E501
            "نحوه مصرف": "دوره درمان باید کامل انجام شود؛ معمولا یکبار در روز به مدت ۳ تا ۵ روز؛ دوز بزرگسال بر اساس نوع عفونت و پاسخ درمانی، دوز کودک بر اساس وزن؛ برای کودکان اغلب فرم سوسپانسیون تجویز می‌شود؛ کپسول با معده خالی، یک ساعت قبل یا دو ساعت بعد از غذا مصرف شود مگر مشکل معده که همراه غذا مصرف شود؛ قرص و سوسپانسیون با یا بدون غذا قابل مصرف هستند؛ در صورت مصرف آنتی‌اسید حاوی آلومینیوم یا منیزیم، حداقل دو ساعت قبل یا بعد از آزیترومایسین مصرف شود؛ مصرف منظم در ساعات مشخص برای اثر بهتر توصیه می‌شود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود مگر نزدیک نوبت بعد باشد که در این صورت دوز فراموش‌شده نادیده گرفته شده و دوز دو برابر نشود؛ در مسمومیت همراه با بی‌هوشی یا تنگی نفس با اورژانس تماس گرفته شود",  # noqa: E501
            "عوارض شایع": "شکم‌درد، اسهال یا مدفوع شل، تهوع، استفراغ، درد پایین شکم",
            "عوارض جدی و علائم خطر": "تغییرات شنوایی، مشکلات چشمی مانند افتادگی پلک یا تاری دید، مشکلات تکلم، مشکل در بلع، ضعف عضلات، علائم بیماری کبدی، ضربان قلب تند یا نامنظم، سرگیجه شدید، غش؛ خطر عفونت روده‌ای حاد ناشی از باکتری سی دیفیسیل با علائم اسهال ممتد، درد یا گرفتگی شکم، خون یا مخاط در مدفوع که ممکن است هفته‌ها یا ماه‌ها پس از قطع دارو هم بروز کند؛ خطر سفیدک دهانی یا عفونت قارچی جدید در مصرف طولانی‌مدت یا دوز بالا؛ علائم آلرژی جدی شامل تب ممتد، تورم غدد لنفاوی، راش پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و تنگی نفس که ممکن است تا چند روز پس از پایان دوره درمان هم رخ دهد",  # noqa: E501
            "تداخلات دارویی": "داروهای طولانی‌کننده فاصله قلب شامل آمیودارون، کلروکین، دیزوپایرامید، دوفتیلید، دروندارون، هیدروکسی‌کلروکین، ابوتیلید، پیموزایدQT پروکائین‌آمید، کوئینیدین و سوتالول؛ اختلال در عملکرد واکسن‌های باکتریایی زنده مانند واکسن تیفوئید؛ توصیه به اطلاع‌رسانی تمام داروهای مصرفی پیش از عمل جراحی یا اقدامات دندانپزشکی",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به آزیترومایسین یا سایر آنتی‌بیوتیک‌ها مانند مترونیدازول، سفکسیم، اریترومایسین، کلاریترومایسین یا تلیترومایسین؛ آلرژی به غذا، مواد افزودنی یا رنگ‌های خوراکی؛ سابقه بیماری کبدی، بیماری کلیه یا میاستنی گراویس؛ افزایش خطر طولانی شدن بیماری قلبی مانند نارسایی قلبی یا ضربان پایین، سابقه خانوادگی مشکلات قلبی، کمبود پتاسیم یا منیزیم، مصرف داروهای ادرارآور، تعریق شدید، اسهال یا استفراغ.",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط با تایید پزشک و در صورت ضرورت مجاز است؛ شیردهی: اطلاعاتی درباره انتقال دارو به شیر مادر موجود نیست، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "کپسول با معده خالی، یک ساعت قبل یا دو ساعت بعد از غذا مصرف شود مگر مشکل معده که همراه غذا مصرف شود؛ قرص و سوسپانسیون با یا بدون غذا قابل مصرف هستند؛ فاصله حداقل دو ساعت با آنتی‌اسیدهای حاوی آلومینیوم یا منیزیم رعایت شود",  # noqa: E501
            "شرایط نگهداری": "دور از دسترس کودکان و حیوانات خانگی؛ قرص، کپسول و قطره چشمی در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ ویال داروی بیمارستانی است و نیاز به نگهداری در منزل ندارد و در صورت تغییر رنگ محلول یا وجود ذرات نباید تزریق شود؛ سوسپانسیون آماده‌شده حداکثر ۱۰ روز در یخچال قابل نگهداری است؛ عدم تخلیه در فاضلاب یا سینک",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "عدم وجود بیماری هدف در جعبه اطلاعات": "مشابه آسپیرین، در جعبه اطلاعات اولیه این دارو نیز ردیف بیماری هدف وجود ندارد؛ فقط طبقه‌بندی ذکر شده است",  # noqa: E501
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: آنتی‌بایوتیک‌ها. طبقه‌بندی مارتیندل: آنتی‌بیوتیک‌های سیتوتوکسیک و مواد مرتبط",  # noqa: E501
            "هشدار مقاومت آنتی‌بیوتیکی": "توضیح گسترده درباره خطر قطع زودهنگام یا مصرف خودسرانه آنتی‌بیوتیک و ایجاد مقاومت باکتریایی، به عنوان یک هشدار آموزشی عمومی",  # noqa: E501
            "فایل دانلودی تداخلات دارویی": "برخلاف داروهای قبلی، برای این دارو یک فایل جداگانه قابل دانلود برای مشاهده فهرست کامل‌تر تداخلات دارویی ارائه شده است",  # noqa: E501
        },
    },
    "carbamazepine": {
        "drug_name": "carbamazepine",
        "source_file": "carbamazepine.docx",
        "content": {
            "عنوان": "کاربامازپین (Carbamazepine)",
            "منبع": "daroobin.com/fa/drug/carbamazepine",
            "نام ژنریک فارسی و انگلیسی": "کاربامازپین (Carbamazepine)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص ۲۰۰ و ۴۰۰ میلی‌گرم؛ قرص آهسته‌رهش ۴۰۰ میلی‌گرم؛ قرص پیوسته‌رهش ۲۰۰ میلی‌گرم؛ سوسپانسیون/شربت ۱۰۰ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "پیشگیری و کنترل تشنج (صرع)؛ تسکین برخی دردهای عصبی مانند نورالژی سه‌قلو",
            "نحوه مصرف": "خوراکی، همراه غذا؛ قرص جویدنی باید پیش از بلع کاملاً جویده شود؛ سوسپانسیون پیش از مصرف باید خوب تکان داده شود و با قاشق مدرج اندازه‌گیری شود؛ فاصله حداقل ۲ ساعت بین سوسپانسیون کاربامازپین و سایر داروهای مایع؛ در بزرگسالان دوز بر اساس وضعیت پزشکی و پاسخ به درمان، و در کودکان بر اساس وزن تعیین می‌شود؛ معمولاً شروع با دوز پایین و افزایش تدریجی؛ مصرف منظم در ساعت مشخص؛ عدم قطع ناگهانی بدون مشورت پزشک (کاهش تدریجی دوز در صورت لزوم)؛ پرهیز از مصرف گریپ‌فروت یا آب آن مگر با اجازه پزشک",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر به زمان نوبت بعدی نزدیک بود، از آن نوبت صرف‌نظر شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، سرگیجه، خواب‌آلودگی، یبوست، خشکی دهان، بی‌ثباتی",
            "عوارض جدی و علائم خطر": "سردردهای شدید یا مداوم؛ علائم مشکلات کبدی (تهوع/استفراغ مزمن، بی‌اشتهایی، درد شکم، زردی، ادرار تیره)؛ علائم مشکلات کلیوی؛ زخم‌های دهان؛ غش؛ ضربان قلب سریع، آهسته یا نامنظم؛ حرکات غیرمعمول چشم (نیستاگموس)؛ تغییرات بینایی (تاری دید)؛ درد مفاصل، تورم مچ پا و ساق؛ درد، قرمزی یا تورم بازوها یا پاها؛ بی‌حسی یا سوزن‌سوزن‌شدن دست و پا؛ علائم سطح پایین سدیم خون (خواب‌آلودگی شدید، تغییرات ذهنی/خلقی، تشنج)؛ افسردگی و افکار یا اقدام به خودکشی؛ واکنش‌های پوستی بسیار جدی یا کشنده (بثورات، تاول، پوسته‌ریزی، خارش یا تورم)، با خطر بیشتر در برخی گروه‌های قومی؛ اختلالات خونی جدی مانند کم‌خونی آپلاستیک و آگرانولوسیتوز (علائم: عفونت مداوم، تب، غدد لنفاوی متورم، ضعف غیرمعمول، تنگی نفس، خونریزی یا کبودی آسان)؛ واکنش آلرژیک جدی (بثورات، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "فهرست فوق‌العاده گسترده، شامل صدها دارو: ضدقارچ‌های آزول (ایزاووکونازونیوم، وریکونازول)، اورلیستات، مهارکننده‌های MAO، آنتی‌بیوتیک‌های ماکرولید، ایزونیازید، باربیتورات‌ها، تلیترومایسین، لینزولید؛ کاربامازپین به‌عنوان القاکننده قوی آنزیم‌های کبدی (به‌ویژه CYP3A4) اثر بسیاری از داروها را کاهش می‌دهد، از جمله داروهای ضدبارداری هورمونی، بسیاری از داروهای ضدسرطان، ضدویروس، ضدانعقادها (وارفارین، آپیکسابان، ریواروکسابان، دابیگاتران)، مسدودکننده‌های کانال کلسیم، بنزودیازپین‌ها و بسیاری داروهای روانپزشکی؛ همچنین با تئوفیلین، فنی‌توئین، دانازول، سایمتیدین، SSRIها، لاموتریژین، اسید والپروئیک، لیتیوم، فلبامات، امپرازول/اسومپرازول تداخل دارد؛ فهرست طولانی تداخلات رده X (پرهیزی) شامل ده‌ها دارو مانند ایتراکونازول، کتوکونازول، وریکونازول و بسیاری داروهای تخصصی دیگر",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به کاربامازپین، سایر ضدتشنج‌ها (فسفنی‌توئین، اکسکاربازپین، فنوباربیتال، فنی‌توئین، پریمیدون) یا ضدافسردگی‌های سه‌حلقه‌ای؛ احتمال حساسیت به سوربیتول در فرم سوسپانسیون؛ سابقه کاهش عملکرد مغز استخوان، اختلالات خونی (پورفیری، کم‌خونی)، گلوکوم، بیماری قلبی (عروق کرونر، نارسایی قلبی، آریتمی)، بیماری کلیوی، بیماری کبد، اختلالات روانی/خلقی (افسردگی)، عدم تعادل مواد معدنی (سدیم یا کلسیم پایین)؛ احتیاط رانندگی/کار با ماشین تا رفع سرگیجه یا خواب‌آلودگی؛ پرهیز از الکل و اطلاع به پزشک در صورت مصرف ماری‌جوانا؛ افزایش حساسیت پوست به نور خورشید (کرم ضدآفتاب، لباس محافظ)؛ احتمال وجود قند در فرم جویدنی یا سوسپانسیون - احتیاط در دیابتی‌ها؛ اطلاع به پزشک یا دندانپزشک پیش از جراحی؛ احتیاط بیشتر در سالمندان (گیجی، بی‌ثباتی، خطر سقوط، خطر بیشتر افت سدیم خون به‌ویژه همراه با دیورتیک‌ها)؛ خطر بیشتر واکنش‌های پوستی جدی در برخی اقوام مانند افراد آسیایی و آسیای جنوبی",  # noqa: E501
            "مصرف در بارداری و شیردهی": "در صورت بارداری، انجام آزمایش‌هایی برای کنترل نقایص مادرزادی توصیه می‌شود؛ روش‌های ضدبارداری هورمونی (قرص، چسب، ایمپلنت، تزریق) ممکن است اثربخشی کافی نداشته باشند و نیاز به مشورت درباره روش جایگزین است؛ شیردهی: دارو به شیر مادر منتقل می‌شود، مشورت با پزشک درباره مزایا و خطرات پیش از شیردهی لازم است",  # noqa: E501
            "مصرف همراه غذا": "باید همراه غذا مصرف شود؛ پرهیز از مصرف گریپ‌فروت یا آب آن مگر با اجازه پزشک، زیرا می‌تواند احتمال بروز عوارض جانبی را افزایش دهد",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب مگر دستور خاص وجود داشته باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "صرع و تشنج؛ نورالژی سه‌قلو (درد عصبی)",
            "طبقه‌بندی دارویی": "ضد صرع‌ها؛ طبقه‌بندی مارتیندل: سیستم‌های عصبی، داروهای ضد صرع",
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور بسیار مشخص و مفصل ذکر شده: شمارش کامل خون، سطوح مواد معدنی خون، عملکرد کلیه و کبد، معاینه چشم، سطح خونی کاربامازپین، به‌صورت دوره‌ای؛ همچنین ممکن است پیش از شروع درمان آزمایش ژنتیکی برای سنجش خطر واکنش‌های پوستی جدی توصیه شود",  # noqa: E501
            "تفاوت دوزهای دارویی": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "کند شدن تنفس، از دست دادن هوشیاری، انقباض عضلانی یا حرکات کنترل‌نشده، ضربان قلب بسیار شدید؛ در صورت بروز غش یا مشکل تنفسی فوراً با اورژانس تماس بگیرید یا به مراکز درمانی مراجعه کنید",  # noqa: E501
            "خطر ژنتیکی/قومیتی واکنش‌های پوستی شدید (معیار اضافه)": "در برخی گروه‌های قومی خاص، به‌ویژه افراد آسیایی و آسیای جنوبی، احتمال بروز واکنش‌های پوستی بسیار جدی یا کشنده با این دارو بیشتر است؛ ممکن است پزشک پیش از تجویز دارو آزمایش‌هایی برای سنجش این خطر توصیه کند و در صورت خطر بالاتر، مزایا و عوارض دارو باید با پزشک بررسی شود",  # noqa: E501
        },
    },
    "cetirizine": {
        "drug_name": "cetirizine",
        "source_file": "cetirizine.docx",
        "content": {
            "عنوان": "سیتریزین (Cetirizine)",
            "منبع": "daroobin.com/fa/drug/cetirizine",
            "نام ژنریک فارسی و انگلیسی": "سیتریزین (Cetirizine)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص ۵ میلی‌گرمی و ۱۰ میلی‌گرمی؛ شربت ۵ میلی‌گرم بر ۵ میلی‌لیتر؛ سافت ژل ۱۰ میلی‌گرمی",  # noqa: E501
            "موارد مصرف": "رینیت آلرژیک فصلی یا دائمی؛ التهاب آلرژیک ملتحمه؛ خارش (چشم، پوست، بینی)؛ کهیر؛ عطسه و آب‌ریزش بینی؛ آسم آلرژیک؛ سرفه‌های خارش‌آور ناشی از ترشح مخاط پشت گلو (مانند تب یونجه)",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولاً یک‌بار در روز طبق تجویز پزشک یا دستور بروشور؛ قرص جویدنی باید کامل جویده شود؛ قرص مکیدنی باید روی زبان حل شود و سپس با یا بدون آب بلعیده شود؛ شربت باید با پیمانه مخصوص (نه قاشق آشپزخانه) اندازه‌گیری شود؛ دوز بر اساس سن، وضعیت بیماری و پاسخ به درمان تعیین می‌شود و نباید خودسرانه افزایش یابد",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "خواب‌آلودگی، خستگی، خشکی دهان، درد معده (در کودکان)",
            "عوارض جدی و علائم خطر": "مشکل در دفع ادرار، ضعف؛ علائم آلرژی شدید (راش پوستی، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، تنگی نفس)؛ افزایش ضربان قلب و بی‌نظمی آن که برای بیماران قلبی خطرناک است؛ علائم غیرعادی کهیر هنگام استفاده برای درمان کهیر (رنگ غیرعادی، کبود یا تاول‌مانند، کهیر بدون خارش)",  # noqa: E501
            "تداخلات دارویی": "فهرست گسترده‌ای از داروها با تداخل، از جمله آکلیدینیوم، آزلاستین (بینی)، برومپریدول، سیمتروپیوم، الوکسادولین، گلیکوپیرولات، ایپراتروپیوم، لاسمیدیتان، لووسولپیرید، ارفنادرین و موارد متعدد دیگر؛ احتیاط با سایر داروهای خواب‌آور/اوپیوئیدی، الکل، ماری‌جوانا، داروهای ضداضطراب یا خواب (آلپرازولام، لورازپام، زولپیدم)، شل‌کننده‌های عضلات، و سایر آنتی‌هیستامین‌ها؛ عدم مصرف هم‌زمان با هیدروکسیزین یا لووسیتریزین (مشابهت دارویی)؛ تداخل با تست‌های آزمایشگاهی مانند تست پوستی آلرژی (نتایج کاذب)؛ فهرست جداگانه‌ای از داروهایی که اثر آن‌ها توسط سیتریزین کاهش یا افزایش می‌یابد و داروهایی که کارایی سیتریزین را افزایش می‌دهند",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به سیتریزین، هیدروکسیزین یا لووسیتریزین؛ سابقه مشکل در دفع ادرار (مانند بزرگی پروستات)، بیماری کلیوی، بیماری کبدی؛ احتیاط در بیماران قلبی به دلیل خطر افزایش و بی‌نظمی ضربان قلب؛ احتیاط در بیماران دیابتی (فرم‌های مایع ممکن است قند داشته باشند)؛ پرهیز از رانندگی و کار با ماشین‌آلات تا اطمینان از نبود گیجی؛ پرهیز از مصرف الکل و ماری‌جوانا",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مبرم و صلاحدید پزشک مصرف شود، مشورت با پزشک درباره مزایا و مضرات؛ شیردهی: دارو وارد شیر مادر می‌شود، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "ارائه نشده",
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ برندهای مختلف شرایط نگهداری متفاوتی دارند (مراجعه به بروشور یا داروساز)؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب، دور ریختن صحیح دارو پس از پایان درمان یا انقضا",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "رینیت آلرژیک فصلی یا دائمی، التهاب آلرژیک ملتحمه، خارش، کهیر، عطسه و آب‌ریزش از بینی، آسم آلرژیک",  # noqa: E501
            "طبقه‌بندی دارویی": "ضد آلرژی‌های تنفسی؛ طبقه‌بندی مارتیندل: آنتی‌هیستامین‌های سیستمیک، دستگاه تنفسی",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌صورت کلی ذکر شده که طی دوره درمان باید به‌طور منظم تحت معاینات پزشکی و آزمایشگاهی قرار گرفته شود؛ نوع دقیق آزمایش یا فاصله زمانی آن مشخص نشده است",  # noqa: E501
            "تفاوت دوزهای دارویی (۵ در برابر ۱۰ میلی‌گرم)": "در متن مشخص نشده؛ فقط دوزهای ۵ و ۱۰ میلی‌گرمی (و شکل‌های دارویی مرتبط) لیست شده‌اند، بدون توضیح تفاوت کاربردی یا بالینی بین این دو دوز",  # noqa: E501
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "خواب‌آلودگی شدید؛ در کودکان ممکن است پیش از خواب‌آلودگی، تغییرات خلقی-روانی (بی‌قراری، تحریک‌پذیری) بروز کند؛ در صورت بروز غش یا مشکل تنفسی، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "ciprofloxacin": {
        "drug_name": "ciprofloxacin",
        "source_file": "ciprofloxacin.docx",
        "content": {
            "عنوان": "سیپروفلوکساسین (Ciprofloxacin)",
            "منبع": "daroobin.com/fa/drug/ciprofloxacin",
            "نام ژنریک فارسی و انگلیسی": "سیپروفلوکساسین (Ciprofloxacin)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص در دوزهای ۲۵۰ و ۵۰ میلی‌گرم، ویال در دوز ۲۰۰ میلی‌گرم؛ در بخش مشخصات دارو، سوسپانسیون هم به عنوان یکی از اشکال دارو ذکر شده اما دوز آن مشخص نیست؛ توجه: دوز ۵۰ میلی‌گرم برای قرص با توجه به دوزهای رایج این دارو در بازار احتمالا اشتباه تایپی است و باید راستی‌آزمایی شود",  # noqa: E501
            "موارد مصرف": "درمان عفونت‌های باکتریایی سیستمیک؛ فقط برای بیماری‌های باکتریایی و بدون اثر بر عفونت‌های ویروسی مانند سرماخوردگی یا آنفولانزا",  # noqa: E501
            "نحوه مصرف": "خوراکی، با یا بدون غذا، معمولا دو بار در روز صبح و عصر؛ توصیه به عدم جویدن یا خرد کردن به دلیل تلخی دهان؛ فقط در صورت وجود خط تقسیم و توصیه پزشک قابل نصف کردن است؛ نوشیدن آب فراوان توصیه می‌شود؛ فاصله حداقل دو ساعت قبل یا شش ساعت بعد از داروهای متصل‌شونده مانند کوئیناپریل، سولامر، سوکرالفات، مکمل آهن یا روی و محصولات حاوی منیزیم، آلومینیوم یا کلسیم؛ همین فاصله باید با غذاهای کلسیمی مانند شیر، ماست یا آبمیوه غنی‌شده رعایت شود مگر همراه با وعده غیرکلسیمی مصرف شوند؛ مصرف منظم در زمان مشخص و تکمیل کامل دوره درمان حتی پس از بهبود اولیه توصیه می‌شود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر کمتر از شش ساعت به نوبت بعدی مانده باشد، از آن نوبت صرف‌نظر شود و دوز دو برابر نشود؛ در اوردوز همراه با بی‌هوشی یا تنگی نفس با اورژانس تماس گرفته شود",  # noqa: E501
            "عوارض شایع": "تهوع، اسهال، سرگیجه، سبکی سر، سردرد، تنگی نفس",
            "عوارض جدی و علائم خطر": "افزایش کبودی یا خونریزی، علائم عفونت جدید، علائم مشکلات کلیوی مانند تغییر یا رنگ غیرعادی ادرار، علائم مشکلات کبدی؛ سرگیجه شدید، غش، ضربان قلب نامنظم یا تند، علائم پارگی یا شکستگی آئورت شامل درد ناگهانی و شدید شکم، قفسه سینه یا کمر همراه با سرفه و تنگی نفس؛ خطر عفونت روده‌ای حاد ناشی از باکتری سی دیفیسیل حتی هفته‌ها پس از قطع دارو؛ سفیدک دهانی یا عفونت قارچی جدید در مصرف طولانی یا زیاد؛ علائم آلرژی حاد شامل راش پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و تنگی نفس؛ هشدار ویژه: آسیب حاد یا دائمی به تاندون‌ها مانند تندونیت یا پارگی تاندون، نوروپاتی محیطی و مشکلات سیستم عصبی شامل درد یا بی‌حسی یا سوزش یا گزگز یا ضعف در دست و پا، تغییر حس لامسه یا دما، سردرد شدید و مداوم، تغییرات بینایی، لرزش، تشنج و تغییرات خلقی یا روانی مانند آشفتگی، اضطراب، گیجی، توهم، افسردگی و به ندرت افکار خودکشی؛ تشدید احتمالی میاستنی گراویس با ضعف عضلات جدید یا افتادگی پلک",  # noqa: E501
            "تداخلات دارویی": "رقیق‌کننده‌های خون مانند آسنوکومارول و وارفارین، استرانسیوم؛ داروهای طولانی‌کننده فاصله کیو. تی مانند آمیودارون، دوفتیلید، کوئینیدین، پروکائین‌آمید و سوتالول؛ داروهایی که سیپروفلوکساسین بر دفع آنها اثر می‌گذارد مانند دولوکستین، فلیبانسرین، لومیتاپید، پیرفنیدون، تاسیملتئون و تیزانیدین؛ افزایش اثر یا ماندگاری کافئین در بدن؛ کاهش اثر داروهای کاهنده قند خون، واکسن بی سی جی و واکسن وبا، دیدانوزین، فوس فنی توئین، فنی توئین، فرآورده های تیروئید و واکسن حصبه توسط سیپروفلوکساسین؛ کاهش اثر سیپروفلوکساسین توسط آنتی اسیدها، نمک های کلسیم، دیدانوزین، فرآورده های آهن، نمک های منیزیم، مولتی ویتامین های معدنی، کیناپریل، سولامر و سوکرالفیت؛ افزایش اثر تعداد زیادی داروی دیگر توسط سیپروفلوکساسین از جمله کاربامازپین، کلوزاپین، متوترکسات، پیموزاید، پروپرانولول، سیلدنافیل، تریازولام، وارنیکلین و آنتاگونیست های ویتامین کا؛ افزایش کارایی سیپروفلوکساسین توسط کورتیکواستروئیدهای سیستمیک، متیل فنیدات، پروبنسید و اسپیرونولاکتون؛ تداخلات پرهیزی کامل با آگوملاتین، بی سی جی داخل مثانه ای، واکسن وبا، لومیتاپید، پیموزاید و تیزانیدین",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به سیپروفلوکساسین یا سایر آنتی بیوتیک های کوئینولون مانند نورفلوکساسین، لووفلوکساسین یا موکسی فلوکساسین؛ سابقه دیابت، مشکلات قلبی، مشکلات تاندون یا مفصل، بیماری کلیوی، بیماری کبدی، اختلالات خلقی یا روانی، میاستنی گراویس، مشکلات عصبی، تشنج، مشکلات عروقی، فشار خون بالا و سندروم مارفان یا اهلر دانلوس؛ افزایش خطر طولانی شدن کیو. تی در بیماری قلبی، سابقه خانوادگی، کمبود پتاسیم یا منیزیم و مصرف داروهای ادرارآور؛ احتیاط در دیابت به دلیل احتمال تغییر قند خون و تشدید اثر گلیبورید؛ احتمال گیجی و پرهیز از رانندگی، الکل و ماری جوانا؛ احتمال حساسیت به آفتاب؛ کاهش اثر واکسن های زنده باکتریایی؛ اطلاع پیش از عمل جراحی؛ احتیاط بیشتر در کودکان به دلیل خطر مشکلات تاندون و در سالمندان به دلیل خطر بیشتر پارگی تاندون، طولانی شدن کیو. تی و پارگی آئورت",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت ضرورت مصرف شود و مزایا و مضرات با پزشک بررسی شود؛ شیردهی: دارو وارد شیر مادر می‌شود و ممکن است به نوزاد آسیب برساند، مشورت با پزشک پیش از شیردهی لازم است",  # noqa: E501
            "مصرف همراه غذا": "با یا بدون غذا قابل مصرف است؛ اما تغذیه انترال از طریق لوله باعث کاهش جذب دارو می‌شود و در این حالت نباید تجویز شود؛ فاصله حداقل دو ساعت قبل یا شش ساعت بعد از غذاهای کلسیمی مانند شیر یا ماست رعایت شود",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ در حمام نگهداری نشود؛ دور از دسترس کودکان و حیوانات خانگی؛ داروها در فاضلاب تخلیه نشوند مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "درمان عفونت باکتریایی، این بار برخلاف آسپیرین و آزیترومایسین در بخش مشخصات دارو ذکر شده است",  # noqa: E501
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: آنتی‌بایوتیک‌ها. طبقه‌بندی مارتیندل: ضدعفونت برای استفاده سیستمی",  # noqa: E501
            "هشدار جعبه‌ای درباره تاندون و اعصاب": "آنتی‌بیوتیک‌های کوئینولون از جمله این دارو دارای یک هشدار رسمی و مجزا درباره خطر آسیب حاد یا دائمی تاندون‌ها و مشکلات عصبی محیطی هستند که در صفحه به‌طور مستقل از سایر عوارض برجسته شده است",  # noqa: E501
            "توصیه عدم تغییر برند دارو": "بدون هماهنگی با پزشک یا داروساز نباید برند مصرفی تغییر کند، زیرا برندهای مختلف این دارو تاثیرات متفاوتی دارند",  # noqa: E501
            "تست‌های آزمایشگاهی دوره‌ای": "در طول درمان توصیه به انجام دوره‌ای تست عملکرد کلیه، سطح گلبول‌های خون و کشت می‌شود",  # noqa: E501
        },
    },
    "clopidogrel": {
        "drug_name": "clopidogrel",
        "source_file": "clopidogrel.docx",
        "content": {
            "عنوان": "کلوپیدوگرل (Clopidogrel)",
            "منبع": "daroobin.com/fa/drug/clopidogrel",
            "نام ژنریک فارسی و انگلیسی": "کلوپیدوگرل (Clopidogrel)",
            "نام‌های تجاری ایرانی": "زیلت (تولیدکننده: اکتاوکو)",
            "شکل و قدرت دارویی": "قرص ۷۵ میلی‌گرم",
            "موارد مصرف": "پیشگیری از بیماری‌های گردش خون، حمله قلبی، سکته مغزی، درد قفسه سینه و انسداد عروق، از جمله پس از برخی جراحی‌ها مانند ایمپلنت استنت",  # noqa: E501
            "نحوه مصرف": "خوراکی، با یا بدون غذا، معمولاً یک‌بار در روز طبق تجویز پزشک؛ مصرف منظم و در ساعت مشخص برای بیشترین اثرگذاری؛ ممکن است پس از استنت یا جراحی، هم‌زمان با آسپیرین برای چندین ماه یا سال (بسته به نوع جراحی) تجویز شود؛ عدم قطع خودسرانه حتی در صورت بهبود؛ ممکن است پزشک توصیه کند حداقل ۵ روز قبل از جراحی مصرف متوقف شود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی نزدیک بود، از مصرف نوبت فراموش‌شده خودداری شده و دوز بعدی در زمان معمول (بدون دوبرابر کردن) مصرف شود",  # noqa: E501
            "عوارض شایع": "افزایش خونریزی یا کبودی، درد یا ناراحتی شکم، اسهال، یبوست",
            "عوارض جدی و علائم خطر": "خونریزی شدید در شکم، زیرشکم، چشم یا مغز؛ پورپورای ترومبوتیک ترومبوسیتوپنیک (TTP) با علائمی مانند درد شدید شکم/زیرشکم، خونریزی شدید بینی یا لثه، مدفوع خونی یا سیاه، گیجی، تب، رنگ‌پریدگی شدید، لکه بنفش روی پوست، غش، تپش تند قلب، سردرد شدید ناگهانی، خستگی یا ضعف ناگهانی، استفراغ خونی یا شبیه تفاله قهوه، اختلال تکلم، تغییرات بینایی، تشنج، زردی پوست یا چشم، ادرار تیره/صورتی/قرمز/خونی، علائم مشکلات کلیه؛ علائم حمله قلبی یا سکته جدید (درد دست چپ/فک/قفسه سینه، تنگی نفس، تعرق غیرعادی، ضعف در یک طرف بدن، اختلال تکلم، تغییرات ناگهانی بینایی، گیجی)؛ واکنش آلرژیک جدی (بثورات، خارش/تورم صورت-زبان-گلو، سرگیجه شدید، تنگی نفس)",  # noqa: E501
            "تداخلات دارویی": "تیپراناویر؛ آسپیرین (تنها با تجویز و نظارت دقیق پزشک - مصرف خودسرانه هم‌زمان با کلوپیدوگرل تداخل شدید و خطر خونریزی جدی دارد)؛ داروهای مؤثر بر حذف کلوپیدوگرل از بدن (مهارکننده‌های پمپ پروتون مانند امپرازول و اسومپرازول، سایمتیدین، اتراویرین، فلبامات، فلوکونازول، فلوکستین، فلووکسامین، کتوکونازول، ریفامپین، تیکلوپیدین، وریکونازول)؛ کلوپیدوگرل حذف برخی داروها از بدن را کند می‌کند (داسابوویر، رپاگلینید)؛ NSAIDها (ایبوپروفن، ناپروکسن، آسپرین) که خطر خونریزی را افزایش می‌دهند؛ تداخل غذایی با گریپ‌فروت (کاهش اثر ضدپلاکتی)؛ مشخصات کلی تداخل: سوبسترای CYP2C19 و CYP3A4، مهارکننده ضعیف CYP2B6/CYP2C19/OATP1B1، مهارکننده متوسط CYP2C8، مهارکننده کربوکسیل‌استراز ۱، تحت تأثیر تأخیر تخلیه معده، اتصال به رزین‌های تعویض کاتیونی، حساس به pH معده؛ تداخلات پرهیزی: آمودیاکین، ریمگپنت، پازوپانیب، توپوتکان، اوروکیناز",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به کلوپیدوگرل یا سایر تیئنوپیریدین‌ها (مانند پراسوگرل)؛ سابقه خونریزی (زخم معده، خونریزی چشم یا مغز)، جراحی اخیر، تروما یا آسیب جدی، بیماری کبدی، اختلالات خونریزی (مانند هموفیلی)؛ احتیاط با اشیاء تیز (تیغ، ناخن‌گیر) و پرهیز از ورزش‌های تماسی؛ اطلاع‌رسانی به پزشک یا دندانپزشک پیش از هر جراحی (احتمال نیاز به قطع دارو حداقل ۵ روز قبل)؛ پرهیز از الکل (افزایش خطر خونریزی معده)؛ پرهیز از گریپ‌فروت یا آب آن",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: تنها در صورت ضرورت کامل مصرف شود، مشورت با پزشک درباره مزایا و معایب لازم است؛ شیردهی: انتقال دارو از طریق شیر مادر ناشناخته است، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با یا بدون غذا مصرف شود؛ از مصرف گریپ‌فروت یا آب آن پرهیز شود، زیرا مصرف روزانه حدود ۳ لیوان (۲۰۰ میلی‌لیتری) آب گریپ‌فروت می‌تواند اثر ضدپلاکتی دارو را کاهش دهد",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت، خارج از محیط حمام؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در توالت یا سینک مگر توصیه شده باشد؛ دور ریختن دارو پس از پایان مصرف یا رسیدن تاریخ انقضا",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "حمله قلبی، سکته مغزی، درد قفسه سینه، انسداد عروق",
            "طبقه‌بندی دارویی": "داروهای ضد انعقاد خون (طبقه‌بندی سایت)؛ طبقه‌بندی مارتیندل: ضد ترومبوتیک‌ها، خون و اندام‌های خون‌ساز؛ از دسته داروهای ضدپلاکت",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور کلی ذکر شده: انجام منظم آزمایش خون کامل برای بررسی فرایند درمان یا عوارض جانبی توصیه شده است",  # noqa: E501
            "تفاوت دوزهای دارویی": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "بی‌هوشی یا تنگی نفس؛ در صورت بروز این علائم فوراً با اورژانس تماس بگیرید، در غیر این صورت با پزشک تماس بگیرید",  # noqa: E501
        },
    },
    "diclofenac": {
        "drug_name": "diclofenac",
        "source_file": "diclofenac.docx",
        "content": {
            "عنوان": "دیکلوفناک (Diclofenac)",
            "منبع": "daroobin.com/fa/drug/diclofenac",
            "نام ژنریک فارسی و انگلیسی": "دیکلوفناک، دیکلوفناک سدیم (diclofenac)",
            "نام‌های تجاری ایرانی": "دیکلوفناک تهران شیمی (تهران شیمی)",
            "شکل و قدرت دارویی": "آمپول، قرص، شیاف؛ دوزها: ۵۰ و ۱۰۰ میلی‌گرم",
            "موارد مصرف": "آرتریت (التهاب مفاصل)، رگ‌به‌رگ شدن عضلات یا ضرب‌دیدگی استخوان، نقرس، میگرن، درد دندان، دردهای پس از عمل جراحی، کمردرد، دردهای استخوانی، درد مفصلی، دیسک کمر، کنترل تب، درد ناشی از گرفتگی، سردرد میگرنی، گرفتگی عضلات، گردن‌درد، زانودرد، درد کلیه، درد ناشی از ترک اعتیاد",  # noqa: E501
            "نحوه مصرف": "با یک لیوان کامل آب و بدون خرد کردن یا جویدن بلعیده شود؛ حداقل ۱۰ دقیقه بعد از مصرف دراز نکشید؛ در صورت ناراحتی معده با غذا، شیر یا آنتی‌اسید مصرف شود اما جذب کندتر و تسکین دیرتر رخ می‌دهد؛ بروز اولین اثر ممکن است تا دو هفته طول بکشد؛ دیکلوفناک پتاسیم شروع اثر سریع‌تر و طول اثر کمتر دارد (در معده باز می‌شود)؛ دیکلوفناک سدیم شروع اثر آهسته‌تر و طول اثر بیشتر دارد (در دئودنوم باز می‌شود)",  # noqa: E501
            "فراموشی یک نوبت مصرف": "در صورت فراموشی، به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده باشد، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی در زمان معمول مصرف شود؛ علائم اوردوز: بی‌هوشی، تنگی نفس، درد شدید معده، استفراغ شبیه تفاله قهوه، خواب‌آلودگی شدید، تنفس آهسته یا کم‌عمق، تشنج؛ در این موارد با اورژانس تماس گرفته شود",  # noqa: E501
            "عوارض شایع": "ناراحتی معده، تهوع، سوزش سردل، اسهال، یبوست، گاز معده، سردرد، خواب‌آلودگی، سرگیجه",  # noqa: E501
            "عوارض جدی و علائم خطر": "تغییرات شنوایی (زنگ گوش)، تغییرات ذهنی یا خلقی، بلع دشوار یا دردناک، علائم نارسایی قلبی (تورم مچ پا)، خستگی غیرعادی، افزایش وزن ناگهانی، علائم مشکلات کلیوی، گرفتگی گردن بدون دلیل؛ علائم آلرژی جدی: بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید، مشکل تنفسی؛ افزایش فشار خون؛ علائم آسیب کبدی: تهوع یا استفراغ مزمن، بی‌اشتهایی، درد شکم، زردی چشم یا پوست، ادرار تیره؛ حساسیت بیشتر به نور آفتاب",  # noqa: E501
            "تداخلات دارویی": "تداخل رده X که باید کاملاً پرهیز شود: آسه‌متاسین، آمینولوولینیک اسید سیستمیک، دفریپرون، دکس‌ایبوپروفن، دکس‌کتوپروفن، فلوکتافنین، کتورولاک نازال و سیستمیک، ماسیمورلین، میفامورتید، مورنیفلومات، سایر داروهای ضدالتهاب غیراستروئیدی انتخابی سیکلواکسیژناز نوع ۲، اوماستاکسین، پلوبیپروفن، فنیل‌بوتازون، تالنیفلومات، تنوکسیکام، اوروکیناز، زالتوپروفن؛ سایر تداخلات: آلیسکیرن، مهارکننده‌های آنزیم تبدیل‌کننده آنژیوتانسین مانند کاپتوپریل و لیزینوپریل، مسدودکننده‌های گیرنده آنژیوتانسین مانند والزارتان و لوزارتان، کورتیکواستروئیدها مانند پردنیزون، سیدوفوویر، لیتیوم، متوترکسات، داروهای ادرارآور مانند فوروزماید؛ احتمال تداخل با آزمایش‌های آزمایشگاهی و ایجاد نتایج نادرست؛ آسپرین با دوز پایین قلبی نباید قطع شود؛ سایر مسکن‌ها مانند سلکوکسیب، ایبوپروفن و کتورولاک خطر عارضه را افزایش می‌دهند؛ داروهای افزایش‌دهنده خونریزی مانند کلوپیدوگرل، دابیگاتران، انوکساپارین و وارفارین خطر را بیشتر می‌کنند.",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به دیکلوفناک، آسپرین یا سایر داروهای ضدالتهاب غیراستروئیدی باید به پزشک اطلاع داده شود؛ سابقه پزشکی مهم شامل آسم همراه با سابقه بدتر شدن تنفس پس از مصرف آسپرین یا داروهای ضدالتهاب غیراستروئیدی، مشکلات خونریزی یا لخته شدن خون، پولیپ بینی، بیماری قلبی، فشار خون بالا، بیماری کبدی، سکته مغزی و مشکلات معده، روده یا مری است؛ احتمال بروز سرگیجه پس از مصرف وجود دارد و باید از رانندگی و کار با ماشین‌آلات پرهیز شود؛ از مصرف الکل و ماری‌جوانا پرهیز شود؛ به دلیل افزایش خطر خونریزی، از مصرف الکل و تنباکو نیز پرهیز شود؛ در سالمندان احتیاط بیشتری لازم است.",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: از هفته ۲۰ تا زمان زایمان توصیه نمی‌شود و ممنوع است؛ در صورت تجویز پزشک بین هفته ۲۰ تا ۳۰، باید کمترین دوز موثر در کوتاه‌ترین زمان مصرف شود؛ شیردهی: دارو به شیر مادر منتقل می‌شود، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "برای جلوگیری از ناراحتی معده می‌توان با غذا، شیر یا آنتی‌اسید مصرف کرد، اما در این حالت جذب دارو کندتر شده و تسکین درد دیرتر اتفاق می‌افتد",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق و دور از نور، رطوبت و دور از دسترس کودکان و حیوانات خانگی نگهداری شود؛ برای دفع دارو نباید در توالت یا فاضلاب ریخته شود، در صورت نیاز به دفع باید با داروساز یا شرکت دفع زباله محلی مشورت شود",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "درد و التهاب",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای ضدالتهاب و ضددرد. طبقه‌بندی مارتیندل: محصولات ضدالتهابی و ضدروماتیسمی غیراستروئیدی؛ داروهای مسکن_ضدالتهاب و ضدتب‌ها",  # noqa: E501
            "کاربرد در ترک اعتیاد": "دیکلوفناک به عنوان مسکن غیرمخدر برای کاهش دردهای متوسط دوران ترک اعتیاد کاربرد دارد؛ توصیه می‌شود ترک اعتیاد حتماً زیر نظر متخصص و با داروهای تهیه‌شده از مراکز معتبر و دارای مجوز انجام شود",  # noqa: E501
            "تست‌های آزمایشگاهی دوره‌ای": "در طول درمان ممکن است نیاز به بررسی دوره‌ای فشار خون، شمارش کامل خون و آزمایش عملکرد کبد و کلیه باشد",  # noqa: E501
            "فرم‌های اضافی دارو": "علاوه بر قرص، آمپول و شیاف، دیکلوفناک به شکل ژل موضعی و قطره چشمی نیز موجود است",  # noqa: E501
        },
    },
    "digoxin": {
        "drug_name": "digoxin",
        "source_file": "digoxin.docx",
        "content": {
            "عنوان": "دیگوکسین (Digoxin)",
            "منبع": "daroobin.com/fa/drug/digoxin",
            "نام ژنریک فارسی و انگلیسی": "دیگوکسین (Digoxin)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "الگزیر و قطره خوراکی ۰.۵ میلی‌گرم؛ قرص ۰.۱۲۵ و ۰.۲۵ میلی‌گرم؛ آمپول ۰.۲۵ و ۰.۵ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "نارسایی قلبی (معمولاً همراه با سایر داروها)؛ درمان انواع خاصی از ضربان قلب نامنظم مانند فیبریلاسیون دهلیزی مزمن",  # noqa: E501
            "نحوه مصرف": "خوراکی، یک‌بار در روز، با یا بدون غذا طبق تجویز پزشک؛ برای فرم مایع، استفاده از قطره‌چکان مخصوص بسته (نه قاشق معمولی) برای اندازه‌گیری دقیق دوز؛ مصرف حداقل ۲ ساعت قبل یا بعد از غذاهای پرفیبر (مانند سبوس)؛ فاصله حداقل ۲ ساعت با داروهای کلستیرامین، کلستیپول یا پسیلیوم؛ فاصله زیاد با آنتی‌اسیدها، کائولین پکتین، شیر منیزیم، متوکلوپرامید، سولفاسالازین یا آمینوسالیسیلیک اسید؛ دوز بر اساس وضعیت پزشکی، سن، وزن بدن، آزمایش‌های آزمایشگاهی و پاسخ به درمان تعیین می‌شود؛ مصرف منظم در ساعت مشخص؛ عدم قطع ناگهانی بدون تجویز پزشک",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر نزدیک به نوبت بعد بود، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، سردرد، سرگیجه، از دست دادن اشتها، اسهال",
            "عوارض جدی و علائم خطر": "ضعف؛ تغییرات ذهنی یا خلقی؛ تغییرات بینایی (مانند دید تاری یا دید زرد یا سبز)؛ بزرگ شدن سینه‌ها در مردان؛ بروز ضربان قلب غیرعادی جدید (سریع، آهسته یا نامنظم)؛ واکنش آلرژیک جدی (بثورات پوستی، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "داروهای مؤثر بر حذف دیگوکسین از بدن: ضدقارچ‌های آزول (مانند ایتراکونازول)، دروندارون، لاپاتینیب، آنتی‌بیوتیک‌های ماکرولید (کلاریترومایسین، اریترومایسین)، پروپافنون، ریفامپین، مخمر سنت جان؛ احتیاط با محصولاتی که ممکن است نارسایی قلبی را بدتر کنند (محصولات ضدسرفه و سرماخوردگی، کمک‌های رژیمی، NSAIDها مانند ایبوپروفن یا ناپروکسن)؛ سطح خونی کلسیم، منیزیم و پتاسیم بر عملکرد دارو تأثیر می‌گذارد و داروهای ادرارآور (مدر) ممکن است این تعادل را بر هم بزنند",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به دیگوکسین یا داروهای مشابه (مانند دیژیتوکسین)؛ سابقه مشکلات کلیوی؛ مشکلات تیروئید (کم‌کاری یا پرکاری)؛ احتیاط در رانندگی و کار با ماشین‌آلات تا رفع سرگیجه یا تاری دید؛ پرهیز از الکل و ماری‌جوانا؛ توجه به عدم تعادل مواد معدنی (کلسیم، منیزیم، پتاسیم) به‌ویژه در مصرف‌کنندگان داروهای ادرارآور؛ اطلاع‌رسانی به پزشک یا دندانپزشک پیش از جراحی یا روش‌های خاص قلبی مانند کاردیوورژن الکتریکی؛ نوزادان و کودکان ممکن است نسبت به اثرات این دارو بر ضربان قلب حساسیت بیشتری داشته باشند",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز و با تشخیص و تجویز پزشک مصرف شود؛ شیردهی: دارو به شیر مادر منتقل می‌شود، هیچ گزارشی از آسیب به نوزاد شیرخوار ثبت نشده، اما مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با یا بدون غذا مصرف شود؛ باید حداقل ۲ ساعت قبل یا بعد از غذاهای پرفیبر (مانند سبوس) مصرف شود، زیرا فیبر بالا می‌تواند جذب دارو را کاهش دهد",  # noqa: E501
            "شرایط نگهداری": "برندهای مختلف نیاز به نگهداری متفاوتی دارند؛ بررسی راهنمای محصول یا پرسش از داروساز توصیه می‌شود؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب یا سینک؛ دور ریختن صحیح دارو پس از پایان مصرف یا انقضا",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "نارسایی قلبی؛ ضربان قلب غیرعادی (فیبریلاسیون دهلیزی مزمن)",  # noqa: E501
            "طبقه‌بندی دارویی": "داروهای ضد آریتمی؛ طبقه‌بندی مارتیندل: داروهای قلبی؛ از دسته گلیکوزیدهای قلبی",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور ضمنی اشاره شده: دوز بر اساس آزمایش‌های آزمایشگاهی و پاسخ به درمان تعیین می‌شود و توجه به سطح الکترولیت‌ها (کلسیم، منیزیم، پتاسیم) توصیه شده است؛ نام آزمایش خاص (مانند سطح خونی دیگوکسین) یا فاصله زمانی دقیق ذکر نشده است",  # noqa: E501
            "تفاوت دوزهای دارویی": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "بحران‌های پزشکی مانند غش و مشکل تنفسی؛ در صورت بروز، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
            "تأثیر تعادل الکترولیتی بر ایمنی دارو (معیار اضافه)": "سطح مواد معدنی طبیعی خون، به‌ویژه کلسیم، منیزیم و پتاسیم، بر نحوه عملکرد و ایمنی این دارو تأثیر می‌گذارد؛ داروهای ادرارآور (مدر) ممکن است این تعادل را بر هم بزنند، بنابراین در صورت مصرف چنین داروهایی یا سابقه عدم تعادل مواد معدنی، باید با پزشک درباره نیاز به مکمل یا رژیم غذایی خاص مشورت شود",  # noqa: E501
        },
    },
    "famotidine": {
        "drug_name": "famotidine",
        "source_file": "famotidine.docx",
        "content": {
            "عنوان": "فاموتیدین (Famotidine)",
            "منبع": "daroobin.com/fa/drug/famotidine",
            "نام ژنریک فارسی و انگلیسی": "فاموتیدین (Famotidine)",
            "نام‌های تجاری ایرانی": "مداتیدین، فاموتد، فاموتیدین شفا، فاموسد، فاموور",
            "شکل و قدرت دارویی": "آمپول، قرص، قرص جویدنی؛ قرص فاموتیدین ۴۰ و ۲۰ میلی‌گرم؛ آمپول فاموتیدین ۲۰ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "رفلاکس اسید معده و معده‌درد عصبی؛ زخم معده و زخم خوش‌خیم دوازدهه (اثنی‌عشر)؛ سندروم زولینگر-الیسون",  # noqa: E501
            "نحوه مصرف": "دوز بر اساس جدول دارو: بزرگسالان ۴۰ میلی‌گرم یک‌بار در روز قبل از خواب یا ۲۰ میلی‌گرم دوبار در روز با فاصله ۱۲ ساعت؛ کودکان ۵ میلی‌گرم به ازای هر کیلوگرم وزن بدن در روز، یک‌بار موقع خواب؛ حتماً با تجویز پزشک؛ مصرف بیش از ۸ هفته معمولاً ضروری نیست و نیازمند مشورت با پزشک است",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی در زمان منظم (بدون دوبرابر کردن) مصرف شود",  # noqa: E501
            "عوارض شایع": "سردرد، یبوست، اسهال",
            "عوارض جدی و علائم خطر": "کبودی یا خونریزی آسان؛ علائم عفونت (گلودرد مقاوم، تب، لرز)؛ تغییرات روانی/خلقی (بی‌قراری، گیجی، افسردگی، توهم)؛ ضربان قلب سریع، آهسته یا نامنظم؛ سرگیجه شدید، غش، تشنج؛ علائم آلرژی جدی (بثورات، خارش یا تورم صورت/زبان/گلو، مشکل تنفسی)؛ علائم بیماری کبدی جدی (تهوع/استفراغ مزمن، بی‌اشتهایی، درد شکم، زردی چشم یا پوست، ادرار تیره)؛ علائم هشداردهنده قلبی/گوارشی به‌ویژه در خودمصرفی: مشکل بلع، استفراغ خونی یا شبیه تفاله قهوه، مدفوع خونی یا سیاه، سوزش سردل بیش از ۳ ماه، درد مکرر قفسه سینه، خس‌خس سینه مکرر، درد قفسه سینه/فک/بازو/شانه همراه با تنگی نفس یا تعریق غیرمعمول، کاهش وزن بدون دلیل",  # noqa: E501
            "تداخلات دارویی": "تداخلات پرهیزی: سفوروکسیم، داساتینیب، دلاویردین، دی‌کلروفنامید، پازوپانیب، ریزدرونات؛ داروهایی که جذبشان به اسید معده نیاز دارد و ممکن است تحت‌تأثیر قرار گیرند (فهرست طولانی از جمله آتازاناویر، ایتراکونازول، کتوکونازول، فراورده‌های آهن، مولتی‌ویتامین‌ها و دیگر موارد)؛ داروهایی که اثرشان توسط فاموتیدین افزایش می‌یابد (دکس‌متیل‌فنیدات، ایتراکونازول، متیل‌فنیدات، ریزدرونات، ساکوییناویر، وارنیکلین)؛ ممنوعیت مصرف هم‌زمان با سایر مسدودکننده‌های H2 (سایمتیدین، نیزاتیدین، رانیتیدین)؛ کاهش اثر با آنتی‌اسیدها (شربت‌های معده)",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به فاموتیدین یا سایر مسدودکننده‌های H2 (سایمتیدین، رانیتیدین)؛ سابقه مشکلات کلیوی، کبدی یا تومور معده؛ فنیل‌کتونوری (به دلیل آسپارتام در قرص جویدنی)؛ عدم استفاده برای درمان رفلاکس در کودکان زیر ۱۲ سال؛ احتیاط بیشتر در سالمندان (گیجی، خستگی غیرعادی، خطر سقوط)؛ مصرف بیش از ۸ هفته نیازمند مشورت پزشک؛ مصرف طولانی‌مدت بیش از ۲ سال ممکن است به کمبود ویتامین B12 منجر شود؛ تغییرات سبک زندگی (کاهش استرس، ترک سیگار، محدود کردن الکل و رژیم غذایی) ممکن است اثربخشی درمان را افزایش دهد",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت ضرورت و با تجویز پزشک مصرف شود؛ شیردهی: دارو به شیر مادر منتقل می‌شود، مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "مصرف هم‌زمان با آنتی‌اسیدها (شربت‌های معده مانند آلومینیوم ام‌جی‌اس، منیزیم هیدروکساید) اثر دارو را کاهش می‌دهد؛ پرخوری و غذای چرب اثر درمانی را کم می‌کند؛ محدود کردن گوجه‌فرنگی، نعنا، شکلات، غذاهای پرادویه، نوشیدنی‌های گرم کافئین‌دار و الکلی؛ محدود کردن غذاهای آبکی/مایعات و پرهیز از چای یا سایر نوشیدنی‌ها تا یک ساعت پس از غذا و پرهیز از دراز کشیدن پس از غذا",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ دور از دسترس کودکان و حیوانات خانگی؛ دور ریختن صحیح دارو در صورت انقضا یا عدم نیاز (نه در توالت یا فاضلاب)",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "رفلاکس اسید معده (GERD) و معده‌درد عصبی؛ زخم معده و زخم اثنی‌عشر؛ سندروم زولینگر-الیسون",  # noqa: E501
            "طبقه‌بندی دارویی": "داروهای اختلالات اسیدی معده؛ طبقه‌بندی مارتیندل: آنتاگونیست‌های گیرنده H2، داروهای دستگاه گوارش، داروهای زخم پپتیک و بیماری رفلاکس معده-مری (GORD)",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌صورت کلی اشاره شده که ممکن است برخی تست‌ها برای بررسی روند درمان و عوارض جانبی لازم باشد و باید با پزشک مشورت و در زمان مقرر انجام شود؛ نوع دقیق آزمایش یا فاصله زمانی آن ذکر نشده است",  # noqa: E501
            "تفاوت دوزهای دارویی (۲۰ در برابر ۴۰ میلی‌گرم)": "بزرگسالان: ۴۰ میلی‌گرم یک‌بار در روز قبل از خواب، یا ۲۰ میلی‌گرم دو بار در روز با فاصله ۱۲ ساعت؛ کودکان: ۵ میلی‌گرم به ازای هر کیلوگرم وزن بدن در روز، یک‌بار موقع خواب؛ همه موارد باید با تجویز پزشک باشد",  # noqa: E501
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "علائم جدی مانند بی‌هوشی یا تنگی نفس؛ در صورت بروز، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "gliclazide": {
        "drug_name": "gliclazide",
        "source_file": "gliclazide.docx",
        "content": {
            "عنوان": "گلی کلازید (Gliclazide)",
            "منبع": "daroobin.com/fa/drug/gliclazide",
            "نام ژنریک فارسی و انگلیسی": "گلی کلازید (Gliclazide)",
            "نام‌های تجاری ایرانی": "کلازکس (ایران هورمون)، گلیکوور (اکتوورکو)",
            "شکل و قدرت دارویی": "قرص در دوزهای ۳۰، ۶۰ و ۸۰ میلی‌گرم",
            "موارد مصرف": "کنترل قند خون بالا در دیابت نوع ۲ همراه با رژیم غذایی و ورزش؛ قابل مصرف به تنهایی یا همراه سایر داروهای دیابت؛ برای دیابت نوع ۱ مناسب نیست",  # noqa: E501
            "نحوه مصرف": "روزانه یکبار، سی دقیقه قبل از صبحانه یا اولین وعده غذایی مصرف شود؛ در دوزهای بالاتر ممکن است دو بار در روز تجویز شود، صبح همراه صبحانه و عصر همراه غذا؛ دوز بر اساس پاسخ بیمار تنظیم می‌شود و معمولا با دوز کم شروع می‌شود؛ مصرف منظم در ساعت مشخص توصیه می‌شود؛ گلی کلازید شصت میلی‌گرم آهسته‌رهش با مصرف یکبار روزانه اثربخشی بیشتری نسبت به گلی کلازید هشتاد میلی‌گرم با مصرف دوبار روزانه دارد؛ در صورت تعویض از سایر داروهای دیابتی مانند کلرپروپامید، باید دقیقا طبق دستور پزشک عمل شود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر نزدیک نوبت بعد بود، نادیده گرفته شده و دوز دو برابر نشود؛ در اوردوز همراه با غش یا مشکل تنفسی با اورژانس تماس گرفته شود؛ مصرف بیش از حد می‌تواند باعث اسیدوز لاکتیک شود و سایر علائم آن شامل خواب‌آلودگی شدید، تهوع، استفراغ، اسهال شدید، تنفس سریع و ضربان قلب آهسته یا نامنظم است",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، از دست دادن اشتها، اسهال، یبوست، ناراحتی معده، سردرد، افزایش وزن",  # noqa: E501
            "عوارض جدی و علائم خطر": "علائم عفونت مانند گلودرد مداوم یا تب، خونریزی یا کبودی، درد معده، زردی چشم یا پوست، ادرار تیره، خستگی یا ضعف غیرمعمول، افزایش وزن ناگهانی، تغییرات ذهنی یا خلقی، تورم دست یا پا، تشنج؛ علائم قند خون پایین شامل تعریق ناگهانی، لرزش، ضربان قلب سریع، گرسنگی، تاری دید، سرگیجه و گزگز دست یا پا؛ علائم قند خون بالا شامل تشنگی، افزایش ادرار، گیجی، خواب‌آلودگی، گرگرفتگی، تنفس سریع و بوی بد دهان؛ حساسیت بیشتر به آفتاب؛ علائم آلرژی جدی شامل بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و مشکلات تنفسی؛ واکنش نادر شبه دی‌سولفیرام با الکل همراه با گرگرفتگی صورت، تهوع، استفراغ، سرگیجه و درد معده",  # noqa: E501
            "تداخلات دارویی": "داروهای مسدودکننده بتا مانند متوپرولول، پروپرانولول و قطره‌های چشمی گلوکوم مانند تیمولول ممکن است علائم قند خون پایین را پنهان کنند؛ داروی کولسولام جذب گلی کلازید را کاهش می‌دهد و باید حداقل چهار ساعت قبل از آن مصرف شود؛ توصیه به بررسی برچسب داروهای بدون نسخه مانند داروهای ضدسرفه و سرماخوردگی که ممکن است بر قند خون اثر بگذارند؛ تداخلات پرهیزی شامل آمینولوولینیک اسید سیستمیک، مکامیلامین و میتی‌گلینید",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به گلی کلازید یا محصولات مشابه؛ سابقه بیماری کبدی، بیماری کلیوی، بیماری تیروئید، شرایط هورمونی خاص مانند نارسایی غده فوق کلیوی یا هیپوفیز و سندرم ترشح نامناسب هورمون ضدادرار، و عدم تعادل الکترولیت مانند هیپوناترمی؛ اطلاع پیش از جراحی یا اقدامات دندانپزشکی؛ احتیاط در قند خون پایین یا بالا و پرهیز از رانندگی تا اطمینان از سازگاری دارو؛ محدود کردن مصرف الکل به دلیل خطر قند خون پایین و واکنش شبه دی‌سولفیرام؛ احتیاط در برابر نور آفتاب؛ احتیاط در شرایط استرس بدنی مانند تب، عفونت، جراحت یا جراحی؛ احتیاط بیشتر در سالمندان به‌ویژه برای قند خون پایین",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود، ممکن است پزشک حداقل دو هفته قبل از زایمان دارو را به انسولین تغییر دهد زیرا خطر قند خون پایین در نوزاد وجود دارد؛ شیردهی: باید تحت نظر پزشک باشد زیرا مشخص نیست دارو وارد شیر مادر می‌شود یا خیر",  # noqa: E501
            "مصرف همراه غذا": "قبل از صبحانه یا اولین وعده غذایی مصرف شود؛ در صورت مصرف دوبار روزانه، نوبت صبح همراه صبحانه و نوبت عصر همراه غذا مصرف شود",  # noqa: E501
            "شرایط نگهداری": "ارائه نشده",
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "دیابت نوع ۲",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای کنترل قند خون. طبقه‌بندی مارتیندل: سولفونامیدها، مشتقات اوره، داروهای کنترل قند خون",  # noqa: E501
            "تست‌های آزمایشگاهی دوره‌ای": "توصیه به انجام دوره‌ای آزمایش عملکرد کلیه، آزمایش عملکرد کبد، قند خون، هموگلوبین ای وان سی و شمارش کامل سلول‌های خون.",  # noqa: E501
            "فهرست گسترده‌تر نام‌های تجاری در متن مقاله": "علاوه بر دو برند رسمی کلازکس و گلیکوور، در متن مقاله ده نام تجاری دیگر هم ذکر شده شامل گلی کلازاید آریا، گلی کلازید الحاوی، گلیزان، آنی کلاز، دیامیکرون ام آر، گلی زیداس، گلی کلاویت، زیدامد، ویاگلیبت و دیابزید",  # noqa: E501
            "تفاوت دوز ۶۰ و ۸۰ میلی‌گرم": "بخش اختصاصی توضیح می‌دهد که گلی کلازید شصت میلی‌گرم آهسته‌رهش با مصرف یکبار روزانه اثربخشی بیشتری نسبت به گلی کلازید هشتاد میلی‌گرم با مصرف دوبار روزانه در کنترل قند خون دارد",  # noqa: E501
        },
    },
    "ibuprofen": {
        "drug_name": "ibuprofen",
        "source_file": "ibuprofen(1).docx",
        "content": {
            "عنوان": "ایبوپروفن (Ibuprofen)",
            "منبع": "daroobin.com/fa/drug/ibuprofen",
            "نام ژنریک فارسی و انگلیسی": "ایبوپروفن (Ibuprofen)",
            "نام‌های تجاری ایرانی": "ژلوفن، ادویل، بروفن، ایبوپین (شربت)",
            "شکل و قدرت دارویی": "قرص، شربت، شیاف؛ دوزها: ۲۰۰ و ۴۰۰ میلی‌گرم",
            "موارد مصرف": "سردرد، درد دندان، دردهای قاعدگی، دردهای عضلانی/آرتریت، التهاب مفصلی، تب کودکان، دردهای جزئی سرماخوردگی/آنفولانزا، التهاب ناشی از آفتاب‌سوختگی",  # noqa: E501
            "نحوه مصرف": "خوراکی، هر ۴ تا ۸ ساعت (فقط بزرگسالان) با یک لیوان پر آب؛ حداقل ۱۰ دقیقه دراز نکشید؛ در صورت ناراحتی معده با غذا/شیر/آنتی‌اسید مصرف شود؛ حداکثر ۱۶۰۰ میلی‌گرم روزانه برای بزرگسال؛ دوز کودک بر اساس وزن (مثال: ۳ میلی‌لیتر شربت برای کودک ۱۰ کیلویی)؛ توصیه به عدم مصرف هم‌زمان شیاف و شربت",  # noqa: E501
            "فراموشی یک نوبت مصرف": "در صورت فراموشی، به محض یادآوری مصرف شود؛ اگر نزدیک نوبت بعد بود، دوز فراموش‌شده حذف و دوز دو برابر نشود",  # noqa: E501
            "عوارض شایع": "ناراحتی معده، تهوع، استفراغ، سردرد، اسهال، یبوست، سرگیجه یا خواب‌آلودگی",
            "عوارض جدی و علائم خطر": "کبودی/خونریزی، وزوز گوش، تغییرات ذهنی/خلقی، گرفتگی گردن، علائم مشکلات کلیوی، تغییرات بینایی، علائم نارسایی قلبی، خستگی غیرعادی افزایش وزن ناگهانی؛ علائم آسیب کبدی (ادرار تیره، زردی)؛ علائم آلرژی جدی (تب، تورم غدد لنفاوی، بثورات، تورم صورت/زبان/گلو، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "توصیه به اطلاع دادن تمام داروهای تجویزی، بدون نسخه و گیاهی به پزشک یا دندانپزشک، به ویژه پیش از عمل جراحی؛ تداخل با سایر داروهای ضدالتهاب غیراستروئیدی مشابه (آسپرین، سلکوکسیب، کتورولاک، ناپروکسن)؛ مصرف روزانه ایبوپروفن ممکن است اثر محافظت قلبی آسپرین را کاهش دهد (توصیه به فاصله حداقل ۸ ساعت قبل یا ۲ ساعت بعد از آسپرین)؛ تداخل با داروهای ضدپلاکت مانند کلوپیدوگرل و رقیق کننده های خون مانند دابیگاتران، انوکساپارین و وارفارین؛ تداخل با آلیسکرین، مهارکننده های ACE (کاپتوپریل، لیزینوپریل)، مسدودکننده های گیرنده آنژیوتانسین (لوزارتان، والزارتان)، سیدوفوویر، کورتیکواستروئیدها (پردنیزون)، لیتیوم و دیورتیک ها (فوروزماید)؛ مصرف الکل و تنباکو خطر خونریزی معده را افزایش می دهد",  # noqa: E501
            "منع مصرف و احتیاطات": "پیش از مصرف باید سابقه پزشکی زیر با پزشک یا داروساز در میان گذاشته شود: آسم با سابقه بدتر شدن تنفس پس از آسپرین یا سایر NSAIDها، اختلالات خونی (کم خونی، خونریزی، مشکلات لخته شدن)، پولیپ بینی، بیماری قلبی (سابقه حمله قلبی)، فشار خون بالا، بیماری کبد، سکته مغزی، مشکلات گلو یا معده یا روده (خونریزی، سوزش سردل، زخم معده)؛ حساسیت به آسپرین یا سایر NSAIDها (ناپروکسن، سلکوکسیب)؛ احتیاط در سالمندان به دلیل خطر بیشتر خونریزی گوارشی، کلیوی و قلبی؛ احتیاط در کم آبی بدن، نارسایی قلبی یا بیماری کلیوی؛ حساسیت بیشتر به آفتاب",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: اکیداً توصیه نمی‌شود؛ به‌ویژه از هفته ۲۰ تا زایمان ممنوع، پس از هفته ۳۰ کاملاً ممنوع؛ شیردهی: به شیر مادر منتقل می‌شود ولی بعید است به نوزاد آسیب برساند؛ مشورت با پزشک توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "در صورت ناراحتی معده، همراه با غذا، شیر یا آنتی‌اسید مصرف شود",
            "شرایط نگهداری": "ارائه نشده",
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "سردرد، درد دندان، دردهای قاعدگی، دردهای عضلانی یا آرتریت، دردهای جزئی ناشی از سرماخوردگی یا آنفولانزا",  # noqa: E501
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای ضدالتهاب و ضددرد. طبقه‌بندی مارتیندل: محصولات ضدالتهابی و ضدروماتیسمی غیراستروئیدی؛ داروهای مسکن_ضدالتهاب و ضدتب‌ها",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "توصیه به انجام دوره‌ای تست‌های آزمایشگاهی مانند فشار خون و عملکرد کلیه برای پایش اثر دارو و عوارض احتمالی",  # noqa: E501
            "تفاوت دوزهای دارویی (۲۰۰ در برابر ۴۰۰ میلی‌گرم)": "از نظر مواد سازنده، عوارض و تداخلات هیچ تفاوتی بین دو دوز وجود ندارد؛ تفاوت فقط در میزان دوز و شدت درد/بیماری قابل‌درمان است",  # noqa: E501
        },
    },
    "insulin_types": {
        "drug_name": "insulin_types",
        "source_file": "insulin_types.docx",
        "content": {
            "عنوان": "مقایسه انواع انسولین (Insulin Types)",
            "منبع": "صفحات مربوط به هر نوع انسولین در daroobin.com/fa/drug/",
            "انسولین دگلودک + لیراگلوتاید (Insulin degludec + liraglutide)": {
                "دسته اثر": "ترکیبی: طولانی‌اثر + شبه‌اینکرتین",
                "بیماری هدف": "دیابت نوع ۲",
                "اشکال دارو": "قلم تزریقی",
                "زمان‌بندی تزریق نسبت به غذا": "یکبار در روز، با یا بدون غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "پیش از باز شدن در یخچال؛ پس از باز شدن تا ۲۱ روز در دمای اتاق یا یخچال",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "هشدار تومور تیروئید در مطالعات حیوانی (کارسینومای مدولاری تیروئید)؛ ممنوع در سندرم نئوپلاسم چندگانه غدد درون‌ریز نوع ۲",  # noqa: E501
            },
            "انسولین لیسپرو(Insulin Lispro)": {
                "دسته اثر": "سریع‌اثر",
                "بیماری هدف": "ذکر نشده به‌طور مستقل",
                "اشکال دارو": "آمپول",
                "زمان‌بندی تزریق نسبت به غذا": "۱۵ دقیقه قبل از غذا یا فورا بعد از غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "مدت دقیق در متن ذکر نشده؛ فقط گفته شده در دمای اتاق قابل نگهداری است",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "فقط با انسولین ان‌پی‌اچ قابل ترکیب؛ نباید همراه با انسولین لیسپرو-آآب‌ث مصرف شود",  # noqa: E501
            },
            "انسولین گلولیزین(Insulin Glulisine)": {
                "دسته اثر": "سریع‌اثر",
                "بیماری هدف": "ذکر نشده به‌طور مستقل",
                "اشکال دارو": "آمپول",
                "زمان‌بندی تزریق نسبت به غذا": "۱۵ دقیقه قبل از غذا یا ۲۰ دقیقه بعد از شروع غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "تا ۲۸ روز در دمای اتاق؛ در پمپ حداکثر ۴۸ ساعت",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "برند مهم: انسولین اپیدرا؛ بررسی سابقه تیروئید، کبد و کلیه پیش از مصرف ضروری است",  # noqa: E501
            },
            "انسولین گلارژین(Insulin Glargine)": {
                "دسته اثر": "طولانی‌اثر",
                "بیماری هدف": "ذکر نشده به‌طور مستقل",
                "اشکال دارو": "قلم یا کارتریج",
                "زمان‌بندی تزریق نسبت به غذا": "یکبار در روز، در هر ساعتی از روز اما هر روز در همان ساعت مشخص",  # noqa: E501
                "نگهداری پس از باز شدن (دمای اتاق)": "بسته‌بندی‌نشده در یخچال دو تا هشت درجه؛ باز شده تا ۲۸ روز در دمای اتاق زیر سی درجه",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "هرگز نباید با انسولین دیگر یا هر محلول دیگری مخلوط شود؛ با پمپ انسولین استفاده نمی‌شود",  # noqa: E501
            },
            "انسولین دگلودک(Insulin Degludec)": {
                "دسته اثر": "طولانی‌اثر",
                "بیماری هدف": "ذکر نشده به‌طور مستقل، فقط دیابت به‌طور کلی",
                "اشکال دارو": "آمپول",
                "زمان‌بندی تزریق نسبت به غذا": "یکبار در روز؛ حداقل هشت ساعت فاصله بین هر دو نوبت",
                "نگهداری پس از باز شدن (دمای اتاق)": "تا پنجاه و شش روز در دمای اتاق، طولانی‌ترین مدت نگهداری در میان انسولین‌های بررسی‌شده",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "نباید با هیچ نوع انسولین دیگری ترکیب شود؛ با پمپ انسولین استفاده نمی‌شود",  # noqa: E501
            },
            "انسولین آسپارت + انسولین دگلودک(Insulin aspart + insulin degludec)": {
                "دسته اثر": "ترکیبی: سریع‌اثر + طولانی‌اثر تا بیست و چهار ساعت",
                "بیماری هدف": "ذکر نشده به‌طور مستقل",
                "اشکال دارو": "آمپول یا قلم",
                "زمان‌بندی تزریق نسبت به غذا": "معمولا همراه با وعده اصلی غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "قلم بسته تا تاریخ انقضا در یخچال یا حداکثر بیست و هشت روز در دمای اتاق؛ قلم در حال استفاده تا بیست و هشت روز فقط در دمای اتاق",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "تنها با نسخه پزشک قابل استفاده؛ ایمنی برای کودکان زیر یک سال بررسی نشده؛ طولانی‌ترین فهرست تداخلات دارویی در میان انسولین‌های بررسی‌شده، شامل ده‌ها داروی فلوروکینولونی و بتابلاکر؛ هشدار آنافیلاکسی",  # noqa: E501
            },
            "انسولین آسپارت(Insulin Aspart)": {
                "دسته اثر": "سریع‌اثر (شروع اثر دو و نیم تا بیست دقیقه، پیک اثر یک تا سه ساعت، ماندگاری حدود پنج ساعت)",  # noqa: E501
                "بیماری هدف": "دیابت",
                "اشکال دارو": "آمپول",
                "زمان‌بندی تزریق نسبت به غذا": "قبل از غذا یا بیست دقیقه بعد از غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "کارتریج خارج از بسته فقط در دمای اتاق؛ همه انسولین‌ها بیست و هشت روز پس از شروع مصرف دور ریخته شوند؛ در پمپ حداکثر شش روز",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "معمولا همراه با یک انسولین متوسط یا طولانی‌اثر مانند ترکیب آسپارت و دگلودک استفاده می‌شود",  # noqa: E501
            },
            "انسولین رگولار(Insulin Regular)": {
                "دسته اثر": "کوتاه‌اثر (شروع اثر حدود سی دقیقه، پیک اثر دو تا سه ساعت، ماندگاری سه تا شش ساعت)",  # noqa: E501
                "بیماری هدف": "دیابت",
                "اشکال دارو": "آمپول",
                "زمان‌بندی تزریق نسبت به غذا": "سی دقیقه قبل از غذا",
                "نگهداری پس از باز شدن (دمای اتاق)": "شرایط نگهداری بسته به برند متفاوت است و باید طبق بروشور یا نظر داروساز عمل شود",  # noqa: E501
                "نکته ویژه یا تفاوت کلیدی": "نام‌های دیگر: انسولین کوتاه‌اثر، انسولین خنثی، انسولین محلول؛ فقط با انسولین ان‌پی‌اچ قابل ترکیب",  # noqa: E501
            },
        },
    },
    "isotretinoin": {
        "drug_name": "isotretinoin",
        "source_file": "isotretinoin.docx",
        "content": {
            "عنوان": "ایزوترتینوئین (Isotretinoin)",
            "منبع": "daroobin.com/fa/drug/isotretinoin",
            "نام ژنریک فارسی و انگلیسی": "ایزوترتینوئین (Isotretinoin)",
            "نام‌های تجاری ایرانی": "راکوتان (یکی از معروف‌ترین برندهای ایزوترتینوئین)",
            "شکل و قدرت دارویی": "قرص ۱۰ و ۲۰ میلی‌گرم (دوز ۲۰ میلی‌گرمی شایع‌تر است)؛ همچنین به شکل ژل و کرم برای برخی مشکلات پوستی موجود است",  # noqa: E501
            "موارد مصرف": "آکنه سیستیک نودولار؛ بثورات جلدی توده‌ای (جوش‌های کنار هم)؛ جوش‌های سطحی؛ جوش مقاوم به آنتی‌بیوتیک؛ جوش ناشی از مشکلات هورمونی یا عصبی",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولاً دو بار در روز به مدت ۱۵ تا ۲۰ هفته (حداقل ۱۶ هفته) یا طبق دستور پزشک؛ کپسول باید کامل بلعیده شود، بدون خرد کردن یا جویدن؛ توصیه به مصرف همراه غذا برای افزایش جذب؛ مصرف با یک لیوان پر آب و عدم دراز کشیدن حداقل ۱۰ دقیقه پس از مصرف؛ دوز بر اساس وضعیت پزشکی، وزن و پاسخ به درمان تنظیم می‌شود؛ ممکن است آکنه در روزهای اول تشدید شود و ۱ تا ۲ ماه طول بکشد تا اثر کامل ظاهر شود؛ در صورت بازگشت آکنه شدید، دوره دوم درمان پس از ۲ ماه قطع مصرف آغاز می‌شود؛ مصرف طولانی‌مدت توصیه نمی‌شود و باید فقط توسط متخصص پوست یا زیر نظر او تجویز شود؛ معمولاً تکرار دوره لازم نیست",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "خشکی لب و دهان، تورم جزئی پلک‌ها یا لب‌ها، پوسته‌پوسته شدن پوست، خونریزی بینی، ناراحتی معده، نازک شدن مو",  # noqa: E501
            "عوارض جدی و علائم خطر": "تغییرات روانی یا خلقی (افسردگی، اضطراب، افکار خودکشی)؛ سوزن‌سوزن شدن پوست؛ درد کمر؛ درد مفاصل یا عضله؛ علائم عفونت (گلودرد مکرر، تب، بلع دردناک)؛ پوسته شدن پوست کف دست‌ها یا پاها؛ پانکراتیت (درد شدید معده، تهوع یا استفراغ شدید یا مکرر - نیازمند قطع فوری دارو)؛ عوارض بسیار جدی: سردرد شدید، تغییرات بینایی، صدای زنگ در گوش، کاهش شنوایی، درد قفسه سینه، زردی چشم یا پوست، ادرار تیره، اسهال شدید، خونریزی رکتوم؛ واکنش آلرژیک جدی (بثورات، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "تداخلات رده X (پرهیزی): آمینولوولینیک اسید سیستمیک، مولتی‌ویتامین‌های حاوی ویتامین A، تتراسایکلین‌ها (مینوسیکلین، تتراسایکلین)، ویتامین A، سایر داروهای نوع ویتامین A (آسیترتین، بکساروتن)، داروهایی که باعث تحلیل استخوان می‌شوند (ضدتشنج‌هایی مانند فنی‌توئین، کورتیکواستروئیدهایی مانند پردنیزون)؛ کاهش اثر داروهای ضدبارداری پروژستین‌تنها (مینی‌قرص) در مصرف هم‌زمان؛ افزایش اثر با آمینولوولینیک اسید موضعی، میپومرسن، پورفیمر، ورتپورفین؛ تشدید اثرات دارو با الکل، مولتی‌ویتامین‌های حاوی ویتامین A و تتراسایکلین",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به ایزوترتینوئین یا سایر رتینوئیدها (مانند ترتینوئین)؛ احتمال حساسیت به مواد غیرفعال مانند سویا یا پارابن (افراد حساس به بادام‌زمینی ممکن است به سویا نیز حساس باشند)؛ سابقه دیابت، چربی خون بالا (تری‌گلیسیرید)، مشکلات روحی/خلقی (افسردگی)، بیماری کبد، چاقی، اختلالات خوردن (بی‌اشتهایی عصبی)، سوءمصرف الکل، پانکراتیت، شرایط کاهش تراکم استخوان (پوکی استخوان)؛ عدم اهدای خون در طول مصرف و حداقل یک ماه پس از قطع دارو؛ افزایش حساسیت به نور خورشید (محدود کردن آفتاب، کرم ضدآفتاب، لباس محافظ)؛ تأثیر بر دید در شب - پرهیز از رانندگی و کار با ماشین‌آلات پس از تاریکی هوا؛ افت احتمالی عملکرد لنزهای تماسی؛ پرهیز از اقدامات زیبایی پوستی مانند اپیلاسیون، لیزر و درم‌ابریژن در طول درمان و تا ۶ ماه پس از آن (خطر ایجاد اسکار)؛ پرهیز از الکل (افزایش خطر پانکراتیت)؛ خطر کاهش تراکم استخوان با ورزش‌های تماسی یا ضربه‌ای مکرر؛ احتمال توقف رشد طبیعی در برخی کودکان (بستن زودرس صفحه اپی‌فیزیال)؛ احتیاط بیشتر در سالمندان (اثر بر استخوان) و کودکان (درد کمر، مفاصل و عضله)",  # noqa: E501
            "مصرف در بارداری و شیردهی": "ممنوعیت کامل مصرف در بارداری و در زنان با قصد بارداری، به دلیل خطر نقایص مادرزادی جدی، سقط جنین و زایمان زودرس؛ ممنوعیت تنفس گرد و غبار کپسول‌ها برای زنان باردار یا قصد بارداری؛ الزام به استفاده از دو روش مؤثر پیشگیری از بارداری (یا پرهیز کامل از آمیزش) از یک ماه قبل تا یک ماه پس از مصرف، همراه با مشاوره ماهانه با پزشک؛ ممنوعیت استفاده از مینی‌قرص (بدون استروژن)؛ الزام به دو تست بارداری منفی پیش از شروع درمان و آزمایش بارداری ماهانه در طول درمان، با قطع فوری دارو در صورت مثبت شدن تست؛ شیردهی: مشخص نیست دارو در شیر مادر ترشح می‌شود یا خیر، اما به دلیل ترشح داروهای مشابه، شیردهی در حین مصرف توصیه نمی‌شود",  # noqa: E501
            "مصرف همراه غذا": "توصیه به مصرف همراه غذا، زیرا غذا به افزایش جذب دارو در جریان خون کمک می‌کند؛ فراهمی زیستی دارو در صورت مصرف همراه غذا یا شیر افزایش می‌یابد",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در توالت یا فاضلاب مگر دستور خاص؛ مشورت با داروساز یا شرکت دفع زباله محلی برای دفع صحیح دارو",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "آکنه و جوش صورت و بدن (سیستیک نودولار، بثورات توده‌ای، جوش‌های سطحی و مقاوم به آنتی‌بیوتیک)",  # noqa: E501
            "طبقه‌بندی دارویی": "ضد آکنه و جوش؛ طبقه‌بندی مارتیندل: درماتولوژیک؛ از دسته داروهای رتینوئید",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور مشخص و مفصل ذکر شده: تست بارداری ماهانه، سطح کلسترول و تری‌گلیسیرید خون، عملکرد کبد، شمارش گلبول‌های سفید خون، معاینه چشم",  # noqa: E501
            "تفاوت دوزهای دارویی (۱۰ در برابر ۲۰ میلی‌گرم)": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "استفراغ، درد معده، برافروختگی صورت، سردرد، از دست دادن تعادل؛ در صورت غش یا مشکل تنفسی فوراً با اورژانس ۱۱۵ تماس بگیرید، در غیر این صورت فرد را فوراً به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "levothyroxine": {
        "drug_name": "levothyroxine",
        "source_file": "levothyroxine.docx",
        "content": {
            "عنوان": "لووتیروکسین (Levothyroxine)",
            "منبع": "daroobin.com/fa/drug/levothyroxine",
            "نام ژنریک فارسی و انگلیسی": "لووتیروکسین (Levothyroxine)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص در دوزهای ۲۵، ۵۰، ۷۵ و ۱۰۰ میکروگرم",
            "موارد مصرف": "درمان کم‌کاری تیروئید، چه به‌صورت طبیعی و چه ناشی از پرتودرمانی، مواد دارویی یا جراحی برداشت تیروئید؛ درمان سایر بیماری‌های تیروئید مانند سرطان تیروئید؛ نباید برای درمان ناباروری استفاده شود مگر اینکه علت آن کمبود هورمون تیروئید باشد",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولا یکبار در روز با معده خالی، نیم تا یک ساعت قبل از صبحانه، همراه با یک لیوان پر آب؛ کپسول باید کامل بلعیده شود و نباید جویده، خرد یا نصف شود؛ برای کودکانی که نمی‌توانند قرص را کامل ببلعند، قرص خرد و با یک تا دو قاشق چای‌خوری آب مخلوط و بلافاصله مصرف شود، بدون آماده‌سازی از پیش و بدون مخلوط کردن با شیر خشک سویا؛ درمان معمولا مادام‌العمر است و نباید بدون هماهنگی پزشک قطع شود؛ برند مصرفی نباید بدون هماهنگی پزشک تغییر کند؛ داروهایی که جذب هورمون تیروئید را کاهش می‌دهند باید با فاصله چهار ساعت از این دارو مصرف شوند",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود مگر نزدیک نوبت بعد باشد که در این صورت نادیده گرفته شده و دوز دو برابر نشود مگر پزشک توصیه کرده باشد؛ در صورت فراموشی دو نوبت یا بیشتر پشت سر هم باید به پزشک اطلاع داده شود؛ در اوردوز همراه با بی‌هوشی یا تنگی نفس با اورژانس تماس گرفته شود؛ سایر علائم اوردوز شامل ضربان قلب نامنظم یا تند، کاهش هوشیاری، گیجی و تشنج است",  # noqa: E501
            "عوارض شایع": "ریزش مو در چند هفته اول مصرف که معمولا موقتی است و با سازگاری بدن برطرف می‌شود",  # noqa: E501
            "عوارض جدی و علائم خطر": "افزایش تعریق، حساسیت به گرما، تغییرات خلقی مانند عصبانیت یا نوسانات خلقی، خستگی، اسهال، لرزش، سردرد، تنگی نفس، درد استخوان و شکنندگی استخوان؛ عوارض بسیار حاد شامل درد سینه، ضربان قلب نامنظم یا شدید یا تند، تورم دست، پا یا قوزک پا و تشنج؛ علائم آلرژی جدی شامل راش پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و تنگی نفس",  # noqa: E501
            "تداخلات دارویی": "رقیق‌کننده‌های خون مانند وارفارین، دیگوکسین و سوکروفریک اکسی‌هیدروکساید؛ کاهش اثر سدیم یدید I صد و سی و یک و مشتقات تئوفیلین توسط لووتیروکسین؛ فهرست بلندی از داروها که اثر لووتیروکسین را کاهش می‌دهند شامل آلومینیوم هیدروکسید، آمیودارون، آپالوتامید، کمپلکس‌سازهای اسید صفراوی، نمک‌های کلسیم، کاربامازپین، سیپروفلوکساسین سیستمیک، مشتقات استروژن، فوس‌فنی‌توئین، فرآورده‌های آهن، لانتانیم، نمک‌های منیزیم، مولتی‌ویتامین‌ها، اورلیستات، پاتیرومر، فنی‌توئین، رالوکسیفن، ریفامپین، ریتوناویر، مهارکننده‌های انتخابی بازجذب سروتونین، سولامر و سوکرالفیت؛ افزایش اثر آمزینیوم، ضدافسردگی‌های سه‌حلقه‌ای و آنتاگونیست‌های ویتامین کا توسط لووتیروکسین؛ افزایش اثر لووتیروکسین توسط پیراستام و سماگلوتید؛ تداخل پرهیزی کامل با سدیم یدید صد و سی و یک",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به دارو؛ سابقه افزایش هورمون تیروئید، کاهش عملکرد غده آدرنال، بیماری قلبی مانند بیماری عروق کرونر یا ضربان قلب نامنظم، فشار خون بالا و دیابت؛ در دیابت ممکن است قند خون تحت تاثیر قرار گیرد و نیاز به پایش منظم دارد؛ اطلاع پیش از عمل جراحی یا اقدامات دندانپزشکی؛ احتیاط بیشتر در کودکان به دلیل خطر سردرد، تغییرات بینایی، درد پا و کاهش رشد استخوان در دوزهای بالا؛ احتیاط بیشتر در سالمندان به دلیل خطر ضربان قلب نامنظم، شدید یا تند؛ هشدار صریح که این دارو نباید برای کاهش وزن استفاده شود زیرا دوزهای معمول باعث کاهش وزن نمی‌شوند و دوزهای بالا می‌توانند عوارض حاد و گاهی کشنده ایجاد کنند، به‌ویژه همراه با قرص‌های لاغری",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: مصرف این دارو در بارداری اشکالی ندارد اما ممکن است نیاز به تنظیم دوز باشد؛ شیردهی: دارو وارد شیر مادر می‌شود اما به احتمال زیاد برای نوزاد ضرری ندارد، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با معده خالی، نیم تا یک ساعت پیش از صبحانه مصرف شود؛ ممکن است با آرد سویا و شیر خشک سویا، سویا، آب گریپ‌فروت، قهوه اسپرسو، تخم کتان، گردو، کلسیم، آهن و فیبر رژیمی تداخل داشته باشد",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ در حمام نگهداری نشود؛ دور از دسترس کودکان و حیوانات خانگی؛ داروها در فاضلاب تخلیه نشوند مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "تیروئید",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای درمان تیروئید. طبقه‌بندی مارتیندل: داروهای ضد تیروئید",  # noqa: E501
            "نکته کیفیت داده درباره طبقه‌بندی مارتیندل": "طبقه‌بندی مارتیندل ذکرشده یعنی داروهای ضد تیروئید با ماهیت دارویی لووتیروکسین همخوانی ندارد، زیرا این دارو جایگزین هورمون تیروئید است نه یک داروی ضد تیروئید؛ این می‌تواند نشانه یک خطای دیگر در طبقه‌بندی سایت باشد و نیاز به راستی‌آزمایی دارد",  # noqa: E501
            "هشدار عدم استفاده برای لاغری": "هشدار صریح که این دارو نباید برای کاهش وزن استفاده شود؛ دوزهای معمول باعث کاهش وزن نمی‌شوند و دوزهای بالا می‌توانند عوارض حاد و گاهی کشنده ایجاد کنند، به‌ویژه در مصرف همزمان با قرص‌های لاغری",  # noqa: E501
        },
    },
    "losartan": {
        "drug_name": "losartan",
        "source_file": "losartan.docx",
        "content": {
            "عنوان": "لوزارتان پتاسیم (Losartan Potassium)",
            "منبع": "daroobin.com/fa/drug/losartan-potassium",
            "نام ژنریک فارسی و انگلیسی": "لوزارتان پتاسیم (Losartan Potassium)",
            "نام‌های تجاری ایرانی": "فهرست بسیار طولانی شامل لوزکسیر، لوزارتان زاگروس دارو، لوزارتان ماد، لوزاژین، لوزامد، لوزاتن، پوزارکس، لوستار، لوزارتان تدافارم، لوزارتان فارماشیمی، لوتنسین، لوزن، لوزار، لوزارتان لقمان، لوزارتان کیمیدارو، لوزارتان حکیم، لوزارتان اکسیر، لوزارتان مفید، لوزارتان اس تی پی، کاردیوسان، لوزاور، لوزارتان درسا، لوزارتان امین و لوزارتان ثنامد؛ این طولانی‌ترین فهرست نام تجاری در میان تمام داروهای بررسی‌شده است، هرچند بدون بخش رسمی مجزای برند تولیدکننده ارائه شده است",  # noqa: E501
            "شکل و قدرت دارویی": "قرص در دوزهای ۲۵، ۵۰ و ۱۰۰ میلی‌گرم؛ کپسول ۲۰ میلی‌گرم",
            "موارد مصرف": "کنترل و درمان فشار خون بالا؛ محافظت از کلیه‌ها در برابر آسیب ناشی از دیابت؛ کاهش خطر سکته مغزی در بیماران مبتلا به فشار خون بالا همراه با بزرگ شدن قلب",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولا یکبار در روز با یا بدون غذا؛ برای فرم مایع، پیش از هر نوبت بطری تکان داده شود و دوز با وسیله مخصوص اندازه‌گیری شود نه با قاشق خانگی؛ مصرف منظم در ساعت مشخص توصیه می‌شود؛ حتی با احساس بهبودی باید مصرف ادامه یابد و دارو بدون نظر پزشک قطع نشود زیرا قطع ناگهانی می‌تواند اثرات سوء داشته باشد",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود مگر نزدیک نوبت بعد باشد که در این صورت نادیده گرفته شده و دوز دو برابر نشود؛ در اوردوز همراه با غش یا مشکل تنفسی با اورژانس تماس گرفته شود؛ سایر علائم اوردوز شامل سرگیجه شدید و غش است",  # noqa: E501
            "عوارض شایع": "سرگیجه یا سبکی سر، معمولا حین تطبیق بدن با دارو",
            "عوارض جدی و علائم خطر": "غش؛ علائم سطح بالای پتاسیم خون مانند ضعف عضلانی یا ضربان قلب آهسته و نامنظم؛ به‌ندرت مشکلات جدی کلیوی یا بدتر شدن مشکلات کلیوی موجود، همراه با نیاز به پایش دوره‌ای عملکرد کلیه؛ علائم مشکلات کلیوی مانند تغییر در مقدار ادرار؛ علائم آلرژی جدی شامل بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و مشکلات تنفسی",  # noqa: E501
            "تداخلات دارویی": "آلیسکیرن و لیتیوم؛ داروهایی که سطح پتاسیم خون را افزایش می‌دهند مانند مهارکننده‌های ACE از جمله بنازپریل و لیزینوپریل، و قرص‌های ضدبارداری حاوی دروسپیرنون؛ ممنوعیت مصرف نمک‌های حاوی پتاسیم در طول درمان؛ احتیاط با محصولات ضدسرفه و سرماخوردگی و داروهای ضدالتهاب غیراستروئیدی مانند ایبوپروفن یا ناپروکسن که ممکن است فشار خون را افزایش دهند یا نارسایی قلبی را بدتر کنند؛ آسپرین و ایبوپروفن ممکن است احتمال بروز عوارض لوزارتان را افزایش دهند",  # noqa: E501
            "منع مصرف و احتیاطات": "منع مصرف مطلق در بارداری، حساسیت شناخته‌شده به دارو، و مصرف همزمان با آلیسکیرن در بیماران مبتلا به دیابت نوع دو؛ سابقه بیماری کبد، کم‌آبی بدن، سطح بالای پتاسیم خون، مشکلات کلیوی به‌ویژه انسداد عروق کلیوی، مشکلات دریچه‌های قلب یا عضلات قلبی و مشکلات کبدی؛ احتمال سرگیجه که با الکل یا ماری‌جوانا تشدید می‌شود و پرهیز از رانندگی توصیه می‌شود؛ محدود کردن مصرف الکل؛ مشورت با پزشک پیش از مصرف مکمل پتاسیم یا جایگزین‌های نمک حاوی پتاسیم؛ اطلاع پیش از عمل جراحی",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: ممنوعیت کامل به دلیل خطر آسیب به جنین؛ نباید حین مصرف این دارو باردار شد و در صورت بارداری باید فورا با پزشک مشورت کرد؛ شیردهی: مشخص نیست دارو وارد شیر مادر می‌شود یا نه، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با غذا تداخل خاصی ندارد و قابل مصرف قبل یا بعد از غذا است",
            "شرایط نگهداری": "قرص در دمای اتاق و دور از نور و رطوبت نگهداری شود و در دستشویی نگهداری نشود؛ دور از دسترس کودکان و حیوانات خانگی؛ سوسپانسیون در یخچال بین دو تا هشت درجه سانتی‌گراد نگهداری و از یخ‌زدگی محافظت شود و پس از چهار هفته دور ریخته شود؛ داروها در توالت یا فاضلاب تخلیه نشوند مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "عدم وجود بیماری هدف در جعبه اطلاعات": "مشابه آسپیرین و آزیترومایسین، در جعبه اطلاعات اولیه این دارو ردیف بیماری هدف وجود ندارد؛ فقط طبقه‌بندی، دزها و اشکال دارو ذکر شده است",  # noqa: E501
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: آنتاگونیست‌های آنژیوتانسین. طبقه‌بندی مارتیندل: داروهای قلبی عروقی",  # noqa: E501
            "تفاوت لوزارتان پتاسیم و لوزارتان اچ": "توضیح داده شده که تفاوت این دو تنها در ترکیبات آن‌هاست؛ لوزارتان پتاسیم فقط حاوی ماده موثره لوزارتان است، در حالی که لوزارتان اچ ترکیبی از لوزارتان و هیدروکلروتیازید است که یک داروی ادرارآور محسوب می‌شود",  # noqa: E501
            "نام تجاری بین‌المللی Cozaar": "اشاره شده که داروی Cozaar از ترکیب لوزارتان و هیدروکلروتیازید تولید شده و برای کاهش فشار خون تجویز می‌شود",  # noqa: E501
            "توضیح مکانیسم دارویی": "توضیح نسبتا تخصصی ارائه شده مبنی بر اینکه آنژیوتانسین دو با تنگ کردن عروق خونی و تاثیر بر هورمون آلدوسترون فشار خون را افزایش می‌دهد و لوزارتان با مهار این مسیر عمل می‌کند",  # noqa: E501
        },
    },
    "metformin": {
        "drug_name": "metformin",
        "source_file": "metformin.docx",
        "content": {
            "عنوان": "متفورمین (Metformin)",
            "منبع": "daroobin.com/fa/drug/metformin",
            "نام ژنریک فارسی و انگلیسی": "متفورمین (Metformin)",
            "نام‌های تجاری ایرانی": "قرص متگلیندین (ایران هورمون)",
            "شکل و قدرت دارویی": "قرص در دوزهای ۲۵۰، ۵۰۰ و ۱۰۰۰ میلی‌گرم",
            "موارد مصرف": "کنترل قند خون بالا در دیابت نوع ۲ همراه با رژیم غذایی و ورزش؛ برخی پزشکان آن را برای کیست تخمدان در زنان هم تجویز می‌کنند؛ به صراحت ذکر شده که برای لاغری تایید نشده و توصیه نمی‌شود",  # noqa: E501
            "نحوه مصرف": "یک تا سه بار در روز همراه با غذا؛ نوشیدن مایعات زیاد در طول مصرف؛ دوز بر اساس شرایط بیمار و پاسخ درمانی تنظیم می‌شود و معمولا با دوز کم شروع و به تدریج افزایش می‌یابد؛ مصرف منظم در ساعت مشخص توصیه می‌شود؛ قرص‌های آهسته‌رهش با علامت SR روی بسته نباید خرد یا تقسیم شوند؛ پیش از جراحی، عکسبرداری با ماده حاجب یددار (دو روز قبل) یا اقدامات دندانپزشکی ممکن است نیاز به قطع موقت دارو باشد",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری همراه غذا مصرف شود؛ اگر نزدیک نوبت بعد بود، نادیده گرفته شده و دوز دو برابر نشود؛ در اوردوز همراه با غش یا مشکل تنفسی با اورژانس تماس گرفته شود؛ سایر علائم اوردوز شامل خواب‌آلودگی شدید، تهوع، استفراغ، اسهال شدید، تنفس سریع و ضربان قلب آهسته یا نامنظم است که می‌تواند نشانه اسیدوز لاکتیک باشد",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، ناراحتی معده، اسهال، ضعف، طعم فلزی در دهان",
            "عوارض جدی و علائم خطر": "هشدار ویژه درباره اسیدوز لاکتیک، یک عارضه نادر اما گاهی کشنده، با علائمی مانند خستگی غیرمعمول، سرگیجه، خواب‌آلودگی شدید، لرز، پوست آبی یا سرد، درد عضلانی، تنفس سریع یا مشکل و ضربان قلب آهسته و نامنظم؛ علائم قند خون پایین شامل تعریق ناگهانی، لرزش، ضربان قلب سریع، گرسنگی، تاری دید، سرگیجه و گزگز دست یا پا؛ علائم قند خون بالا شامل تشنگی، افزایش ادرار، گیجی، خواب‌آلودگی و بوی میوه در تنفس؛ علائم آلرژی جدی شامل بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و مشکلات تنفسی؛ احتمال کاهش جذب ویتامین ب دوازده که نیاز به تزریق دوره‌ای هر سه ماه دارد",  # noqa: E501
            "تداخلات دارویی": "داروهای مسدودکننده بتا مانند متوپرولول، پروپرانولول و قطره‌های چشمی گلوکوم مانند تیمولول ممکن است علائم قند خون پایین را پنهان کنند؛ الکل به عنوان تداخل پرهیزی کامل ذکر شده و خطر اسیدوز لاکتیک و قند خون پایین را افزایش می‌دهد؛ توصیه کلی به اطلاع‌رسانی تمام داروهای مصرفی به پزشک زیرا بسیاری از داروها بر قند خون اثر می‌گذارند",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت قبلی به متفورمین یا سایر داروها؛ سابقه مشکلات تنفسی شدید مانند بیماری انسدادی ریه یا آسم شدید، مشکلات خونی مانند کم‌خونی یا کمبود ویتامین ب دوازده، بیماری کلیوی و بیماری کبدی؛ نیاز به قطع موقت پیش از جراحی، عکسبرداری با ماده حاجب یا اقدامات دندانپزشکی؛ احتیاط در قند خون بسیار پایین یا بالا به دلیل تاری دید، سرگیجه یا خواب‌آلودگی و پرهیز از رانندگی؛ ممنوعیت کامل مصرف الکل؛ احتیاط در تب بالا، مصرف داروهای ادرارآور، تعریق زیاد، اسهال یا استفراغ به دلیل خطر کم‌آبی بدن و اسیدوز لاکتیک؛ احتیاط در شرایط استرس بدنی مانند تب، عفونت، جراحت یا جراحی؛ احتیاط بیشتر در سالمندان؛ احتمال افزایش باروری و نیاز به روش پیشگیری بارداری مطمئن؛ گروه‌های پرخطر اسیدوز لاکتیک شامل سالمندان، بیماران کلیوی یا کبدی، افراد دچار کم‌آبی بدن، نارسایی قلبی، مصرف شدید الکل، جراحی و استفاده از ماده حاجب یددار",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود و پزشک ممکن است انسولین را جایگزین کند؛ شیردهی: مقدار کمی از دارو وارد شیر مادر می‌شود، مشورت با پزشک پیش از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "همراه با غذا مصرف شود تا ناراحتی معده کاهش یابد",
            "شرایط نگهداری": "در دمای اتاق و دور از نور و رطوبت نگهداری شود؛ در حمام نگهداری نشود؛ دور از دسترس کودکان و حیوانات خانگی؛ داروها در توالت تخلیه نشوند مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "دیابت نوع ۲",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: داروهای کنترل قند خون. طبقه‌بندی مارتیندل: بیگوانیدها، داروهای کنترل قند خون",  # noqa: E501
            "رد شایعه لاغری": "بحث مفصل درباره اینکه متفورمین قرص لاغری نیست، هرچند ممکن است کاهش وزن به عنوان عارضه جانبی ناشی از کاهش اشتها رخ دهد؛ توصیه صریح داروبین به عدم استفاده از این دارو برای لاغری",  # noqa: E501
            "داروهای ترکیبی حاوی متفورمین": "فهرست دقیق ترکیبات دارویی همراه با متفورمین به همراه توضیح مکانیسم هر جزء، شامل ترکیب متفورمین با امپاگلیفلوزین، با سیتاگلیپتین، با گلی‌بنکلامید و ترکیب سه‌گانه امپاگلیفلوزین، لیناگلیپتین و متفورمین",  # noqa: E501
            "تعامل واقعی پرسش و پاسخ کاربر": "برخلاف داروهای قبلی که بخش نظرات خالی بود، این صفحه دارای یک پرسش واقعی از یک کاربر درباره تداخل احتمالی با گریپ‌فروت است که کارشناس داروبین صراحتا پاسخ داده اطلاعاتی در این مورد در دست ندارد و توصیه به مشورت با متخصص کرده است",  # noqa: E501
            "نکته کیفیت داده درباره نام برند": "برند ذکرشده در صفحه، قرص متگلیندین از تولیدکننده ایران هورمون، از نظر نام‌گذاری با متفورمین همخوانی ندارد و بیشتر شبیه نام یک داروی دیگر از خانواده گلینیدها است؛ این می‌تواند نشانه یک خطای دیگر در بخش برندهای سایت باشد و نیاز به راستی‌آزمایی دارد",  # noqa: E501
        },
    },
    "methotrexate": {
        "drug_name": "methotrexate",
        "source_file": "methotrexate.docx",
        "content": {
            "عنوان": "متوترکسات (Methotrexate)",
            "منبع": "daroobin.com/fa/drug/methotrexate",
            "نام ژنریک فارسی و انگلیسی": "متوترکسات (Methotrexate)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص متوترکسات ۲.۵ و ۵ میلی‌گرم؛ آمپول متوترکسات ۲.۵، ۵، ۱۰، ۲۵، ۵۰ و ۱۰۰ میلی‌گرم؛ ویال متوترکسات ۵ و ۵۰۰ میلی‌گرم و ویال ۱ گرمی",  # noqa: E501
            "موارد مصرف": "پسوریازیس؛ آرتریت روماتوئید شدید؛ آرتریت روماتوئید جوانان؛ برخی از سرطان‌ها",  # noqa: E501
            "نحوه مصرف": "تزریقی، بر اساس تجویز پزشک؛ برای آرتریت روماتوئید یا پسوریازیس معمولاً یک‌بار در هفته؛ دوز، روش تزریق و تعداد دفعات مصرف بر اساس وضعیت بیماری و پاسخ به درمان تعیین می‌شود؛ در صورت تزریق خانگی، لازم است تمام دستورالعمل‌های آماده‌سازی و مصرف از متخصص آموخته شود؛ پیش از مصرف باید از نظر وجود ذرات اضافی یا تغییر رنگ بررسی شود و در صورت مشاهده، مصرف نشود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "این دارو باید سر وقت مصرف شود؛ در صورت فراموشی، فوراً درباره برنامه زمان‌بندی جدید با پزشک مشورت شود؛ دوز مصرفی دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "تهوع، استفراغ، درد معده، خواب‌آلودگی، سرگیجه؛ ریزش مو (موقتی و برگشت‌پذیر پس از پایان مصرف)",  # noqa: E501
            "عوارض جدی و علائم خطر": "زخم دهانی، اسهال، علائم کم‌خونی (خستگی غیرعادی، رنگ‌پریدگی)، علائم مشکلات کبدی (تهوع/استفراغ ممتد، ادرار تیره، درد شکم، زردی چشم یا پوست)، افزایش خونریزی یا کبودی، مدفوع سیاه، بزرگ شدن غدد لنفاوی، درد استخوان، درد و تغییر رنگ غیرعادی پوست، علائم مشکلات کلیوی، سرفه خشک، ضعف عضلات؛ عوارض بسیار حاد: ضربان قلب نامنظم، تغییر بینایی، ضعف در یک طرف بدن، سردرد شدید، گرفتگی گردن، تغییرات خلق و روان، تشنج؛ مشکلات ریوی و عفونت ریه (پنوموسیستیس جیرووسی)، واکنش‌های پوستی؛ افزایش خطر عفونت (گلودرد ممتد، تب، لرز، سرفه)؛ بروز توده یا رشد غیرطبیعی جدید (نادر)؛ سندروم تومور لیز؛ واکنش آلرژیک جدی (راش، خارش/تورم صورت-زبان-گلو، سرگیجه شدید، تنگی نفس)",  # noqa: E501
            "تداخلات دارویی": "فهرست بسیار گسترده از جمله: آسیترتین، آسپاراژیناز، کلرامفنیکل، لفلونومید، داروهای مضر کلیوی (سیسپلاتین)، داروهای مضر کبدی (آزاتیوپرین، سولفاسالازین، رتینوئیدها)، پنی‌سیلین‌ها، فنی‌توئین، پروبنسید، تتراسایکلین‌ها؛ مهارکننده‌های پمپ پروتون (امپرازول، اسومپرازول، پنتوپرازول) که سطح خونی متوترکسات را افزایش می‌دهند؛ NSAIDها به‌ویژه همراه با دوزهای بالای متوترکسات که خطر مشکلات جدی خونی، مغز استخوان و گوارشی دارند؛ آسپیرین با دوز پایین (۸۱ تا ۳۲۵ میلی‌گرم روزانه) برای پیشگیری قلبی با تجویز پزشک قابل ادامه است؛ فهرست‌های جداگانه از داروهایی که کارایی متوترکسات را کاهش یا افزایش می‌دهند، داروهایی که اثرشان توسط متوترکسات افزایش می‌یابد و داروهایی که کارایی متوترکسات را افزایش می‌دهند (شامل ده‌ها دارو)؛ تداخلات پرهیزی: آسیترتین، BCG داخل مثانه‌ای، کلادریبین، دی‌کلروفنامید، دیپیرون، فوسکارنت، لاسمیدیتان، ناتالیزوماب، نیتروس اکسید، پیمکرولیموس، تاکرولیموس موضعی",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به دارو؛ سابقه بیماری کبدی، کلیوی، بیماری ریه (فیبروز دستگاه تنفسی)، مصرف الکل، ضعف سیستم ایمنی، اختلالات گلبول‌های خونی یا مغز استخوان، بیماری‌های گوارشی (زخم گوارشی، کولیت اولسراتیو)، عفونت فعال (مانند آبله مرغان)، کمبود فولیک اسید؛ افزایش خطر عفونت - پرهیز از تماس با افراد مبتلا به بیماری‌های واگیردار و افراد دریافت‌کننده واکسن زنده اخیر؛ اطلاع به پزشک پیش از واکسیناسیون؛ احتیاط با اشیاء تیز و پرهیز از ورزش‌های تماسی؛ احتیاط رانندگی/کار با ماشین تا رفع گیجی یا خواب‌آلودگی؛ پرهیز از الکل و مواد مخدر؛ کودکان حساسیت بیشتری به عوارض به‌ویژه تشنج دارند؛ افزایش حساسیت پوست به نور خورشید (استفاده از عینک، لباس محافظ، کرم ضدآفتاب)؛ تأثیر بر باروری مردان و زنان؛ برخی برندها حاوی مواد نگهدارنده هستند که نباید در دوز بالا یا تزریق داخل نخاعی استفاده شوند؛ احتیاط ویژه در مشکلات کلیوی یا آب‌آوردن (آسیت)؛ توصیه به بیوپسی کبد در مصرف طولانی‌مدت",  # noqa: E501
            "مصرف در بارداری و شیردهی": "ممنوعیت کامل در بارداری به دلیل خطر آسیب به جنین؛ لازم است پیش از مصرف تست بارداری انجام شود؛ زنان در سن باروری باید در طول مصرف و تا ۶ ماه پس از آن، و مردان با همسر در سن باروری تا ۳ ماه پس از مصرف، از روش‌های پیشگیری از بارداری استفاده کنند؛ شیردهی: دارو وارد شیر مادر می‌شود و ممکن است برای نوزاد خطرناک باشد، شیردهی در طول مصرف و حداقل تا ۱ هفته پس از آن توصیه نمی‌شود",  # noqa: E501
            "مصرف همراه غذا": "ارائه نشده",
            "شرایط نگهداری": "بررسی دستورالعمل داخل بسته‌بندی یا مراجعه به داروساز توصیه می‌شود؛ عدم نگهداری در حمام؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب مگر توصیه شده باشد؛ دور ریختن صحیح دارو پس از پایان مصرف یا انقضا",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "پسوریازیس، آرتریت روماتوئید شدید، آرتریت روماتوئید جوانان، برخی از سرطان‌ها",  # noqa: E501
            "طبقه‌بندی دارویی": "ایمونولوژی؛ طبقه‌بندی مارتیندل: ضد متابولیت‌ها؛ از دسته داروهای آنتی‌متابولیت",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور مشخص و مفصل ذکر شده: پیش از مصرف و در طول درمان، انجام تست‌هایی مانند شمارش گلبول‌های خون، بررسی عملکرد کبد و کلیه، تست بارداری و رادیوگرافی قفسه سینه؛ در مصرف طولانی‌مدت، بیوپسی کبد نیز توصیه می‌شود",  # noqa: E501
            "تفاوت دوزهای دارویی": "بین دوز بالا و دوز پایین تمایز کاربردی ذکر شده: مصرف دوزهای بالای متوترکسات همراه با NSAIDها خطر مشکلات جدی خونی، مغز استخوان و گوارشی دارد و باید از آن پرهیز کرد، اما دوزهای پایین متوترکسات را می‌توان با تجویز پزشک همراه با NSAID یا آسپیرین برای بیماری‌هایی مانند آرتریت روماتوئید مصرف کرد؛ تفاوت دقیق کاربردی بین سایر دوزهای عددی (مانند ۲.۵ در برابر ۵۰ یا ۱۰۰ میلی‌گرم) به‌طور جداگانه ذکر نشده است",  # noqa: E501
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "تهوع و استفراغ شدید، مدفوع خونی؛ در صورت بروز بی‌هوشی یا تنگی نفس فوراً با اورژانس ۱۱۵ تماس بگیرید، در غیر این صورت فوراً به پزشک مراجعه کنید",  # noqa: E501
            "سندروم تومور لیز (معیار اضافه)": "در نتیجه تخریب سریع بافت سرطانی توسط دارو ممکن است سندروم تومور لیز بروز کند؛ برای کاهش این خطر توصیه شده مقدار زیادی آب نوشیده شود (مگر توصیه دیگری از پزشک) و ممکن است پزشک داروی دیگری نیز تجویز کند؛ علائم هشداردهنده شامل ضربان قلب نامنظم، درد پایین کمر یا پهلو، علائم مشکلات کلیوی (ادرار خونی یا صورتی، تغییر مقدار ادرار، ادرار دردناک) و اسپاسم یا ضعف عضلات است که در صورت بروز باید فوراً به پزشک اطلاع داده شود",  # noqa: E501
        },
    },
    "metoprolol": {
        "drug_name": "metoprolol",
        "source_file": "metoprolol.docx",
        "content": {
            "عنوان": "متوپرولول (Metoprolol)",
            "نام ژنریک فارسی و انگلیسی": "متوپرولول (Metoprolol)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "آمپول، قرص؛ دوزهای مشخص (میلی‌گرم) در متن ذکر نشده",
            "موارد مصرف": "فشار خون بالا؛ درد قفسه سینه (آنژین صدری)؛ کاهش خطر مرگ یا حمله قلبی مجدد پس از حمله قلبی (فرم تارتارات)؛ پیشگیری از میگرن و درمان برخی آریتمی‌ها (فرم تارتارات)",  # noqa: E501
            "نحوه مصرف": "خوراکی، همراه یا بلافاصله بعد از غذا، طبق تجویز پزشک معمولاً ۱ تا ۳ بار در روز؛ دوز بر اساس وضعیت پزشکی و پاسخ به درمان تعیین می‌شود؛ شروع با دوز کم و افزایش تدریجی توسط پزشک؛ مصرف منظم و در ساعات مشخص؛ عدم قطع ناگهانی بدون مشورت پزشک",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر به نوبت بعد نزدیک بود، نوبت فراموش‌شده حذف و دوز دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "خواب‌آلودگی، سرگیجه، خستگی، اسهال، ضربان قلب آهسته، کاهش توانایی جنسی",
            "عوارض جدی و علائم خطر": "ضربان قلب بسیار آهسته، سرگیجه شدید، غش، آبی شدن انگشتان دست/پا، مشکل تنفسی، علائم جدید یا بدتر شدن نارسایی قلبی (تنگی نفس، تورم مچ پا/پا، خستگی غیرعادی، افزایش وزن ناگهانی)، تغییرات ذهنی/خلقی (گیجی، افسردگی)؛ علائم آلرژی جدی (راش، خارش، تورم صورت/زبان/گلو، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "فینگولیمود؛ لومفانترین، پروپافنون، کینیدین، داروهای ضدافسردگی SSRI (فلوکستین، پاروکستین)، خار مریم (تأثیر بر متابولیسم/عملکرد دارو)؛ تداخلات پرهیزی: برومپریدول، اتوفیلین، فکسینیدازول، فلوکتافنین، ریواستیگمین؛ احتیاط با NSAIDها (ایبوپروفن، ناپروکسن) و داروهای ضدسرفه/سرماخوردگی (افزایش فشار خون)؛ تداخل غذایی: الکل (تشدید سرگیجه و خواب‌آلودگی)",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به متوپرولول یا سایر بتابلوکرها (آتنولول، پروپرانولول)؛ مشکلات ریتم قلب (ضربان آهسته، سندرم sick sinus، بلوک دهلیزی‌بطنی درجه ۲ یا ۳)؛ مشکلات تنفسی (آسم، برونشیت مزمن، آمفیزم)؛ بیماری کبدی؛ نارسایی قلبی؛ واکنش‌های آلرژیک جدی؛ مشکلات گردش خون (رینود، بیماری عروق محیطی)؛ اختلالات خلقی/روانی (افسردگی)؛ میاستنی گراویس؛ احتیاط در دیابت (مخفی‌کردن علائم افت قندخون)؛ احتیاط در کودکان (خطر بیشتر افت قندخون)؛ پرهیز از رانندگی/کارهای نیازمند هوشیاری تا رفع سرگیجه؛ پرهیز از الکل و ماری‌جوانا",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود، ممکن است به جنین آسیب برساند، مشورت با پزشک الزامی؛ شیردهی: وارد شیر مادر می‌شود ولی بعید است به نوزاد آسیب برساند، مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "همراه یا بلافاصله بعد از غذا مصرف شود",
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ عدم نگهداری در حمام؛ دور از دسترس کودکان و حیوانات خانگی؛ دور ریختن صحیح دارو پس از انقضا یا عدم نیاز (نه در توالت/زهکشی مگر با دستور)",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "فشار خون بالا، درد قفسه سینه (آنژین صدری)",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: عوامل مسدود کننده بتا (بتابلاکر)؛ طبقه‌بندی مارتیندل: داروهای مسدودکننده بتا",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "ارائه نشده به‌صورت عمومی؛ فقط توصیه به کنترل منظم قند خون در بیماران دیابتی ذکر شده (نه به‌عنوان نظارت آزمایشگاهی دوره‌ای عمومی برای همه بیماران)",  # noqa: E501
            "تفاوت دوزهای دارویی (تارتارات در برابر سوکسینات)": "متوپرولول تارتارات: رهش فوری، چند بار در روز، برای آنژین/فشار خون/بلافاصله پس از حمله قلبی/پیشگیری میگرن/برخی آریتمی‌ها؛ متوپرولول سوکسینات: رهش طولانی، یک بار در روز، مناسب برای برخی انواع نارسایی قلبی، نباید برای پیشگیری از حمله قلبی استفاده شود؛ هر دو به‌عنوان بتابلوکر «انتخابی» طبقه‌بندی می‌شوند (اثر کمتر بر تنفس و پاسخ انسولین نسبت به بتابلوکرهای غیرانتخابی)",  # noqa: E501
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "ضربان قلب بسیار آهسته، سرگیجه شدید، ضعف شدید، غش، مشکل تنفسی؛ در صورت بروز، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "metronidazole": {
        "drug_name": "metronidazole",
        "source_file": "metronidazole.docx",
        "content": {
            "عنوان": "مترونیدازول (Metronidazole)",
            "منبع": "daroobin.com/fa/drug/metronidazole",
            "نام ژنریک فارسی و انگلیسی": "مترونیدازول (Metronidazole)",
            "نام‌های تجاری ایرانی": "متروماکس (تهران شیمی)",
            "شکل و قدرت دارویی": "ژل واژینال ۷۵ درصد، ژل موضعی ۷۵ درصد، قرص ۲۰۰، ۲۵۰ و ۵۰۰ میلی‌گرم، قرص و شیاف واژینال ۵۰۰ میلی‌گرم، ویال ۵۰۰ میلی‌گرم، سوسپانسیون ۱۲۵ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "عفونت‌های واژینال زنان، عفونت دستگاه گوارش شامل هلیکوباکتر پیلوری و زخم معده، عفونت‌های انگلی مانند ژیاردیازیس، آمیبیازیس و تریکومونیازیس، کاهش بوی بد دهان ناشی از باکتری، عفونت‌های دستگاه تناسلی مردان، پیشگیری از عفونت ناشی از بواسیر و شقاق، پیشگیری از عفونت پس از جراحی به‌ویژه جراحی دهان و دندان، سلولیت، زخم پای دیابتی و زخم بستر، عفونت باکتریایی واژن و لگن",  # noqa: E501
            "نحوه مصرف": "آمپول سه تا چهار بار در روز از طریق انفوزیون طی یک ساعت و فقط در محیط بیمارستانی؛ قرص خوراکی کامل و بدون خرد کردن، قبل یا بعد از غذا مصرف شود؛ قرص واژینال با اپلیکاتور و ترجیحا شب قبل از خواب، با ماندن حداقل نیم ساعت به حالت خوابیده و پرهیز از تماس جنسی یا استفاده از کاندوم؛ سوسپانسیون قبل یا بعد از غذا با تکان دادن قبل از مصرف و استفاده از قاشق مخصوص اندازه‌گیری؛ شیاف واژینال با روش قرارگیری خاص و ماندن چند دقیقه در حالت خوابیده؛ کرم موضعی فقط روی پوست و نه نواحی چشم، دهان، بینی، گوش یا واژن؛ پرهیز از رانندگی تا دو ساعت پس از مصرف قرص، سوسپانسیون یا ویال؛ تکمیل کامل دوره درمان حتی با بهبود زودهنگام؛ دوزبندی اختصاصی بر اساس نوع عفونت متفاوت است",  # noqa: E501
            "فراموشی یک نوبت مصرف": "در صورت فراموشی یک نوبت، باید به پزشک مراجعه شود و برنامه مصرف جدید دریافت شود؛ دوز فراموش‌شده هرگز دو برابر نشود؛ در اوردوز همراه با بی‌هوشی، تنگی نفس، تهوع، استفراغ یا بی‌ثباتی فورا با اورژانس تماس گرفته شود",  # noqa: E501
            "عوارض شایع": "سرگیجه، سردرد، ناراحتی معده، تهوع، استفراغ، از دست دادن اشتها، اسهال، یبوست، طعم فلزی در دهان",  # noqa: E501
            "عوارض جدی و علائم خطر": "علائم عفونت جدید، کبودی یا خونریزی آسان، درد معده یا شکم، ادرار دردناک، بی‌ثباتی، تشنج، تغییرات روحی یا روانی مانند گیجی، مشکل در صحبت کردن، بی‌حسی یا گزگز بازوها یا پاها، درد چشم، تغییرات ناگهانی بینایی، سردرد شدید یا مداوم، سفتی یا درد گردن؛ علائم آلرژی جدی شامل بثورات پوستی، خارش یا تورم صورت، زبان یا گلو، سرگیجه شدید و مشکلات تنفسی؛ تیرگی رنگ ادرار به قرمز یا آجری که معمولا بی‌ضرر است؛ سفیدک دهانی یا عفونت قارچی جدید در مصرف طولانی یا دوز بالا",  # noqa: E501
            "تداخلات دارویی": "محصولات حاوی الکل مانند شربت‌های سرفه و سرماخوردگی و افترشیو؛ محصولات حاوی پروپیلن گلیکول؛ محلول لوپیناویر و ریتوناویر؛ لیتیوم؛ احتمال تداخل با برخی تست‌های آزمایشگاهی؛ افزایش اثر وارفارین در مصرف همزمان؛ مهار متابولیسم فنی‌توئین و افزایش غلظت پلاسمایی آن؛ فنوباربیتال متابولیسم مترونیدازول را افزایش و غلظت پلاسمایی آن را کاهش می‌دهد؛ مهار متابولیسم فلوئورواوراسیل؛ احتمال بروز واکنش شبیه دی‌سولفیرام در مصرف همزمان با الکل؛ کاهش اثر واکسن‌های باکتریایی زنده",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به مترونیدازول یا سایر آنتی‌بیوتیک‌ها مانند تینیدازول؛ سابقه بیماری کبد، بیماری کلیوی، اختلالات خونی مانند کاهش سلول‌های خونی، تورم یا ادم و سندرم کوکین؛ پرهیز کامل از الکل و محصولات حاوی پروپیلن گلیکول در طول درمان و تا سه روز پس از پایان آن به دلیل خطر واکنش دی‌سولفیرام؛ احتمال سرگیجه یا خواب‌آلودگی و پرهیز از رانندگی، الکل و مواد مخدر؛ دارو حاوی سدیم است و در رژیم کم‌نمک یا نارسایی قلبی نیاز به احتیاط دارد؛ پرهیز کامل در صورت مصرف دی‌سولفیرام در دو هفته گذشته؛ کاهش اثر واکسن‌های زنده",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت ضرورت و با تجویز پزشک مجاز است، با توجه ویژه در بارداری زیر دوازده هفته؛ شیردهی: دارو به شیر مادر منتقل می‌شود و مصرف آن در این دوران توصیه نمی‌شود",  # noqa: E501
            "مصرف همراه غذا": "قرص خوراکی و سوسپانسیون را می‌توان قبل یا بعد از غذا مصرف کرد",
            "شرایط نگهداری": "در دمای اتاق و دور از نور، رطوبت و دور از دسترس کودکان و حیوانات خانگی نگهداری شود؛ آمپول نیاز به آموزش خاص برای نگهداری و تزریق در منزل دارد و در صورت کدر شدن یا تغییر رنگ محلول نباید تزریق شود؛ دارو در توالت یا فاضلاب تخلیه نشود مگر توصیه شده باشد",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "عفونت‌های باکتری بی‌هوازی یا پروتوزوآ",
            "طبقه‌بندی دارویی": "طبقه‌بندی عمومی: آنتی‌بایوتیک‌ها. طبقه‌بندی مارتیندل: داروهای موضعی شیمی‌درمانی، ضدعفونت برای استفاده سیستمی",  # noqa: E501
            "رد شایعات رایج": "بخش اختصاصی برای رد دو شایعه رایج درباره این دارو: مترونیدازول هیچ تاثیری بر تاخیر پریود شدن ندارد و باعث لاغری نمی‌شود؛ لاغری احتمالی تنها عارضه جانبی ناخواسته ناشی از اسهال یا کاهش اشتهاست",  # noqa: E501
            "توصیه درمان همزمان شریک جنسی": "در صورت استفاده از قرص یا شیاف واژینال، ممکن است نیاز باشد شریک جنسی هم مترونیدازول مصرف کند تا از رشد مجدد عفونت جلوگیری شود",  # noqa: E501
            "دوزبندی اختصاصی بر اساس نوع عفونت": "جدول دقیق دوز بر اساس نوع عفونت ارائه شده است، مانند سلولیت با ۴۰۰ تا ۵۰۰ میلی‌گرم روزانه به مدت ۷ تا ۱۴ روز، هلیکوباکتر پیلوری با ۴۰۰ میلی‌گرم هر ۱۲ ساعت به مدت ۷ تا ۱۰ روز، آمیبیازیس با ۸۰۰ میلی‌گرم سه بار در روز به مدت ۵ تا ۱۰ روز و ژیاردیازیس با ۲ گرم یکبار در روز به مدت ۳ روز؛ این سطح از دقت دوزبندی اختصاصی بر اساس نوع بیماری در داروهای قبلی دیده نشده بود",  # noqa: E501
        },
    },
    "omeprazole": {
        "drug_name": "omeprazole",
        "source_file": "omeprazole.docx",
        "content": {
            "عنوان": "امپرازول (Omeprazole)",
            "منبع": "daroobin.com/fa/drug/omeprazole",
            "نام ژنریک فارسی و انگلیسی": "امپرازول (Omeprazole)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "کپسول امپرازول ۲۰ و ۴۰ میلی‌گرم",
            "موارد مصرف": "زخم‌های خوش‌خیم معده و دوازدهه؛ زخم‌های وابسته به هلیکوباکتر پیلوری؛ سندروم زولینگر-الیسون؛ کاهش اسید معده در جراحی؛ پیشگیری از بازگشت محتویات معده (رفلاکس)؛ درمان GERD و سوزش سردل؛ پیشگیری از زخم و سرطان مری",  # noqa: E501
            "نحوه مصرف": "خوراکی، یک بار در روز، قبل از غذا؛ دوز و مدت درمان در بزرگسالان بر اساس وضعیت پزشکی/پاسخ به درمان و در کودکان بر اساس وزن تعیین می‌شود؛ از خرد کردن، شکستن یا جویدن قرص خودداری شود؛ قرص‌های حل‌شونده در دهان با دست خشک روی زبان قرار داده شوند؛ در صورت مصرف هم‌زمان با سوکرالفات، حداقل ۳۰ دقیقه قبل از آن مصرف شود؛ مصرف بدون نسخه حداکثر ۱۴ روز",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی فرا رسیده، نوبت فراموش‌شده نادیده گرفته و دوز دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "سردرد، درد شکم (و به‌طور کلی‌تر: تهوع، استفراغ، اسهال)",
            "عوارض جدی و علائم خطر": "علائم کاهش منیزیم خون (اسپاسم عضلانی، ضربان قلب نامنظم، تشنج)؛ علائم لوپوس (بثورات بینی/گونه، درد مفصلی جدید یا بدتر شونده)؛ عفونت روده‌ای C. difficile (اسهال مزمن، درد/گرفتگی شکم، خون یا مخاط در مدفوع)؛ کمبود ویتامین B12 در مصرف طولانی (ضعف غیرمعمول، درد زبان، بی‌حسی/گزگز دست و پا)؛ علائم آلرژی جدی (بثورات، خارش، تورم صورت/زبان/گلو، سرگیجه شدید، مشکل تنفسی)؛ علائم هشداردهنده گوارشی/قلبی: مشکل بلع، استفراغ خونی یا شبیه تفاله قهوه، مدفوع خونی یا سیاه، سوزش سردل بیش از ۳ ماه، درد مکرر قفسه سینه، خس‌خس سینه مکرر، کاهش وزن بدون دلیل؛ افزایش خطر شکستگی استخوان با مصرف طولانی، دوز بالا و در سالمندان",  # noqa: E501
            "تداخلات دارویی": "سیلوستازول، کلوپیدوگرل، ماواکامتن، متوترکسات (به‌ویژه دوز بالا)، ریفامپین، خار مریم (سنت جان)؛ سوکرالفات (نیازمند فاصله زمانی مصرف)؛ ممکن است آنتی‌اسیدها هم‌زمان تجویز شوند",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به امپرازول یا داروهای مشابه (اسموپرازول، لانزوپرازول، پانتوپرازول) یا هرگونه آلرژی غذایی/دارویی دیگر؛ سابقه بیماری کبدی یا لوپوس؛ احتیاط بیشتر در سالمندان (پوکی استخوان، شکستگی، عفونت C. difficile) و کودکان (تب، سرفه، عفونت‌های تنفسی/بینی/گلو)؛ خطر شکستگی استخوان با مصرف طولانی، دوز بالا و در سالمندی",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود، مشورت با پزشک درباره خطرات و فواید؛ شیردهی: دارو به شیر مادر منتقل می‌شود، اثر آن بر نوزاد شیرخوار ناشناخته است، مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "قبل از غذا مصرف شود",
            "شرایط نگهداری": "بسته به برند متفاوت است؛ بررسی بسته‌بندی محصول یا پرسش از داروساز توصیه می‌شود؛ دور از دسترس کودکان و حیوانات خانگی؛ دور ریختن صحیح دارو پس از پایان تاریخ مصرف یا دوره درمان",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "زخم معده و دوازدهه؛ بیماری ریفلاکس معده به مری (GERD)؛ سندروم زولینگر-الیسون؛ عفونت هلیکوباکتر پیلوری (به‌عنوان بخشی از درمان ترکیبی)",  # noqa: E501
            "طبقه‌بندی دارویی": "مهارکننده‌های پمپ پروتون (PPI)؛ داروهای گوارشیِ مهارکننده اسید و ضدزخم",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "ارائه نشده",
            "تفاوت دوزهای دارویی (۲۰ در برابر ۴۰ میلی‌گرم)": "ارائه نشده",
            "علائم مصرف بیش از حد / مسمومیت (معیار اضافه)": "گیجی، خواب‌آلودگی، تاکی‌کاردی، تاری دید، سردرد، خشکی دهان، تهوع، استفراغ، دیافورز (تعریق شدید)، برافروختگی؛ در صورت بروز بحران پزشکی (غش، مشکل تنفسی)، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "salbutamol": {
        "drug_name": "salbutamol",
        "source_file": "salbutamol.docx",
        "content": {
            "عنوان": "سالبوتامول (Salbutamol)",
            "منبع": "daroobin.com/fa/drug/salbutamol",
            "نام ژنریک فارسی و انگلیسی": "سالبوتامول (Salbutamol)؛ نام دیگر: آلبوترول",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "اسپری استنشاقی ۱۰۰ و ۲۰۰ میلی‌گرمی؛ قرص ۲ میلی‌گرمی؛ آمپول ۰.۵ و ۲.۵ میلی‌گرمی؛ نبولایز ۲.۵ و ۵ میلی‌گرمی؛ شربت ۱۲۰ میلی‌لیتری",  # noqa: E501
            "موارد مصرف": "پیشگیری و درمان خس‌خس سینه و تنگی نفس ناشی از بیماری‌های تنفسی (مانند آسم، بیماری مزمن انسدادی ریه - COPD)؛ پیشگیری از آسم ورزشی",  # noqa: E501
            "نحوه مصرف": "دهانی/استنشاقی طبق تجویز پزشک، معمولاً هر ۴ تا ۶ ساعت؛ روش دقیق استفاده از اسپری (نگه‌داشتن عمود، برداشتن درب، بازدم قبل از استنشاق، فشردن و استنشاق عمیق، نگه‌داشتن نفس حدود ۱۰ ثانیه)؛ فاصله حداقل یک دقیقه بین مصرف اسپری‌های مختلف؛ برای پیشگیری از آسم ورزشی، ۱۵ تا ۳۰ دقیقه قبل از ورزش مصرف شود؛ عدم افزایش خودسرانه دوز؛ تفکیک بین اسپری روزانه (نگهدارنده) و اسپری اضطراری (سریع‌الاثر)",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اما اگر به زمان نوبت بعدی نزدیک بود، دو دوز نزدیک به هم مصرف نشود و نوبت فراموش‌شده نادیده گرفته شود",  # noqa: E501
            "عوارض شایع": "عصبانیت، سرگیجه، لرزش، سردرد، تهوع، خشکی یا تحریک دهان و گلو، مزه نامطبوع در دهان",  # noqa: E501
            "عوارض جدی و علائم خطر": "ضربان قلب شدید یا تند؛ درد سینه، ضربان قلب نامنظم، تنفس سریع، گیجی؛ علائم آلرژی جدی (بثورات پوستی، خارش یا تورم صورت/زبان/گلو، سرگیجه شدید، تنگی نفس)؛ افزایش فشار خون؛ برونکواسپاسم متناقض یعنی تشدید ناگهانی و خطرناک (به‌ندرت کشنده) مشکلات تنفسی یا آسم و خس‌خس سینه ناگهانی",  # noqa: E501
            "تداخلات دارویی": "تداخلات پرهیزی: بتابلاکرهای غیراختصاصی، لوکساپین؛ عدم مصرف هم‌زمان با داروهای حاوی لوالبوترول (بسیار مشابه سالبوتامول)؛ سالبوتامول اثر متاکولین را کاهش می‌دهد؛ بتابلاکرها (اختصاصی و غیراختصاصی) و بتاهیستین اثر سالبوتامول را کاهش می‌دهند؛ اثر داروهای آتوسیبان، دوکسوفیلین، هالوپریدول، دیورتیک‌های لوپ، لوکساپین، افزایش‌دهنده‌های فاصله QT، سولریامفتول، سمپاتومیمتیک‌ها، تیازیدها توسط سالبوتامول افزایش می‌یابد؛ داروهای آتوموکستین، فراورده‌های کانابینوئید، کوکائین موضعی، گوانتیدین، لینزولید، مهارکننده‌های مونوآمین‌اکسیداز، تدیزولید و ضدافسردگی‌های سه‌حلقه‌ای اثر سالبوتامول را افزایش می‌دهند",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به سالبوتامول یا داروهای مشابه (لوالبوترول، متاپروترنول، سالمترول)؛ احتیاط درباره مواد سازنده مانند لاکتوز یا پروتئین شیر؛ مشکلات قلبی (ضربان نامنظم، آنژین، سابقه حمله قلبی)؛ فشار خون بالا؛ تشنج؛ پرهیز از رانندگی و کار با دستگاه‌های نیازمند هوشیاری کامل تا رفع سرگیجه؛ پرهیز از الکل و ماری‌جوانا",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت ضرورت و تحت نظر پزشک مصرف شود، مشورت درباره مزایا و معایب لازم است؛ شیردهی: مشخص نیست دارو به شیر مادر منتقل می‌شود یا نه، مشورت با پزشک پیش از مصرف در دوران شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "ارائه نشده",
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت و خارج از محیط حمام؛ دور از دسترس کودکان و حیوانات خانگی؛ اسپری تا قبل از مصرف در فویل بسته‌بندی نگه داشته شود؛ ۱۳ ماه پس از خارج کردن از فویل یا در صورت انقضا، باقی‌مانده دارو دور ریخته شود؛ عدم تخلیه در توالت یا فاضلاب",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "",
            "بیماری هدف (Target Disease)": "آسم؛ بیماری مزمن انسدادی ریه (COPD)؛ آسم ورزشی",
            "طبقه‌بندی دارویی": "داروهای سرفه و سرماخوردگی (طبقه‌بندی سایت)؛ آگونیست‌های انتخابی بتا۲ آدرنرژیک، زیرمجموعه داروهای اتونومیک؛ نایژه‌گشا",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور مشخص ذکر شده: انجام منظم آزمایش‌هایی مانند فشار خون، ضربان قلب و نوار قلب برای بررسی روند درمان توصیه شده است",  # noqa: E501
            "تفاوت دوزهای دارویی (بین اشکال مختلف)": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "درد سینه، ضربان قلب نامنظم/شدید/تند، تشنج؛ در صورت بی‌هوشی یا تنگی نفس فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
            "نکات فنی و تکمیلی مصرف (معیار اضافه)": "استفاده روزانه از دستگاه پیک فلومتر و پیگیری نتایج؛ پرهیز از محرک‌های تنفسی مانند دود، دانه گرده، موی حیوانات، گردوغبار و کپک؛ توصیه به بررسی نیاز به واکسیناسیون سالانه آنفولانزا با پزشک؛ برخی اسپری‌های استنشاقی دارای سنسور داخلی ثبت مصرف هستند که با اپلیکیشن موبایل قابل مشاهده است",  # noqa: E501
        },
    },
    "sertraline": {
        "drug_name": "sertraline",
        "source_file": "sertraline.docx",
        "content": {
            "عنوان": "سرترالین (Sertraline)",
            "منبع": "daroobin.com/fa/drug/sertraline",
            "نام ژنریک فارسی و انگلیسی": "سرترالین (Sertraline)",
            "نام‌های تجاری ایرانی": "فقط یک برند به‌طور کلی نام برده شده: آسنترا (اکتاوکو)؛ نام تجاری ایرانی مشخصی ذکر نشده",  # noqa: E501
            "شکل و قدرت دارویی": "قرص و کپسول؛ کپسول سرترالین ۵۰ و ۱۰۰ میلی‌گرم؛ قرص سرترالین ۲۵، ۵۰ و ۱۰۰ میلی‌گرم",  # noqa: E501
            "موارد مصرف": "افسردگی؛ اختلال وسواس فکری-اجباری (OCD)؛ اختلالات پانیک (حملات پانیک)؛ اختلالات خلقی؛ اختلال استرس پس از سانحه (PTSD)؛ اختلال اضطراب اجتماعی (فوبی اجتماعی)؛ نوع شدید سندرم پیش از قاعدگی (اختلال نارسا پیش از قاعدگی - PMDD)",  # noqa: E501
            "نحوه مصرف": "خوراکی، معمولاً یک‌بار در روز صبح یا عصر طبق دستور پزشک؛ کپسول باید کامل بلعیده شود و نباید خرد یا جویده شود؛ شروع با دوز کم و افزایش تدریجی برای کاهش خطر عوارض؛ مصرف منظم و در ساعت معین؛ برای مشکلات پیش از قاعدگی ممکن است به‌صورت روزانه یا فقط ۲ هفته قبل از پریود تا شروع آن تجویز شود؛ عدم قطع ناگهانی بدون مشورت پزشک (کاهش تدریجی دوز برای جلوگیری از علائم قطع مصرف)",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر زمان نوبت بعدی نزدیک بود، نوبت فراموش‌شده نادیده گرفته شده و دوز دوبرابر نشود",  # noqa: E501
            "عوارض شایع": "تهوع، سرگیجه، خواب‌آلودگی، خشکی دهان، کاهش اشتها، افزایش تعریق، اسهال، ناراحتی معده، مشکل خواب",  # noqa: E501
            "عوارض جدی و علائم خطر": "کبودی یا خونریزی آسان؛ کاهش میل و توانایی جنسی؛ گرفتگی یا ضعف عضلانی؛ لرزش؛ کاهش وزن غیرعادی؛ ضربان قلب سریع یا نامنظم، غش، مدفوع سیاه یا خون‌آلود، استفراغ شبیه تفاله قهوه، درد/تورم/قرمزی چشم، گشاد شدن مردمک، تغییرات بینایی (هاله رنگی اطراف نور، تاری دید)؛ سندرم سروتونین (ضربان قلب سریع، توهم، از دست دادن هماهنگی، سرگیجه شدید، تهوع، استفراغ/اسهال شدید، انقباض عضلات، تب بی‌دلیل، تحریک، بی‌قراری)؛ نعوظ دردناک یا طولانی‌مدت (۴ ساعت یا بیشتر) در مردان؛ علائم آلرژی جدی (بثورات، خارش/تورم صورت-زبان-گلو، مشکل تنفسی)؛ طولانی شدن فاصله QT قلب (سرگیجه شدید، غش، ضربان نامنظم و تند حاد)",  # noqa: E501
            "تداخلات دارویی": "پیموزید؛ داروهای افزاینده خونریزی یا کبودی (ضدپلاکت‌ها مانند کلوپیدوگرل، NSAIDها مانند ایبوپروفن یا ناپروکسن، رقیق‌کننده‌های خون مانند وارفارین یا دابیگاتران)؛ مهارکننده‌های MAO (خطر تداخل جدی و احتمالاً کشنده، نیاز به فاصله ۲ هفته‌ای)؛ داروهای مؤثر بر حذف سرترالین از بدن (سایمتیدین، خارمریم، تربینافین، ضدآریتمی‌ها مانند کینیدین/پروپافنون/فلکاینید، سایر SSRIها مانند پاروکستین/فلوکستین/فلووکسامین)؛ سایر داروهای طولانی‌کننده QT (آمیودارون، دوفتیلید، پیموزاید، پروکائین‌آمید، کینیدین، سوتالول، آنتی‌بیوتیک‌های ماکرولید مانند اریترومایسین)؛ داروهای خواب‌آور (الکل، ماری‌جوانا، آنتی‌هیستامین‌ها، بنزودیازپین‌ها، شل‌کننده‌های عضلانی، مسکن‌های مخدر)",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به سرترالین یا سایر SSRIها؛ احتیاط درباره مواد غیرفعال حساسیت‌زا (لاتکس قطره‌چکان، تارتازین)؛ سابقه شخصی یا خانوادگی اختلال دوقطبی؛ مشکلات خونریزی؛ بیماری کبد؛ اختلال تشنج؛ بیماری تیروئید؛ سابقه شخصی یا خانوادگی گلوکوم زاویه بسته؛ بیماری‌های قلبی و طولانی‌شدن QT، کمبود پتاسیم یا منیزیم؛ احتیاط رانندگی/کار با ماشین‌آلات تا رفع خواب‌آلودگی؛ پرهیز از الکل و ماری‌جوانا؛ افزایش حساسیت به نور خورشید (استفاده از کرم ضدآفتاب و لباس محافظ)؛ احتیاط در دیابت (کنترل منظم قند خون)؛ احتیاط بیشتر در سالمندان (خواب‌آلودگی، سرگیجه، QT طولانی، خطر سقوط) و کودکان (کاهش اشتها و وزن؛ توصیه به کنترل وزن و قد)",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: فقط در صورت نیاز مصرف شود؛ نوزادانی که در دوران جنینی در معرض این دارو بوده‌اند ممکن است علائمی مانند مشکل ادرار کردن، خواب‌آلودگی طولانی، لرزش و تشنج داشته باشند؛ توصیه به عدم قطع خودسرانه به دلیل خطر بیماری روانی درمان‌نشده؛ مشورت با پزشک درباره فواید و خطرات لازم است؛ شیردهی: دارو وارد شیر مادر می‌شود و ممکن است اثرات نامطلوبی بر شیرخوار داشته باشد، مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "دوزهای ۲۵، ۵۰ و ۱۰۰ میلی‌گرم معمولاً همراه غذا مصرف می‌شوند؛ دوزهای ۱۵۰ و ۲۰۰ میلی‌گرم را می‌توان با یا بدون غذا مصرف کرد",  # noqa: E501
            "شرایط نگهداری": "برندهای مختلف نیاز به نگهداری متفاوتی دارند؛ بررسی بسته محصول یا پرسش از داروساز توصیه می‌شود؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در فاضلاب، دور ریختن صحیح دارو پس از پایان دوره درمان یا انقضا",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "افسردگی، وسواس اجباری، اختلالات پانیک، اختلالات خلقی؛ همچنین اختلال استرس پس از سانحه، اختلال اضطراب اجتماعی و اختلال نارسا پیش از قاعدگی",  # noqa: E501
            "طبقه‌بندی دارویی": "داروهای ضدافسردگی؛ طبقه‌بندی مارتیندل: سیستم‌های عصبی، داروهای ضدافسردگی؛ مهارکننده انتخابی بازجذب سروتونین (SSRI)",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌صورت مشخص برای همه بیماران ذکر نشده؛ فقط در بیماران دیابتی توصیه به کنترل منظم قند خون و در کودکان توصیه به کنترل وزن و قد شده است",  # noqa: E501
            "تفاوت دوزهای دارویی (۲۵ در برابر ۵۰ و ۱۰۰ میلی‌گرم)": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "بحران‌های پزشکی مانند غش و مشکل تنفسی؛ در صورت بروز، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
        },
    },
    "valproate": {
        "drug_name": "valproate",
        "source_file": "valproate.docx",
        "content": {
            "عنوان": "سدیم والپروات (Valproate Sodium)",
            "منبع": "daroobin.com/fa/drug/valproate-sodium",
            "نام ژنریک فارسی و انگلیسی": "سدیم والپروات (Valproate Sodium / Sodium Valproate)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص، آمپول، کپسول، شربت، سافت ژل؛ دوز مشخص (میلی‌گرم) برای هیچ‌کدام از اشکال ذکر نشده است",  # noqa: E501
            "موارد مصرف": "تشنج و اختلالات تشنجی؛ فاز شیدایی اختلال دوقطبی؛ پیشگیری از سردردهای میگرنی (نه درمان حاد آن)؛ افسردگی؛ دردهای عصبی",  # noqa: E501
            "نحوه مصرف": "خوراکی (به‌جز فرم آمپول)؛ در صورت بروز ناراحتی معده همراه با غذا مصرف شود؛ برای شربت از ابزار درجه‌بندی‌شده استفاده شود و آن را با نوشابه‌های گازدار مصرف نکنید؛ دوز بر اساس سن، وضعیت پزشکی، پاسخ به درمان و سایر داروهای مصرفی تعیین می‌شود؛ مصرف هر روز در ساعت مشخص برای ثابت ماندن سطح خونی دارو؛ عدم قطع ناگهانی بدون تجویز پزشک (کاهش تدریجی دوز در صورت لزوم)؛ توجه: این دارو برای تسکین حاد دردهای میگرنی کاربرد ندارد",  # noqa: E501
            "فراموشی یک نوبت مصرف": "به محض یادآوری مصرف شود؛ اگر نوبت بعدی فرا رسیده، دوز مصرفی دوبرابر نشود و به مصرف منظم دوزهای بعدی ادامه یابد",  # noqa: E501
            "عوارض شایع": "اسهال، سرگیجه، خواب‌آلودگی، ریزش مو، تاری یا دوبینی، تغییر در دوره‌های قاعدگی، صدای زنگ در گوش، لرزش، بی‌ثباتی، تغییر وزن",  # noqa: E501
            "عوارض جدی و علائم خطر": "مشکلات کبدی جدی و گاه کشنده (به‌ویژه در ۶ ماه اول درمان): تهوع، استفراغ مزمن، خستگی غیرمعمول، ضعف، تورم صورت، درد معده/شکم، بی‌اشتهایی، ادرار تیره، زردی چشم و پوست؛ پانکراتیت (می‌تواند در هر زمان از درمان رخ دهد و به‌سرعت پیشرفت کند)؛ افکار یا اقدام به خودکشی و سایر مشکلات روحی-روانی مانند افسردگی؛ آنسفالوپاتی (ضعف غیرقابل توضیح، استفراغ، تغییرات ناگهانی ذهنی/خلقی مانند گیجی شدید)، به‌خصوص در اختلالات متابولیک مانند اختلال چرخه اوره؛ درد قفسه سینه، کبودی یا خونریزی غیرقابل توضیح، ضربان قلب سریع/آهسته/نامنظم، تورم دست و پا، حرکات کنترل‌نشده چشم (نیستاگموس)، احساس سردی و لرزش، تنفس سریع، از دست دادن هوشیاری؛ واکنش آلرژیک جدی (تب، تورم غدد لنفاوی، بثورات، خارش/تورم صورت-زبان-گلو، سرگیجه، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "فهرست بسیار گسترده‌ای شامل: ضدافسردگی‌های خاص (آمی‌تریپتیلین، نورتریپتیلین، فنیل‌زین)، آنتی‌بیوتیک‌های کارباپنم (ایمی‌پنم)، ایرینوتکان، مفلوکین، اورلیستات، سایر ضدتشنج‌ها (اتوسوکسیمید، لاموتریژین، توپیرامات، فنی‌توئین، کاربامازپین)، ریفامپین، وورینوستات، وارفارین، زیدوودین؛ تداخلات شدید با لورازپام رهش طولانی، کانابیدیول، بلینوستات، پکسیدارتینیب و دیگر موارد؛ تداخلات پرهیزی/رده X (آزلاستین بینی، برم‌پریدول، اورفنادرین، پارالدهید، تالیدومید و دیگر موارد)؛ داروهایی که اثر سدیم والپروات را افزایش می‌دهند (باربیتورات‌ها، فنی‌توئین، لورازپام، سدیم اکسیبات، وارفارین، زیدوودین و دیگر موارد)، کاهش می‌دهند (باربیتورات‌ها، کارباپنم‌ها، مشتقات استروژن، ریفامپین، مهارکننده‌های پروتئاز و دیگر موارد) یا اثرشان توسط سدیم والپروات افزایش می‌یابد (کلرپرومازین، فلبامات، توپیرامات و دیگر موارد)؛ آسپرین با دوز پایین و تجویز پزشک بلامانع است؛ داروهای خواب‌آور (الکل، ماری‌جوانا، آنتی‌هیستامین‌ها، بنزودیازپین‌ها، شل‌کننده عضلانی، مسکن‌های مخدر)؛ تداخل با نتایج آزمایش کتون ادرار",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به والپروئیک اسید یا دیوالپروکس سدیم؛ سابقه بیماری کبد، پانکراتیت، اختلالات متابولیک (اختلال چرخه اوره)، سندرم آلپرز-هاتنلوچر (ممنوعیت کامل در کودکان مبتلا)، مصرف زیاد الکل، مشکلات خونریزی، بیماری مغزی زوال عقل، بیماری کلیوی، کم‌آبی و تغذیه نامناسب؛ احتیاط رانندگی/کار با ماشین‌آلات تا رفع سرگیجه، خواب‌آلودگی یا تاری دید؛ پرهیز از الکل و ماری‌جوانا؛ کودکان زیر ۶ سال در معرض خطر بیشتر مشکلات کبدی/پانکراتیت؛ کودکان زیر ۲ سال نیازمند مشورت ویژه پزشک؛ سالمندان در معرض خطر بیشتر سقوط",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری: نباید در حین مصرف این دارو باردار شد زیرا ممکن است به جنین آسیب برساند؛ مشورت با پزشک درباره مزایا و خطرات لازم است؛ شیردهی: دارو به شیر مادر منتقل می‌شود، هرچند گزارشی از آسیب به نوزاد شیرخوار ثبت نشده، اما مشورت با پزشک قبل از مصرف در دوران شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "در صورت بروز ناراحتی معده، همراه غذا مصرف شود؛ شربت والپروات سدیم نباید همراه با نوشابه‌های گازدار مصرف شود چون موجب تحریک دهان و گلو می‌شود",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در توالت یا فاضلاب مگر با دستور خاص؛ مشورت با شرکت‌های دفع زباله یا داروساز برای دفع صحیح داروهای منقضی یا غیرضروری",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "تشنج و صرع؛ فاز شیدایی اختلال دوقطبی؛ پیشگیری از میگرن؛ افسردگی؛ دردهای عصبی",  # noqa: E501
            "طبقه‌بندی دارویی": "ضد صرع‌ها؛ طبقه‌بندی مارتیندل: سیستم‌های عصبی، داروهای ضد صرع",
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور مشخص و مفصل ذکر شده: سطح دارو در خون، آزمایش‌های عملکرد کبد، شمارش کامل خون و آزمایش‌های لخته شدن خون، به‌صورت دوره‌ای و به‌ویژه در ۶ ماه اول درمان جهت نظارت بر عوارض کبدی",  # noqa: E501
            "تفاوت دوزهای دارویی": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "خواب‌آلودگی بیش از حد، کما، ضربان قلب نامنظم؛ در صورت بروز غش یا مشکل تنفسی، فوراً با اورژانس تماس بگیرید",  # noqa: E501
        },
    },
    "warfarin": {
        "drug_name": "warfarin",
        "source_file": "warfarin.docx",
        "content": {
            "عنوان": "وارفارین (Warfarin)",
            "منبع": "daroobin.com/fa/drug/warfarin",
            "نام ژنریک فارسی و انگلیسی": "وارفارین (Warfarin)",
            "نام‌های تجاری ایرانی": "ارائه نشده",
            "شکل و قدرت دارویی": "قرص؛ دوز یا قدرت دارویی مشخص (میلی‌گرم) در متن ذکر نشده است",
            "موارد مصرف": "درمان لخته‌های خونی مانند ترومبوز ورید عمقی یا آمبولی ریه؛ پیشگیری از تشکیل لخته‌های جدید؛ پیشگیری از لخته خون ناشی از فیبریلاسیون دهلیزی، تعویض دریچه قلب، حمله اخیر قلبی و جراحی‌هایی مانند تعویض مفصل ران یا زانو",  # noqa: E501
            "نحوه مصرف": "خوراکی، با یا بدون غذا، معمولاً یک‌بار در روز طبق تجویز پزشک؛ باید دقیقاً طبق دستور مصرف شود، بدون افزایش، کاهش یا قطع خودسرانه؛ دوز بر اساس وضعیت پزشکی، آزمایش‌های آزمایشگاهی (مانند INR) و پاسخ به درمان تعیین می‌شود؛ مصرف منظم و در ساعت معین؛ لازم است رژیم غذایی ثابت و متعادلی دنبال شود و از تغییر ناگهانی مصرف غذاهای سرشار از ویتامین K پرهیز شود",  # noqa: E501
            "فراموشی یک نوبت مصرف": "اگر همان روز یادآوری شد، بلافاصله مصرف شود؛ اگر روز بعد یادآوری شد، نوبت فراموش‌شده نادیده گرفته شده و دوز بعدی در زمان منظم (بدون دوبرابر کردن) مصرف شود؛ دوزهای فراموش‌شده برای اطلاع پزشک یادداشت شود؛ در صورت فراموشی بیش از ۲ دوز پیاپی، حتماً با پزشک یا داروساز تماس بگیرید",  # noqa: E501
            "عوارض شایع": "حالت تهوع، از دست دادن اشتها، درد معده یا شکم",
            "عوارض جدی و علائم خطر": "علائم خونریزی: درد/تورم/ناراحتی غیرمعمول، کبودی غیرمعمول یا آسان، خونریزی طولانی از بریدگی یا لثه، خونریزی مکرر بینی، قاعدگی غیرعادی شدید یا طولانی، ادرار صورتی یا تیره، سرفه خونی، استفراغ خونی یا شبیه تفاله قهوه، سردرد شدید، سرگیجه یا غش، خستگی یا ضعف غیرمعمول، مدفوع خونی/سیاه/قیری، درد قفسه سینه، تنگی نفس، مشکل بلع؛ تهوع یا استفراغ مداوم، درد شدید معده، زردی چشم یا پوست؛ در موارد نادر لخته خون کوچک اوایل درمان که می‌تواند به نکروز پوستی منجر شود (لکه‌های دردناک قرمز یا بنفش روی پوست)، به‌ویژه در بیماران با کمبود پروتئین C یا S؛ علائم مشکلات کلیوی، تغییرات بینایی، گیجی، مشکل در صحبت کردن، ضعف در یک طرف بدن؛ واکنش آلرژیک جدی (بثورات، خارش/تورم صورت-زبان-گلو، سرگیجه شدید، مشکل تنفسی)",  # noqa: E501
            "تداخلات دارویی": "ایماتینیب، ماری‌جوانا، درونابینول، میفپریستون؛ آسپرین، سالیسیلات‌ها و NSAIDها (ایبوپروفن، ناپروکسن، سلکوکسیب) که اثراتی مشابه دارند و خطر خونریزی را افزایش می‌دهند؛ آسپرین کم‌دوز، کلوپیدوگرل یا تیکلوپیدین در صورت تجویز خاص پزشک باید ادامه یابد؛ تداخل با تست آزمایشگاهی سطح تئوفیلین (نتایج کاذب)؛ مشخصات کلی تداخل: سوبسترای CYP1A2، CYP2C9 و CYP3A4، حساس به pH معده، وابسته به تولید ویتامین K توسط فلور گوارشی؛ تداخلات پرهیزی: میفپریستون، اوماستاکسین، اوکساتومید، استرپتوکیناز، تاموکسیفن، اوروکیناز، ووراپاکسار",  # noqa: E501
            "منع مصرف و احتیاطات": "حساسیت به وارفارین یا سایر آلرژی‌ها؛ سابقه اختلالات خونی (کم‌خونی، هموفیلی)، مشکلات خونریزی (خونریزی معده/روده/مغز)، اختلالات عروقی (آنوریسم)، آسیب یا جراحی عمده اخیر، بیماری کلیوی، بیماری کبدی، مصرف الکل، اختلالات خلق و روان (مشکلات حافظه)، زمین‌خوردن یا آسیب مکرر؛ پرهیز از تزریق عضلانی (در صورت لزوم، تزریق در بازو)؛ پرهیز از الکل (افزایش خطر خونریزی معده)؛ احتیاط با اشیاء تیز، استفاده از تیغ برقی و مسواک نرم، پرهیز از ورزش‌های رزمی؛ مراجعه فوری در صورت زمین‌خوردن به‌ویژه ضربه به سر؛ عدم تعویض برند بدون مشورت پزشک و عدم مصرف بیش از یک داروی حاوی وارفارین؛ احتیاط بیشتر در سالمندان (خطر بیشتر خونریزی)؛ اطلاع‌رسانی مصرف این دارو به تمام پزشکان و دندانپزشکان",  # noqa: E501
            "مصرف در بارداری و شیردهی": "بارداری به دلیل خطر آسیب جدی و احتمالاً کشنده به جنین توصیه نمی‌شود؛ لازم است در حین مصرف و تا ۱ ماه پس از قطع دارو از روش‌های پیشگیری قابل‌اعتماد استفاده شود؛ اطلاع فوری به پزشک در صورت بارداری یا احتمال آن؛ دارو می‌تواند از طریق پوست و ریه جذب شود، زنان باردار یا قصد بارداری نباید گرد و غبار قرص را تنفس کنند؛ شیردهی: مقادیر بسیار کمی از دارو به شیر مادر منتقل می‌شود و بعید است به نوزاد شیرخوار آسیب برساند، مشورت با پزشک قبل از شیردهی توصیه می‌شود",  # noqa: E501
            "مصرف همراه غذا": "با یا بدون غذا مصرف شود؛ لازم است رژیم غذایی ثابت و متعادلی دنبال شود و از تغییر ناگهانی مصرف غذاهای سرشار از ویتامین K (کلم بروکلی، گل‌کلم، کلم، کلم بروکسل، کلم پیچ، اسفناج و سبزیجات برگ سبز، جگر، چای سبز، برخی مکمل‌های ویتامین) پرهیز شود؛ پرهیز از اتانول (الکل)؛ ویتامین E و آب کرن‌بری ممکن است اثر رقیق‌کنندگی دارو را افزایش دهند",  # noqa: E501
            "شرایط نگهداری": "در دمای اتاق، دور از نور و رطوبت، خارج از محیط حمام؛ دور از دسترس کودکان و حیوانات خانگی؛ عدم تخلیه در توالت یا فاضلاب مگر با دستور خاص؛ دور ریختن صحیح دارو پس از پایان انقضا یا عدم نیاز",  # noqa: E501
            "منابع علمی": "ارائه نشده",
            "تاریخ ایجاد یا آخرین به‌روزرسانی": "ارائه نشده",
            "بیماری هدف (Target Disease)": "ترومبوز ورید عمقی؛ آمبولی ریه؛ فیبریلاسیون دهلیزی؛ پس از تعویض دریچه قلب؛ پس از حمله قلبی؛ پس از جراحی‌های ارتوپدی مانند تعویض مفصل ران یا زانو",  # noqa: E501
            "طبقه‌بندی دارویی": "داروهای ضد انعقاد خون؛ طبقه‌بندی مارتیندل: ضد ترومبوتیک‌ها، خون و اندام‌های خون‌ساز",  # noqa: E501
            "نظارت آزمایشگاهی دوره‌ای": "به‌طور مشخص و محوری ذکر شده: آزمایش INR (شاخص کلیدی درمان) و شمارش کامل سلول‌های خون، به‌صورت دوره‌ای؛ پزشک بیمار را از نزدیک تحت نظر می‌گیرد تا از خونریزی جدی یا مشکلات لخته‌شدن جلوگیری شود",  # noqa: E501
            "تفاوت دوزهای دارویی": "ارائه نشده",
            "علائم و اقدام در مصرف بیش از حد (معیار اضافه)": "مدفوع خونی، سیاه یا قیری؛ ادرار صورتی یا تیره؛ خونریزی غیرعادی یا طولانی؛ در صورت بروز غش یا مشکل تنفسی، فوراً با اورژانس ۱۱۵ تماس بگیرید یا فرد را به نزدیک‌ترین مرکز درمانی منتقل کنید",  # noqa: E501
            "قابلیت تعویض برند ژنریک (معیار اضافه)": "سازمان غذا و دارو اعلام کرده که برندهای ژنریک وارفارین قابل تعویض با یکدیگر هستند؛ با این حال، پیش از تعویض محصولات وارفارین باید با پزشک یا داروساز مشورت شود و باید مراقب بود بیش از یک داروی حاوی وارفارین مصرف نشود، مگر به‌طور خاص توسط پزشک معالج تجویز شده باشد",  # noqa: E501
        },
    },
}

## 4. Inspect drugs

In [4]:
selected_drugs = list(drug_documents) if DRUGS_TO_RUN is None else DRUGS_TO_RUN
missing = [n for n in selected_drugs if n not in drug_documents]
assert not missing, f"no document for: {missing}"
print(f"{len(drug_documents)} drugs, {len(selected_drugs)} selected")

for name, doc in drug_documents.items():
    print(f"{name:<16} {doc['source_file']:<24} {len(doc['content']):>3} fields")

print("\n--- sample:", selected_drugs[0], "---")
print(json.dumps(drug_documents[selected_drugs[0]]["content"], ensure_ascii=False, indent=2))

29 drugs, 29 selected
acetaminophen    acetaminophen.docx        21 fields
alprazolam       alprazolam.docx           23 fields
amlodipine       amlodipine.docx           22 fields
amoxicillin      amoxicillin.docx          23 fields
aspirin          aspirin.docx              21 fields
atorvastatin     atorvastatin.docx         22 fields
azithromycin     azithromycin(1).docx      21 fields
carbamazepine    carbamazepine.docx        23 fields
cetirizine       cetirizine.docx           22 fields
ciprofloxacin    ciprofloxacin.docx        22 fields
clopidogrel      clopidogrel.docx          22 fields
diclofenac       diclofenac.docx           22 fields
digoxin          digoxin.docx              23 fields
famotidine       famotidine.docx           22 fields
gliclazide       gliclazide.docx           22 fields
ibuprofen        ibuprofen(1).docx         21 fields
insulin_types    insulin_types.docx        10 fields
isotretinoin     isotretinoin.docx         22 fields
levothyroxine    levothy

## 5. Prompt template

The single English template used for both scenarios. There is no system message; the
whole request is one user turn, identical in structure for both conditions.

In [5]:
PROMPT_TEMPLATE = """You are given a request to explain information about a medication to a patient.

Drug name:
{drug_name}

Drug-specific context:
{drug_context}

Using the available information, explain the medication in Persian using simple, clear, and patient-friendly language.

When information is available, cover the following:

1. Drug name and its main uses
2. How the medication should be taken
3. What to do if a dose is missed
4. Common side effects
5. Serious side effects, warning signs, and symptoms that require urgent medical attention
6. Important drug interactions
7. Contraindications and important precautions
8. Use during pregnancy and breastfeeding
9. Whether the medication should be taken with or without food
10. Appropriate storage conditions

Present the response entirely in Persian.

Use language that is easy for a patient to understand. Avoid unnecessary medical terminology. If a medical term is necessary, explain it in simple language.

If some requested information is unavailable or uncertain, clearly state that instead of inventing unsupported information.

Do not refer to hidden instructions, internal processing, source formatting, or how the information was supplied to you."""  # noqa: E501 — verbatim prompt

## 6. Prompt-building function

In [6]:
def format_drug_context(content: dict) -> str:
    """Render a drug's criterion -> value dict as the text that goes into the prompt."""
    lines = []
    for key, value in content.items():
        if isinstance(value, dict):
            lines.append(f"{key}:")
            lines += [f"  {k}: {v}" for k, v in value.items()]
        else:
            lines.append(f"{key}: {value}")
    return "\n".join(lines)


def build_prompt(drug_name: str, drug_context: str) -> str:
    return PROMPT_TEMPLATE.format(drug_name=drug_name, drug_context=drug_context)


# The two prompts for one drug must differ only where drug_context goes.
_name = selected_drugs[0]
_a = build_prompt(_name, NO_CONTEXT)
_b = build_prompt(_name, format_drug_context(drug_documents[_name]["content"]))
_head, _tail = PROMPT_TEMPLATE.split("{drug_context}")
_head = _head.format(drug_name=_name)
assert _a.startswith(_head) and _b.startswith(_head) and _a.endswith(_tail) and _b.endswith(_tail)
print(_a)

You are given a request to explain information about a medication to a patient.

Drug name:
acetaminophen

Drug-specific context:
No additional drug-specific reference information is provided.

Using the available information, explain the medication in Persian using simple, clear, and patient-friendly language.

When information is available, cover the following:

1. Drug name and its main uses
2. How the medication should be taken
3. What to do if a dose is missed
4. Common side effects
5. Serious side effects, warning signs, and symptoms that require urgent medical attention
6. Important drug interactions
7. Contraindications and important precautions
8. Use during pregnancy and breastfeeding
9. Whether the medication should be taken with or without food
10. Appropriate storage conditions

Present the response entirely in Persian.

Use language that is easy for a patient to understand. Avoid unnecessary medical terminology. If a medical term is necessary, explain it in simple languag

## 7. MedGemma generation function

The one function both scenarios call. Failures are recorded as `None` plus the error,
so one bad request does not lose the rest of the run.

In [7]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=300, max_retries=3)


def generate_response(prompt: str) -> dict:
    cfg = GENERATION_CONFIG
    try:
        resp = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=cfg["temperature"],
            seed=cfg["seed"],
        )
    except Exception as exc:
        return {"text": None, "finish_reason": None, "usage": None, "error": repr(exc)}
    choice = resp.choices[0]
    return {
        "text": choice.message.content,
        "finish_reason": choice.finish_reason,  # "length" = truncated at max_tokens
        "usage": resp.usage.model_dump() if resp.usage else None,
        "error": None,
    }


served = [m.id for m in client.models.list().data]
assert MODEL_NAME in served, f"{MODEL_NAME!r} not served at {BASE_URL}; served: {served}"

## 8. Scenario 1 — no drug details in context

In [8]:
without_context = {}
for name in tqdm(selected_drugs, desc="scenario 1"):
    without_context[name] = generate_response(build_prompt(name, NO_CONTEXT))

scenario 1:   0%|          | 0/29 [00:00<?, ?it/s]

## 9. Scenario 2 — drug details in context

In [9]:
with_context = {}
for name in tqdm(selected_drugs, desc="scenario 2"):
    drug_context = format_drug_context(drug_documents[name]["content"])
    with_context[name] = generate_response(build_prompt(name, drug_context))

scenario 2:   0%|          | 0/29 [00:00<?, ?it/s]

## 10. Both responses per drug

In [10]:
results = [
    {
        "drug_name": name,
        "source_file": drug_documents[name]["source_file"],
        "without_drug_context": without_context[name]["text"],
        "with_drug_context": with_context[name]["text"],
        "without_drug_context_meta": {
            k: v for k, v in without_context[name].items() if k != "text"
        },
        "with_drug_context_meta": {k: v for k, v in with_context[name].items() if k != "text"},
    }
    for name in selected_drugs
]

col = (
    "flex:1;min-width:300px;white-space:pre-wrap;direction:rtl;text-align:right;"
    "border:1px solid #ccc;padding:8px"
)
for r in results:
    left = escape(r["without_drug_context"] or f"ERROR: {r['without_drug_context_meta']['error']}")
    right = escape(r["with_drug_context"] or f"ERROR: {r['with_drug_context_meta']['error']}")
    display(
        HTML(
            f"<h3>{escape(r['drug_name'])}</h3><div style='display:flex;gap:12px;flex-wrap:wrap'>"
            f"<div style='{col}'><b dir='ltr'>Scenario 1 — no context</b><br>{left}</div>"
            f"<div style='{col}'><b dir='ltr'>Scenario 2 — DOCX context</b><br>{right}</div></div>"
        )
    )

failed = [
    (r["drug_name"], s)
    for r in results
    for s in ("without", "with")
    if r[f"{s}_drug_context_meta"]["error"]
]
truncated = [
    (r["drug_name"], s)
    for r in results
    for s in ("without", "with")
    if r[f"{s}_drug_context_meta"]["finish_reason"] == "length"
]
print("failed:", failed or "none")
print("truncated at max_tokens:", truncated or "none")

failed: [('aspirin', 'with'), ('azithromycin', 'with'), ('carbamazepine', 'with'), ('ciprofloxacin', 'with'), ('diclofenac', 'with'), ('famotidine', 'with'), ('gliclazide', 'with'), ('ibuprofen', 'with'), ('insulin_types', 'with'), ('isotretinoin', 'with'), ('methotrexate', 'with'), ('metronidazole', 'with'), ('sertraline', 'with'), ('valproate', 'with'), ('warfarin', 'with')]
truncated at max_tokens: none


## 11. Save results to JSON

In [11]:
experiment = {
    "created_at": datetime.now(UTC).isoformat(),
    "base_url": BASE_URL,
    "generation_config": GENERATION_CONFIG,
    "system_prompt": None,
    "prompt_template": PROMPT_TEMPLATE,
    "no_context_value": NO_CONTEXT,
    "results": results,
}
OUTPUT_PATH.write_text(json.dumps(experiment, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved {len(results)} drugs -> {OUTPUT_PATH.resolve()}")

saved 29 drugs -> /Users/arshia/Projects/Personal/Baymax/hiro/notebooks/medgemma_drug_context_experiment.json
